# KLTN Attack v4.3.3 H1 CF-VLM

Chạy **Run all**. Notebook kiểm tra 2 ảnh × 300 bước, sau đó chạy/resume 8 ablation × 200 ảnh. Mỗi ablation đủ 200 ảnh sẽ được caption và GPT-4o đánh giá trước khi chạy ablation tiếp theo. Checkpoint được kiểm tra SHA-256 trên Drive. Thêm `OPENAI_API_KEY` vào Secrets hoặc `/marimo/.env`; không lưu key trong notebook. Giữ notebook evaluation v3.1 cùng thư mục.

FOA giữ lịch sử giữa các ảnh nên attack xử lý đúng thứ tự; captioning dùng batch 64 và giảm batch khi OOM theo protocol v3.1. Runtime attack/evaluation được cô lập để giữ đúng phiên bản thư viện.


In [ ]:
# Molab: Run all. All attack/control code below belongs to this notebook.
import os
from pathlib import Path
_NOTEBOOK_STAGE = os.environ.get('KLTN_NOTEBOOK_STAGE', 'control')
MOLAB_ROOT = Path('/marimo/molab_v433_campaign')
MOLAB_MAX_HOURS = 7.5
MOLAB_CAPTION_BATCH_SIZE = 64
MOLAB_DRIVE_PARENT = 'KLTN_Molab_Hypotheses_n200_v432_h1'
MOLAB_DRIVE_PREFIX = MOLAB_DRIVE_PARENT + '/molab/v433_notebook_campaign'


In [ ]:
ATTACK_RUNTIME_SOURCE = '"""Sequential, resumable execution of the original attack and evaluation notebooks.\n\nRun from the extracted campaign directory. Original notebook algorithms are left\nunchanged. Google Drive credentials must already be connected in the Molab UI.\n"""\n\nimport argparse\nimport ast\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\nimport subprocess\nimport sys\nimport time\nfrom zipfile import ZipFile, ZIP_DEFLATED\n\n\nclass BoundaryPause(BaseException):\n    """Bypass notebook sample-error handling at a completed checkpoint boundary."""\n\n\nclass BackupFailure(BaseException):\n    """A storage failure must stop execution instead of becoming a sample error."""\n\n\ndef read_json(path):\n    return json.loads(Path(path).read_text(encoding=\'utf-8\'))\n\n\ndef write_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_suffix(path.suffix + \'.tmp\')\n    temporary.write_text(json.dumps(value, indent=2, default=str) + \'\\n\', encoding=\'utf-8\')\n    temporary.replace(path)\n\n\ndef digest(path):\n    result = hashlib.sha256()\n    with Path(path).open(\'rb\') as stream:\n        for block in iter(lambda: stream.read(1024 * 1024), b\'\'):\n            result.update(block)\n    return result.hexdigest()\n\n\ndef notebook_cells(path):\n    return [(i, \'\'.join(cell[\'source\'])) for i, cell in enumerate(read_json(path)[\'cells\'])\n            if cell[\'cell_type\'] == \'code\' and \'attack\' in cell.get(\'metadata\', {}).get(\'tags\', [])]\n\n\nclass DriveBackup:\n    def __init__(self, prefix, job):\n        from gdrive_fsspec import GoogleDriveFileSystem\n        self.fs = GoogleDriveFileSystem(token=\'cache\', skip_instance_cache=True)\n        self.prefix = prefix.rstrip(\'/\') + \'/\' + job\n        self.fs.makedirs(self.prefix, exist_ok=True)\n\n    def put(self, local, relative):\n        try:\n            remote = self.prefix + \'/\' + relative\n            parent = remote.rsplit(\'/\', 1)[0]\n            self.fs.makedirs(parent, exist_ok=True)\n            self.fs.put_file(str(local), remote)\n            # Verify remote bytes before declaring the checkpoint durable.\n            check = hashlib.sha256()\n            with self.fs.open(remote, \'rb\') as stream:\n                for block in iter(lambda: stream.read(1024 * 1024), b\'\'):\n                    check.update(block)\n            if check.hexdigest() != digest(local):\n                raise RuntimeError(f\'Remote checkpoint checksum mismatch: {relative}\')\n        except Exception as exc:\n            raise BackupFailure(f\'Checkpoint storage failed ({type(exc).__name__}); stopping the campaign.\') from None\n\n    def snapshot(self, run, sample=None, metadata=True):\n        run = Path(run)\n        staging = run.parent / \'backup_staging\'\n        staging.mkdir(exist_ok=True)\n        if sample is not None:\n            sample = Path(sample)\n            relative = sample.relative_to(run)\n            archive = staging / \'sample.zip\'\n            with ZipFile(archive, \'w\', ZIP_DEFLATED) as output:\n                for path in sorted(sample.rglob(\'*\')):\n                    if path.is_file() and not path.name.endswith(\'.tmp\'):\n                        output.write(path, path.relative_to(run).as_posix())\n            self.put(archive, relative.as_posix() + \'.zip\')\n        if not metadata:\n            return\n        metadata = staging / \'root_latest.zip\'\n        with ZipFile(metadata, \'w\', ZIP_DEFLATED) as output:\n            for path in sorted(run.rglob(\'*\')):\n                if path.is_file() and \'samples\' not in path.relative_to(run).parts and not path.name.endswith(\'.tmp\'):\n                    output.write(path, path.relative_to(run).as_posix())\n        self.put(metadata, \'root_latest.zip\')\n\n\ndef setup_environments(root, plan):\n    base_python = getattr(sys, \'_base_executable\', sys.executable)\n    for stage in (\'attack\', \'eval\'):\n        env = root / \'.venvs\' / stage\n        executable = env / (\'Scripts/python.exe\' if os.name == \'nt\' else \'bin/python\')\n        if not executable.is_file():\n            subprocess.run([base_python, \'-m\', \'venv\', \'--system-site-packages\', str(env)], check=True)\n        if stage == \'attack\':\n            tree = ast.parse(notebook_cells(root / plan[\'attack_notebook\'])[1][1])\n            pins = next(ast.literal_eval(node.value) for node in tree.body\n                        if isinstance(node, ast.Assign)\n                        and any(isinstance(t, ast.Name) and t.id == \'PINNED_PACKAGES\' for t in node.targets))\n            pins += [\'numpy\', \'Pillow\', \'gdrive-fsspec\']\n        else:\n            pins = [\'transformers==4.57.6\', \'accelerate==1.10.1\', \'bitsandbytes==0.48.1\',\n                    \'openai==1.109.1\', \'tenacity==9.0.0\', \'tqdm==4.67.1\',\n                    \'sentencepiece==0.2.1\', \'Pillow\', \'ipython\', \'gdrive-fsspec\']\n        subprocess.run([str(executable), \'-m\', \'pip\', \'install\', *pins], check=True)\n        print(f\'{stage} interpreter: {executable}\', flush=True)\n\n\ndef attack_worker(root, plan, job, state_path, backup_prefix, max_hours):\n    state = read_json(state_path) if state_path.is_file() else {\'attack_runs\': {}, \'evaluations\': {}}\n    main_run = state[\'attack_runs\'].get(\'primary_ablations\')\n    overrides = dict(job.get(\'overrides\', {}))\n    revisions = plan.get(\'attack_model_revisions\')\n    if job.get(\'reuse_primary_captions\'):\n        if not main_run:\n            raise RuntimeError(\'Primary run is required before H4 controls.\')\n        texts = read_json(Path(main_run) / \'text_records.json\')\n        overrides[\'MANUAL_TEXT_OVERRIDES\'] = {sid: {key: row[key] for key in (\'source\', \'target\')}\n                                             for sid, row in texts.items() if not row.get(\'caption_error\')}\n        revisions = read_json(Path(main_run) / \'run_config.json\')[\'model_revisions\']\n    ns = {\'__name__\': \'__main__\', \'__file__\': str(root / plan[\'attack_notebook\']), \'_NOTEBOOK_STAGE\': \'attack\'}\n    cells = notebook_cells(root / plan[\'attack_notebook\'])\n    exec(compile(cells[0][1], f\'{plan["attack_notebook"]}:cell_{cells[0][0]}\', \'exec\'), ns)\n    ns.update(DATASET_ROOT=root / \'data\', PAIR_MANIFEST=root / plan[\'dataset_manifest\'],\n              WORK_ROOT=root / \'work\', OUTPUT_ROOT=root / \'work\' / job[\'name\'],\n              NUM_SAMPLES=plan[\'num_samples\'], SAMPLE_START=0, STEPS=plan[\'steps\'], SEED=plan[\'seed\'],\n              EPSILON_255=plan[\'epsilon_255\'], ALPHA_255=plan[\'alpha_255\'],\n              RUN_VARIANTS=job[\'variants\'], RUN_ATTACK=False, SETUP_ONLY=False,\n              INSTALL_DEPENDENCIES=False, ATTACK_DEVICE=\'cuda:0\', ATTENTION_DEVICE=\'cuda:0\', CAPTION_DEVICE=\'cuda:0\',\n              SAVED_TENSORS_ON_CPU=False, OFFLOAD_CLIP_DURING_QWEN=False,\n              RESUME_RUN_DIR=state[\'attack_runs\'].get(job[\'name\']), **overrides)\n    ns[\'validate_configuration\']()\n    os.environ[\'HF_HOME\'] = str(root / \'hf_cache\')\n    for index, source in cells[1:]:\n        exec(compile(source, f\'{plan["attack_notebook"]}:cell_{index}\', \'exec\'), ns)\n    ns[\'verify_devices_and_runtime\']()\n    if len(ns[\'prepare_pairs\']()) != plan[\'num_samples\']:\n        raise RuntimeError(\'Expected exactly 200 selected pairs.\')\n    backup = DriveBackup(backup_prefix, job[\'name\'])\n    probe = root / \'work\' / \'backup_probe.json\'\n    write_json(probe, {\'experiment\': plan[\'experiment\'], \'job\': job[\'name\'], \'check\': \'write_readback\'})\n    backup.put(probe, \'backup_probe.json\')\n    original_identity = ns[\'run_identity\']\n    driver_sha = digest(Path(__file__))\n    ns[\'run_identity\'] = lambda pairs: {**original_identity(pairs), \'campaign_driver_sha256\': driver_sha,\n                                       \'plan_sha256\': digest(root / \'molab_hypotheses_n200.json\')}\n    if revisions is not None:\n        original_resolve = ns[\'resolve_revisions\']\n        ns[\'resolve_revisions\'] = lambda ids, existing=None: original_resolve(ids, existing or {k: revisions[k] for k in ids})\n    original_write = ns[\'write_json\']\n    original_run_one = ns[\'run_one\']\n    deadline = time.monotonic() + max_hours * 3600\n    text_writes = 0\n    target_variant = os.environ.get(\'KLTN_TARGET_VARIANT\')\n\n    def checkpoint_write(path, value):\n        nonlocal text_writes\n        original_write(path, value)\n        path = Path(path)\n        run = ns.get(\'RUN_DIR\')\n        if path.name == \'run_config.json\' and run is not None and path.parent == Path(run):\n            state[\'attack_runs\'][job[\'name\']] = str(run)\n            write_json(state_path, state)\n            seed = root / \'provenance/seed_text_records.json\'\n            if seed.is_file() and not (Path(run) / \'text_records.json\').exists():\n                ns[\'write_json_original\'](Path(run) / \'text_records.json\', read_json(seed))\n            backup.put(state_path, \'campaign_state.json\')\n            backup.snapshot(run)\n        elif run is not None and path.name == \'complete.json\':\n            backup.snapshot(run, path.parent, metadata=False)\n            print(f\'DURABLE SAMPLE: {path.parent.relative_to(run)}\', flush=True)\n\n        elif run is not None and path.name == \'status.json\' and path.parent.name in job[\'variants\']:\n            backup.snapshot(run)\n            if path.parent.name == target_variant and value.get(\'status\') == \'complete\':\n                raise BoundaryPause(\'Configuration complete; evaluate before the next ablation.\')\n            if time.monotonic() >= deadline:\n                raise BoundaryPause(\'Time limit reached after images, state and manifest were flushed.\')\n        elif run is not None and path.name == \'text_records.json\':\n            text_writes += 1\n            if text_writes % 5 == 0 or time.monotonic() >= deadline:\n                backup.snapshot(run)\n            if time.monotonic() >= deadline:\n                raise BoundaryPause(\'Time limit reached after saving text preparation progress.\')\n\n    def boundary_run_one(*args, **kwargs):\n        if time.monotonic() >= deadline:\n            raise BoundaryPause(\'Time limit reached before starting another sample.\')\n        return original_run_one(*args, **kwargs)\n\n    ns[\'write_json_original\'] = original_write\n    ns[\'write_json\'], ns[\'run_one\'], ns[\'RUN_ATTACK\'] = checkpoint_write, boundary_run_one, True\n    try:\n        run = ns[\'run_experiment\']()\n    except BoundaryPause as exc:\n        run = ns.get(\'RUN_DIR\')\n        if run:\n            backup.snapshot(run)\n        print(f\'PAUSED AT CHECKPOINT: {exc}. Restore and rerun with identical configuration.\', flush=True)\n        return 75\n    finally:\n        run = ns.get(\'RUN_DIR\')\n        if run and Path(run).exists():\n            backup.snapshot(run)\n    status = read_json(Path(run) / \'status.json\')\n    if status[\'status\'] != \'complete\':\n        raise RuntimeError(f\'Attack incomplete: inspect {run}/summary.json and variant errors.jsonl.\')\n    return 0\n\n'
EVALUATION_RUNTIME_SOURCE = '"""Evaluate completed attack configurations without waiting for the full campaign.\n\nUses the unchanged text_gen_eval_v3.1.ipynb protocol in an isolated interpreter.\nThis runner never writes into the attack run or its campaign_state.json.\n"""\nimport argparse\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\nimport shutil\nimport subprocess\nimport sys\nimport threading\nimport time\nfrom zipfile import ZipFile, ZIP_DEFLATED\n\nVERSION = \'molab_completed_eval_v1\'\n# Audited compatibility: the first release differs only in quota/deadline\n# orchestration. Caption/judge protocols, manifests and model locks are unchanged.\nCOMPATIBLE_ENGINE_HASHES = {\'cc9de38eb2debb89c991bf4d1cd20d5db8ee9adf9899952cfe54a5efda30cc5a\'}\nSOURCE_PREFIX = \'KLTN_Molab_Hypotheses_n200_v432_h1/results\'\nEVAL_PREFIX = \'KLTN_Molab_Hypotheses_n200_v432_h1/evaluation_completed_v1\'\nPINS = [\'transformers==4.57.6\', \'accelerate==1.10.1\', \'bitsandbytes==0.48.1\',\n        \'openai==1.109.1\', \'tenacity==9.0.0\', \'tqdm==4.67.1\',\n        \'sentencepiece==0.2.1\', \'Pillow\', \'ipython\', \'gdrive-fsspec\', \'python-dotenv\']\n\n\nclass SafePause(BaseException):\n    pass\n\n\nclass QuotaPauseError(RuntimeError):\n    pass\n\n\ndef install_quota_guard(ns, deadline, pause_file):\n    """Abort waiting workers cooperatively; do not shorten provider cooldowns."""\n    limiter = ns[\'JUDGE_LIMITER\']\n    cancelled = threading.Event()\n    guard_lock = threading.Lock()\n    original_cooldown = limiter.cooldown\n    original_fatal = ns[\'judge_fatal\']\n\n    def cooldown(duration, *, count_429=False):\n        original_cooldown(duration, count_429=count_429)\n        if duration > 120 or time.monotonic() + duration >= deadline:\n            with guard_lock:\n                if not cancelled.is_set():\n                    write_json(pause_file, {\'reason\': \'provider_cooldown\', \'http_429\': count_429,\n                                           \'retry_after_seconds\': duration,\n                                           \'not_before_unix\': time.time() + duration})\n                    cancelled.set()\n                    print(f\'QUOTA PAUSE: provider requires {duration:.1f}s. Save and resume later; no new requests.\', flush=True)\n\n    def acquire(predicted_tokens):\n        predicted_tokens = int(predicted_tokens)\n        if predicted_tokens > limiter.tpm:\n            raise ValueError(\'One request exceeds JUDGE_TPM\')\n        while True:\n            if cancelled.is_set() or time.monotonic() >= deadline:\n                raise QuotaPauseError(\'Judge paused for quota/time budget; successful scores remain resumable\')\n            with limiter.lock:\n                now = time.monotonic()\n                while limiter.events and now - limiter.events[0][0] >= 60:\n                    limiter.events.popleft()\n                delay = max(0., limiter.cooldown_until - now, limiter.next_start - now)\n                if len(limiter.events) >= limiter.rpm:\n                    delay = max(delay, 60 - (now - limiter.events[0][0]) + 0.05)\n                used = sum(item[1] for item in limiter.events)\n                if used + predicted_tokens > limiter.tpm:\n                    reclaimed = 0\n                    for stamp, tokens in limiter.events:\n                        reclaimed += tokens\n                        if used - reclaimed + predicted_tokens <= limiter.tpm:\n                            delay = max(delay, 60 - (now - stamp) + 0.05)\n                            break\n                if delay <= 0:\n                    limiter.events.append((now, predicted_tokens))\n                    limiter.next_start = now + 60 / limiter.rpm\n                    limiter.requests_reserved += 1\n                    return\n            cancelled.wait(min(5., max(0.02, delay)))\n\n    limiter.cooldown = cooldown\n    limiter.acquire = acquire\n    ns[\'judge_fatal\'] = lambda error: isinstance(error, QuotaPauseError) or original_fatal(error)\n    return cancelled\n\n\ndef read_json(path):\n    return json.loads(Path(path).read_text(encoding=\'utf-8\'))\n\n\ndef write_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_name(path.name + \'.tmp\')\n    tmp.write_text(json.dumps(value, ensure_ascii=False, indent=2) + \'\\n\', encoding=\'utf-8\')\n    tmp.replace(path)\n\n\ndef sha(path):\n    h = hashlib.sha256()\n    with Path(path).open(\'rb\') as stream:\n        for block in iter(lambda: stream.read(1024 * 1024), b\'\'):\n            h.update(block)\n    return h.hexdigest()\n\n\ndef rows(path):\n    return [json.loads(line) for line in Path(path).read_text(encoding=\'utf-8\').splitlines() if line.strip()]\n\n\ndef contained(root, relative):\n    if not isinstance(relative, str) or \'\\\\\' in relative or \':\' in relative:\n        raise ValueError(\'Invalid relative artifact path\')\n    root = Path(root).resolve()\n    target = (root / relative).resolve()\n    if Path(relative).is_absolute() or not target.is_relative_to(root):\n        raise ValueError(\'Artifact path escapes its root\')\n    return target\n\n\ndef configuration_jobs(plan):\n    return [{\'id\': job[\'name\'] + \'__\' + variant, \'job\': job[\'name\'], \'variant\': variant}\n            for job in plan[\'attack_jobs\'] for variant in job[\'variants\']]\n\n\ndef drive_fs():\n    from gdrive_fsspec import GoogleDriveFileSystem\n    return GoogleDriveFileSystem(token=\'cache\', skip_instance_cache=True)\n\n\ndef verified_upload(fs, local, remote):\n    fs.makedirs(remote.rsplit(\'/\', 1)[0], exist_ok=True)\n    fs.put_file(str(local), remote)\n    h = hashlib.sha256()\n    with fs.open(remote, \'rb\') as stream:\n        for block in iter(lambda: stream.read(1024 * 1024), b\'\'):\n            h.update(block)\n    if h.hexdigest() != sha(local):\n        raise RuntimeError(\'Drive backup readback checksum mismatch\')\n\n\ndef unpack_metadata(archive, target):\n    with ZipFile(archive) as z:\n        if z.testzip() is not None:\n            raise ValueError(\'Corrupt metadata ZIP\')\n        for item in z.infolist():\n            path = contained(target, item.filename)\n            if item.is_dir():\n                continue\n            # Root snapshots contain configuration manifests, text and status.\n            if \'samples\' in Path(item.filename).parts:\n                continue\n            path.parent.mkdir(parents=True, exist_ok=True)\n            path.write_bytes(z.read(item))\n\n\ndef setup(root):\n    exe = root / \'.venv\' / (\'Scripts/python.exe\' if os.name == \'nt\' else \'bin/python\')\n    if not exe.exists():\n        subprocess.run([sys.executable, \'-m\', \'venv\', \'--system-site-packages\', str(root / \'.venv\')], check=True)\n    subprocess.run([str(exe), \'-m\', \'pip\', \'install\', *PINS], check=True)\n    return exe\n\n\ndef validate_manifest(root, expected=200):\n    manifest = root / \'manifest.jsonl\'\n    records = rows(manifest)\n    indices = [r[\'index\'] for r in records]\n    if len(records) != expected or indices != list(range(expected)):\n        raise ValueError(f\'Expected the full ordered cohort of {expected} pairs; got {len(records)}\')\n    fingerprints = {r[\'run_fingerprint\'] for r in records}\n    if len(fingerprints) != 1:\n        raise ValueError(\'Mixed attack identities\')\n    for row in records:\n        if row[\'sample_id\'] != f"{row[\'index\']:05d}":\n            raise ValueError(\'Inconsistent sample ID\')\n        for role in (\'source\', \'target\', \'adversarial\'):\n            p = contained(root, row[role])\n            if not p.is_file() or sha(p) != row[\'image_sha256\'][role]:\n                raise ValueError(f"Missing/changed image: {row[\'sample_id\']} / {role}")\n        if row.get(\'saved_linf_255\', float(\'inf\')) > row.get(\'epsilon_255\', -1):\n            raise ValueError(\'Saved perturbation exceeds attack budget\')\n        complete = read_json(root / \'samples\' / row[\'sample_id\'] / \'complete.json\')\n        for field in (\'index\', \'sample_id\', \'run_fingerprint\', \'image_sha256\', \'source\', \'target\', \'adversarial\'):\n            if complete.get(field) != row.get(field):\n                raise ValueError(f\'Completion marker differs at {field}\')\n    return records\n\n\ndef restore_sample(fs, remote, local_zip, destination, item, variant):\n    fs.get_file(remote, str(local_zip))\n    with ZipFile(local_zip) as z:\n        if z.testzip() is not None:\n            raise ValueError(\'Corrupt sample ZIP\')\n        prefix = variant + \'/\'\n        allowed = {item[role] for role in (\'source\', \'target\', \'adversarial\')}\n        allowed.add(\'samples/\' + item[\'sample_id\'] + \'/complete.json\')\n        members = {name[len(prefix):]: name for name in z.namelist() if name.startswith(prefix)}\n        for relative in allowed:\n            target = contained(destination, relative)\n            if relative not in members:\n                raise ValueError(\'Sample ZIP lacks a required evaluation artifact\')\n            target.parent.mkdir(parents=True, exist_ok=True)\n            target.write_bytes(z.read(members[relative]))\n    local_zip.unlink()\n\n\ndef prepare(root, source_root=None, source_prefix=SOURCE_PREFIX, only=None):\n    plan = read_json(root / \'molab_hypotheses_n200.json\')\n    fs = None\n    local_state = source_root / \'campaign_state.json\' if source_root else None\n    state = read_json(local_state) if local_state and local_state.is_file() else {}\n    metadata_roots = {}\n    inventory, ready = [], []\n    for config in configuration_jobs(plan):\n        if only and config[\'id\'] not in only:\n            continue\n        job, variant = config[\'job\'], config[\'variant\']\n        local_run = Path(state.get(\'attack_runs\', {}).get(job, \'/__missing_attack_run__\'))\n        metadata = local_run\n        if not (metadata / variant / \'manifest.jsonl\').is_file():\n            if job not in metadata_roots:\n                if fs is None:\n                    fs = drive_fs()\n                remote = source_prefix + \'/\' + job + \'/root_latest.zip\'\n                if not fs.exists(remote):\n                    metadata_roots[job] = None\n                else:\n                    metadata = root / \'source_metadata\' / job\n                    archive = root / (job + \'_metadata.zip\')\n                    fs.get_file(remote, str(archive))\n                    unpack_metadata(archive, metadata)\n                    archive.unlink()\n                    metadata_roots[job] = metadata\n            metadata = metadata_roots[job]\n        manifest = metadata / variant / \'manifest.jsonl\' if metadata else None\n        records = rows(manifest) if manifest and manifest.is_file() else []\n        entry = {**config, \'completed_samples\': len(records), \'expected_samples\': plan[\'num_samples\'],\n                 \'eligible\': len(records) == plan[\'num_samples\']}\n        inventory.append(entry)\n        print(f"{config[\'id\']}: {len(records)}/{plan[\'num_samples\']}", flush=True)\n        if not entry[\'eligible\']:\n            continue\n        destination = root / \'inputs\' / config[\'id\']\n        destination.mkdir(parents=True, exist_ok=True)\n        frozen_manifest = destination / \'manifest.jsonl\'\n        if frozen_manifest.is_file() and sha(frozen_manifest) != sha(manifest):\n            raise ValueError(\'Frozen evaluation input changed; use a new evaluation root\')\n        for item in records:\n            roles = [item[r] for r in (\'source\', \'target\', \'adversarial\')]\n            marker = \'samples/\' + item[\'sample_id\'] + \'/complete.json\'\n            if all(contained(destination, r).is_file() for r in [*roles, marker]):\n                # Full checksums are verified below, not inferred from file presence.\n                continue\n            source_variant = local_run / variant\n            if all(contained(source_variant, r).is_file() for r in [*roles, marker]):\n                for relative in [*roles, marker]:\n                    target = contained(destination, relative)\n                    target.parent.mkdir(parents=True, exist_ok=True)\n                    shutil.copy2(contained(source_variant, relative), target)\n            else:\n                if fs is None:\n                    fs = drive_fs()\n                remote = f"{source_prefix}/{job}/{variant}/samples/{item[\'sample_id\']}.zip"\n                restore_sample(fs, remote, root / \'sample_download.zip\', destination, item, variant)\n        shutil.copy2(manifest, frozen_manifest)\n        if (metadata / \'run_config.json\').is_file():\n            shutil.copy2(metadata / \'run_config.json\', destination / \'run_config.json\')\n        validate_manifest(destination, plan[\'num_samples\'])\n        ready.append({**config, \'input_dir\': str(destination), \'manifest_sha256\': sha(frozen_manifest),\n                      \'samples\': len(records)})\n    write_json(root / \'inventory.json\', inventory)\n    write_json(root / \'ready_configurations.json\', ready)\n    print(f\'READY: {len(ready)} complete configurations; partial configurations excluded.\', flush=True)\n    return ready\n\n\ndef backup_task(root, state, task, fs):\n    run = Path(state[\'tasks\'][task])\n    archive = root / \'eval_backup.zip\'\n    with ZipFile(archive, \'w\', ZIP_DEFLATED) as z:\n        for p in sorted(run.rglob(\'*\')):\n            if p.is_file() and not p.name.endswith(\'.tmp\'):\n                z.write(p, p.relative_to(run).as_posix())\n    prefix = state[\'backup_prefix\'] + \'/\' + task\n    # Pointer is published only after a verified archive upload.\n    archive_name = \'checkpoint_\' + sha(archive) + \'.zip\'\n    verified_upload(fs, archive, prefix + \'/\' + archive_name)\n    pointer = root / \'eval_pointer.json\'\n    write_json(pointer, {\'version\': VERSION, \'task\': task, \'run\': str(run),\n                         \'archive\': archive_name, \'archive_sha256\': sha(archive),\n                         \'model_revisions\': state[\'model_revisions\']})\n    verified_upload(fs, pointer, prefix + \'/pointer.json\')\n\n\ndef restore_evaluations(root, backup_prefix=EVAL_PREFIX):\n    fs = drive_fs()\n    state_path = root / \'evaluation_state.json\'\n    state = read_json(state_path) if state_path.is_file() else {\'tasks\': {}, \'model_revisions\': {}}\n    state[\'backup_prefix\'] = backup_prefix\n    for config in read_json(root / \'ready_configurations.json\'):\n        for model in read_json(root / \'molab_hypotheses_n200.json\')[\'evaluation\'][\'models\']:\n            task = config[\'id\'] + \'__\' + model\n            prefix = backup_prefix + \'/\' + task\n            if not fs.exists(prefix + \'/pointer.json\'):\n                continue\n            with fs.open(prefix + \'/pointer.json\', \'rb\') as stream:\n                pointer = json.load(stream)\n            run = Path(pointer[\'run\']).resolve()\n            if not run.is_relative_to(root):\n                raise ValueError(\'Restore requires the original absolute evaluation root\')\n            archive = root / \'restore_eval.zip\'\n            archive_name = pointer[\'archive\']\n            if \'/\' in archive_name or \'\\\\\' in archive_name or not archive_name.startswith(\'checkpoint_\'):\n                raise ValueError(\'Unsafe remote evaluation archive name\')\n            fs.get_file(prefix + \'/\' + archive_name, str(archive))\n            if sha(archive) != pointer[\'archive_sha256\']:\n                raise ValueError(\'Evaluation archive differs from its durable pointer\')\n            with ZipFile(archive) as z:\n                if z.testzip() is not None:\n                    raise ValueError(\'Corrupt evaluation ZIP\')\n                for item in z.infolist():\n                    target = contained(run, item.filename)\n                    if not item.is_dir():\n                        target.parent.mkdir(parents=True, exist_ok=True)\n                        target.write_bytes(z.read(item))\n            for model_id, revision in pointer[\'model_revisions\'].items():\n                if model_id in state[\'model_revisions\'] and state[\'model_revisions\'][model_id] != revision:\n                    raise ValueError(\'Conflicting victim model revision locks\')\n                state[\'model_revisions\'][model_id] = revision\n            state[\'tasks\'][task] = str(run)\n            print(\'RESTORED\', task, flush=True)\n    write_json(state_path, state)\n\n\ndef caption_signature(config):\n    return {key: config[key] for key in (\'model\', \'caption\', \'roles\', \'seed\')}\n\n\ndef cached_items(root, state, signature, wanted):\n    cached = {}\n    for directory in state[\'tasks\'].values():\n        run = Path(directory)\n        if not (run / \'run_config.json\').is_file():\n            continue\n        prior = read_json(run / \'run_config.json\')[\'caption_config\']\n        if caption_signature(prior) != signature:\n            continue\n        if (run / \'caption_items.jsonl\').is_file():\n            # Only complete JSONL rows; never repair another task\'s checkpoint.\n            for line in (run / \'caption_items.jsonl\').read_bytes().splitlines(keepends=True):\n                if not line.endswith(b\'\\n\'):\n                    continue\n                row = json.loads(line)\n                if row.get(\'sha256\') in wanted and row.get(\'status\') == \'ok\':\n                    cached.setdefault(row[\'sha256\'], row)\n    return cached\n\n\ndef worker(root, config_id, model, batch_size=32, judge=True, max_hours=8, backup=True):\n    plan = read_json(root / \'molab_hypotheses_n200.json\')\n    config = next(c for c in read_json(root / \'ready_configurations.json\') if c[\'id\'] == config_id)\n    input_dir = Path(config[\'input_dir\'])\n    validate_manifest(input_dir, plan[\'num_samples\'])\n    if sha(input_dir / \'manifest.jsonl\') != config[\'manifest_sha256\']:\n        raise ValueError(\'Evaluation input manifest changed\')\n    state_path = root / \'evaluation_state.json\'\n    state = read_json(state_path) if state_path.is_file() else {\'tasks\': {}, \'model_revisions\': {}}\n    state.setdefault(\'backup_prefix\', EVAL_PREFIX)\n    task = config_id + \'__\' + model\n    ns = {\'__name__\': \'__main__\', \'__file__\': str(root / \'text_gen_eval_v3.1.ipynb\')}\n    notebook = read_json(root / \'text_gen_eval_v3.1.ipynb\')\n    cells = {i: \'\'.join(c[\'source\']) for i, c in enumerate(notebook[\'cells\']) if c[\'cell_type\'] == \'code\'}\n    exec(compile(cells[2], \'evaluation_v3.1:config\', \'exec\'), ns)\n    ns.update(INPUT_DIR=input_dir, OUTPUT_DIR=root / \'runs\' / task, NUM_SAMPLES=plan[\'num_samples\'],\n              RUN_PIPELINE=True, RUN_EVALUATION=judge, MODEL=model, SEED=plan[\'seed\'],\n              GPU_MODE=\'single\', MAX_GPU_REPLICAS=1, LOCAL_BATCH_SIZE=batch_size,\n              LOCAL_MODEL=True, JUDGE_WORKERS=4, JUDGE_RPM=plan[\'evaluation\'][\'judge_rpm\'],\n              JUDGE_TPM=plan[\'evaluation\'][\'judge_tpm\'], JUDGE_MODEL=\'gpt-4o\',\n              SUCCESS_THRESHOLD=0.3, SUCCESS_OPERATOR=\'>=\',\n              EXPERIMENT_NAME=VERSION, RESUME_CAPTION_RUN_DIR=state[\'tasks\'].get(task))\n    os.environ[\'HF_HOME\'] = str(root / \'hf_cache\')\n    fs = drive_fs() if backup else None\n    if backup:\n        probe = root / \'backup_probe.json\'\n        write_json(probe, {\'runner\': VERSION, \'check\': \'upload_readback\'})\n        verified_upload(fs, probe, state[\'backup_prefix\'] + \'/backup_probe.json\')\n    deadline = time.monotonic() + max_hours * 3600\n    last_backup = 0.0\n    original_write = None\n\n    def checkpoint(path, value):\n        nonlocal last_backup\n        original_write(path, value)\n        run = ns.get(\'CAPTION_RUN_DIR\')\n        if not run:\n            return\n        state[\'tasks\'][task] = str(run)\n        write_json(state_path, state)\n        path = Path(path)\n        if path.name == \'run_config.json\' and path.parent == Path(run):\n            wanted = {digest for s in ns[\'selected_samples\'] for digest in s[\'image_sha256\'].values()}\n            cache_path = Path(run) / \'caption_items.jsonl\'\n            if not cache_path.exists():\n                cache = cached_items(root, state, caption_signature(ns[\'CAPTION_CONFIG\']), wanted)\n                if cache:\n                    cache_path.write_text(\'\'.join(json.dumps(row, ensure_ascii=False) + \'\\n\'\n                                                  for row in cache.values()), encoding=\'utf-8\')\n                    print(f\'REUSED {len(cache)} identical-image captions from the same victim protocol.\', flush=True)\n        expired = time.monotonic() >= deadline\n        if backup and (time.monotonic() - last_backup >= 120 or expired):\n            backup_task(root, state, task, fs)\n            last_backup = time.monotonic()\n        if expired and path.name == \'status.json\':\n            raise SafePause(\'Time budget reached after a flushed caption/judge checkpoint\')\n\n    try:\n        for index in (3, 4, 5, 6, 7, 9, 10, 11):\n            if index == 10 and ns.get(\'caption_status\', {}).get(\'status\') != \'complete\':\n                print(\'Captions incomplete; resume before judging the full cohort.\', flush=True)\n                return 75\n            if index == 10 and not judge:\n                break\n            exec(compile(cells[index], f\'evaluation_v3.1:cell_{index}\', \'exec\'), ns)\n            if index == 4:\n                from huggingface_hub import HfApi\n                model_id = ns[\'cfg\'].caption.model_id\n                locks = state[\'model_revisions\']\n                if model_id not in locks:\n                    locks[model_id] = HfApi().model_info(model_id, revision=\'main\').sha\n                ns[\'cfg\'].caption.revision = locks[model_id]\n                # cfg.caption is copied into CAPTION_CONFIG in the original notebook.\n                ns[\'CAPTION_CONFIG\'][\'caption\'][\'revision\'] = locks[model_id]\n                ns[\'CAPTION_CONFIG\'][\'evaluation_notebook_sha256\'] = sha(root / \'text_gen_eval_v3.1.ipynb\')\n                ns[\'CAPTION_CONFIG\'][\'runner_version\'] = VERSION\n                engine_hash = sha(__file__)\n                effective_hash = engine_hash\n                if state[\'tasks\'].get(task):\n                    prior_config = read_json(Path(state[\'tasks\'][task]) / \'run_config.json\')[\'caption_config\']\n                    prior_hash = prior_config.get(\'runner_sha256\')\n                    if prior_hash in COMPATIBLE_ENGINE_HASHES:\n                        effective_hash = prior_hash\n                ns[\'CAPTION_CONFIG\'][\'runner_sha256\'] = effective_hash\n                ns[\'CAPTION_CONFIG_SHA256\'] = hashlib.sha256(\n                    json.dumps(ns[\'CAPTION_CONFIG\'], sort_keys=True).encode()).hexdigest()\n                write_json(state_path, state)\n                if state[\'tasks\'].get(task):\n                    write_json(Path(state[\'tasks\'][task]) / \'execution_engine.json\', {\n                        \'engine_sha256\': engine_hash, \'protocol_engine_sha256\': effective_hash,\n                        \'compatibility\': \'quota/deadline guard only; all other fingerprint fields still verified\'})\n            if index == 5:\n                original_write = ns[\'write_json_atomic\']\n                ns[\'write_json_atomic\'] = checkpoint\n            if index == 7:\n                install_quota_guard(ns, deadline, root / \'judge_quota_pause.json\')\n        if judge and ns.get(\'evaluation_status\', {}).get(\'status\') != \'complete\':\n            return 75\n        return 0\n    except SafePause as exc:\n        print(\'EVALUATION PAUSED SAFELY:\', str(exc), flush=True)\n        return 75\n    finally:\n        if ns.get(\'CAPTION_RUN_DIR\') and backup:\n            backup_task(root, state, task, fs)\n\n\ndef report(root):\n    state_path = root / \'evaluation_state.json\'\n    state = read_json(state_path) if state_path.exists() else {\'tasks\': {}}\n    summary = []\n    for task, run in state[\'tasks\'].items():\n        run = Path(run)\n        caption = read_json(run / \'status.json\') if (run / \'status.json\').exists() else {}\n        scored = read_json(run / \'evaluation\' / \'summary.json\') if (run / \'evaluation\' / \'summary.json\').exists() else {}\n        checkpoint = run / \'evaluation\' / \'scores.jsonl\'\n        if checkpoint.exists():\n            latest = {}\n            for line in checkpoint.read_bytes().splitlines(keepends=True):\n                if line.endswith(b\'\\n\'):\n                    row = json.loads(line)\n                    latest[row[\'sample_id\']] = row\n            valid = [r for r in latest.values() if r.get(\'status\') == \'ok\']\n            n = caption.get(\'selected_samples\', 200)\n            if len(valid) != scored.get(\'total_evaluated\'):\n                scored = {\'status\': \'partial\', \'total_evaluated\': len(valid), \'coverage\': len(valid) / n,\n                          \'full_cohort_success_rate\': None,\n                          \'average_similarity_score\': sum(r[\'score\'] for r in valid) / len(valid) if valid else None}\n        summary.append({\'task\': task, \'caption_status\': caption.get(\'status\'),\n                        \'ready_pairs\': caption.get(\'completed_samples\', 0),\n                        \'judge_status\': scored.get(\'status\', \'pending\'),\n                        \'judged_pairs\': scored.get(\'total_evaluated\', 0),\n                        \'coverage\': scored.get(\'coverage\', 0),\n                        \'ASR\': scored.get(\'full_cohort_success_rate\'),\n                        \'AvgSim\': scored.get(\'average_similarity_score\'), \'run\': str(run)})\n    write_json(root / \'summary_completed.json\', summary)\n    print(json.dumps(summary, ensure_ascii=False, indent=2), flush=True)\n    return summary\n\n'
MOLAB_PLAN = {'experiment': 'kltn_v433_cfvlm_notebook_full_backward_n200', 'num_samples': 200, 'steps': 300, 'seed': 2023, 'epsilon_255': 16.0, 'alpha_255': 1.0, 'dataset_manifest': 'data/pairs_200.jsonl', 'attack_notebook': 'kltn-attack-v4.3.3-h1-cfvlm.ipynb', 'eval_notebook': 'text_gen_eval_v3.1.ipynb', 'attack_model_revisions': {'OpenGVLab/InternVL3-1B-hf': '014c0583a0d4bedf29fbe2dbff4f865eb998e171', 'Qwen/Qwen3-VL-8B-Instruct': '0c351dd01ed87e9c1b53cbc748cba10e6187ff3b', 'laion/CLIP-ViT-G-14-laion2B-s12B-b42K': '4b0305adc6802b2632e11cbe6606a9bdd43d35c9', 'openai/clip-vit-base-patch16': '57c216476eefef5ab752ec549e440a49ae4ae5f3', 'openai/clip-vit-base-patch32': '3d74acf9a28c67741b2f4f2ea7635f0aaf6f0268'}, 'attack_jobs': [{'name': 'primary_ablations', 'variants': ['hybrid_plain', 'hybrid_negative', 'hybrid_fixed', 'hybrid_dynamic', 'hybrid_random_region', 'hybrid_random_neg'], 'overrides': {}}, {'name': 'h4_mpc_zero', 'variants': ['hybrid_plain'], 'overrides': {'LOSS_WEIGHTS': {'foa': 1.0, 'mpc': 0.0, 'negative': 0.5}}, 'reuse_primary_captions': True}, {'name': 'h4_image_only', 'variants': ['hybrid_plain'], 'overrides': {'FUSION_LAMBDA': 1.0}, 'reuse_primary_captions': True}], 'evaluation': {'models': ['LLaVA-1.5-7B', 'InternVL3-8B'], 'diagnostic_surrogate_model': 'Qwen3-VL-8B', 'run_surrogate_diagnostic': False, 'success_threshold': 0.3, 'success_operator': '>=', 'judge_model': 'gpt-4o', 'judge_rpm': 12, 'judge_tpm': 8000, 'local_batch_size': 64}, 'comparisons': [{'hypothesis': 'H1_negative_loss', 'control': 'primary_ablations/hybrid_plain', 'treatment': 'primary_ablations/hybrid_negative'}, {'hypothesis': 'H1_hard_mining', 'control': 'primary_ablations/hybrid_random_neg', 'treatment': 'primary_ablations/hybrid_dynamic', 'condition': 'Report all pairs and the prespecified bank_size > 2 subset; do not silently drop bank_size == 2 pairs.'}, {'hypothesis': 'H2_attention_location', 'control': 'primary_ablations/hybrid_random_region', 'treatment': 'primary_ablations/hybrid_dynamic'}, {'hypothesis': 'H3_refresh', 'control': 'primary_ablations/hybrid_fixed', 'treatment': 'primary_ablations/hybrid_dynamic'}, {'hypothesis': 'H4_mpc_term', 'control': 'h4_mpc_zero/hybrid_plain', 'treatment': 'primary_ablations/hybrid_plain'}, {'hypothesis': 'H4_image_text_fusion', 'control': 'h4_image_only/hybrid_plain', 'treatment': 'primary_ablations/hybrid_plain'}], 'notes': ['Eight remaining configurations, each with the original 200 ordered image pairs and 300 steps.', 'mpc_clip is not full MPC. Optional mpc_full_reference is outside this initial campaign.', 'Qwen attention checkpoint is a surrogate diagnostic, not an independent held-out victim.', 'InternVL3-8B is a different checkpoint from the InternVL3-1B captioner but belongs to the same model family; report this explicitly.', 'One seed is an initial paired experiment, not evidence of across-seed robustness.', 'Keep failures and report completion/judging coverage. Compute paired differences on common valid pair IDs.', 'Use grouped uncertainty estimates for repeated source/target images, based on original image hashes.']}
import ast
import hashlib
import json
import signal
import subprocess
import sys
import time
from types import ModuleType
from zipfile import ZipFile, ZIP_DEFLATED
import shutil


def molab_read(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))


def molab_write(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
    temporary.replace(path)


def molab_sha(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()


def molab_module(name, source):
    module = ModuleType(name)
    module.__file__ = str(Path(__file__).resolve())
    exec(compile(source, str(__file__) + ':' + name, 'exec'), module.__dict__)
    return module


def molab_drive():
    from gdrive_fsspec import GoogleDriveFileSystem
    return GoogleDriveFileSystem(token='cache', use_listings_cache=False, skip_instance_cache=True)


def molab_extract(archive, destination):
    destination = Path(destination).resolve()
    with ZipFile(archive) as z:
        if z.testzip():
            raise ValueError('ZIP CRC failure')
        for member in z.infolist():
            if not (destination / member.filename).resolve().is_relative_to(destination):
                raise ValueError('Unsafe archive member')
        z.extractall(destination)


def molab_interpreter(stage):
    root = MOLAB_ROOT / ('attack' if stage == 'attack' else 'evaluation')
    own = root / ('.venvs/attack/bin/python' if stage == 'attack' else '.venv/bin/python')
    if own.exists():
        return own
    previous = Path('/marimo/cfvlm_n200_v2/.venvs/attack/bin/python' if stage == 'attack' else '/marimo/eval_cfvlm_n200_v2/.venv/bin/python')
    if previous.exists():
        return previous
    env = own.parent.parent
    subprocess.run([sys.executable, '-m', 'venv', '--system-site-packages', str(env)], check=True)
    if stage == 'attack':
        cells = molab_read(__file__)['cells']
        tree = ast.parse(next(''.join(c['source']) for c in cells if c['cell_type'] == 'code' and 'PINNED_PACKAGES = [' in ''.join(c['source']) and 'attack' in c.get('metadata', {}).get('tags', [])))
        pins = next(ast.literal_eval(n.value) for n in ast.walk(tree) if isinstance(n, ast.Assign) and any(isinstance(t, ast.Name) and t.id == 'PINNED_PACKAGES' for t in n.targets))
        pins += ['numpy', 'Pillow', 'gdrive-fsspec', 'python-dotenv']
    else:
        pins = molab_module('embedded_eval', EVALUATION_RUNTIME_SOURCE).PINS
    subprocess.run([str(own), '-m', 'pip', 'install', *pins], check=True)
    return own


def molab_initialize():
    attack, evaluation = MOLAB_ROOT / 'attack', MOLAB_ROOT / 'evaluation'
    for root in (attack, evaluation):
        root.mkdir(parents=True, exist_ok=True)
    fs = molab_drive()
    attack_code = attack / MOLAB_PLAN['attack_notebook']
    if attack_code.exists() and molab_sha(attack_code) != molab_sha(__file__):
        raise ValueError('Notebook changed. Use a new MOLAB_ROOT and Drive namespace, never mix attack fingerprints.')
    shutil.copyfile(__file__, attack_code)
    eval_code = Path(__file__).parent / 'text_gen_eval_v3.1.ipynb'
    destination = evaluation / eval_code.name
    if destination.exists() and molab_sha(destination) != molab_sha(eval_code):
        raise ValueError('Evaluation protocol changed')
    shutil.copyfile(eval_code, destination)
    provenance = attack / 'provenance'
    provenance.mkdir(exist_ok=True)
    legacy = provenance / 'legacy_primary_root.zip'
    if not legacy.exists():
        prior = Path('/marimo/cfvlm_n200_v2/provenance/legacy_primary_root.zip')
        if prior.exists(): shutil.copyfile(prior, legacy)
        else: fs.get_file(MOLAB_DRIVE_PARENT + '/results/primary_ablations/root_latest.zip', str(legacy))
    with ZipFile(legacy) as z:
        previous = json.loads(z.read('run_config.json'))
        texts = json.loads(z.read('text_records.json'))
        for variant in ('foa_reference', 'mpc_clip'):
            status = json.loads(z.read(variant + '/status.json'))
            rows = [json.loads(line) for line in z.read(variant + '/manifest.jsonl').decode().splitlines() if line.strip()]
            if status.get('status') != 'complete' or [r['index'] for r in rows] != list(range(200)):
                raise ValueError('Original baseline cohort is not complete')
    if len(texts) != 200 or any(not r.get('source') or not r.get('target') or r.get('caption_error') for r in texts.values()):
        raise ValueError('Require all 200 original factual caption pairs')
    if any(previous['model_revisions'].get(k) != v for k, v in MOLAB_PLAN['attack_model_revisions'].items()):
        raise ValueError('Attack model revision mismatch')
    factual = {sid: {k:r[k] for k in ('source', 'target')} for sid,r in texts.items()}
    molab_write(provenance / 'factual_captions.json', factual)
    seed = provenance / 'seed_text_records.json'
    if not seed.exists():
        candidates = sorted(Path('/marimo/cfvlm_n200_v2/work/primary_ablations').glob('*/text_records.json'))
        remote_seed = MOLAB_DRIVE_PREFIX + '/code/seed_text_records.json'
        if fs.exists(remote_seed):
            fs.get_file(remote_seed, str(seed))
        elif candidates:
            # Existing CF-VLM text bank is revalidated by the notebook policy;
            # attack outputs/history from the failed code are never imported.
            shutil.copyfile(candidates[-1], seed)
        else:
            molab_write(seed, texts)
    plan = json.loads(json.dumps(MOLAB_PLAN))
    for job in plan['attack_jobs']:
        job.pop('reuse_primary_captions', None)
        job['overrides']['MANUAL_TEXT_OVERRIDES'] = factual
    plan['text_provenance'] = {'legacy_metadata_sha256': molab_sha(legacy), 'seed_bank_sha256': molab_sha(seed), 'attack_samples_imported': 0}
    for root in (attack, evaluation):
        p = root / 'molab_hypotheses_n200.json'
        if p.exists() and molab_read(p) != plan:
            raise ValueError('Frozen campaign plan differs')
        molab_write(p, plan)
    archive = attack / 'attack_data_n200.zip'
    if not archive.exists():
        prior = Path('/marimo/cfvlm_n200_v2/attack_data_n200.zip')
        if prior.exists(): shutil.copyfile(prior, archive)
        else: fs.get_file(MOLAB_DRIVE_PARENT + '/attack_data_n200.zip', str(archive))
    if molab_sha(archive) != '3518b43ac953db8192a31698105cee4f277cc543351bf28e1eec6d7452987103':
        raise ValueError('Dataset checksum mismatch')
    if not (attack / 'data/pairs_200.jsonl').exists(): molab_extract(archive, attack / 'data')
    manifest = attack / 'data/pairs_200.jsonl'
    pairs = [json.loads(line) for line in manifest.read_text().splitlines() if line.strip()]
    hashes = molab_read(attack / 'data/data_provenance.json')['sha256']
    if [r['index'] for r in pairs] != list(range(200)): raise ValueError('Wrong cohort/order')
    for pair in pairs:
        for role in ('source', 'target'):
            p = (manifest.parent / pair[role]).resolve()
            if not p.is_relative_to(manifest.parent.resolve()) or molab_sha(p) != hashes[pair[role]]:
                raise ValueError('Input image differs from original cohort')
    for root in (attack, evaluation):
        cache = root / 'hf_cache'
        old = Path('/marimo/cfvlm_n200/hf_cache')
        if not cache.exists():
            if old.exists(): cache.symlink_to(old, target_is_directory=True)
            else: cache.mkdir()
    state = attack / 'campaign_state.json'
    if not state.exists():
        molab_restore_attack(attack, plan, fs)
    eval_state = evaluation / 'evaluation_state.json'
    if not eval_state.exists():
        locks = {}
        for model in plan['evaluation']['models']:
            with fs.open(MOLAB_DRIVE_PARENT + '/evaluation_completed_v1/primary_ablations__foa_reference__' + model + '/pointer.json','rb') as stream:
                pointer = json.load(stream)
            locks.update(pointer['model_revisions'])
        molab_write(eval_state, {'tasks':{}, 'model_revisions':locks, 'backup_prefix':MOLAB_DRIVE_PREFIX + '/evaluation'})
    backup = molab_module('embedded_attack', ATTACK_RUNTIME_SOURCE).DriveBackup(MOLAB_DRIVE_PREFIX, 'code')
    for path in (attack_code, destination, attack/'molab_hypotheses_n200.json', seed):
        backup.put(path, path.name)
    molab_interpreter('attack'); molab_interpreter('eval')
    print('VALIDATED: 200 original pairs; immutable model locks; notebook-only workers; Drive readback OK', flush=True)
    return plan


def molab_restore_attack(root, plan, fs):
    state = {'attack_runs':{},'evaluations':{}}
    for job in plan['attack_jobs']:
        prefix = MOLAB_DRIVE_PREFIX + '/results/' + job['name']
        if not fs.exists(prefix + '/campaign_state.json'): continue
        with fs.open(prefix + '/campaign_state.json','rb') as stream: saved=json.load(stream)
        run = Path(saved['attack_runs'][job['name']]).resolve()
        if not run.is_relative_to(root.resolve()): raise ValueError('Restore at the original absolute root')
        staging = root / 'restore.zip'
        for remote in fs.find(prefix):
            if remote.endswith('.zip'):
                fs.get_file(remote,str(staging)); molab_extract(staging,run)
        staging.unlink(missing_ok=True)
        state['attack_runs'][job['name']]=str(run)
    molab_write(root/'campaign_state.json',state)


# Child interpreters read code cells from this exact notebook. No .py runner files.
MOLAB_LOADER = "import json,sys; p=sys.argv[1]; n={'__file__':p,'__name__':'embedded_molab'}; b=json.load(open(p)); [exec(compile(''.join(c['source']),p+':runtime','exec'),n) for c in b['cells'] if c['cell_type']=='code' and 'molab_runtime' in c.get('metadata',{}).get('tags',[])]; sys.exit(n['molab_dispatch'](*sys.argv[2:]))"


def molab_completed(job, variant, smoke=False):
    state_path = MOLAB_ROOT / 'attack' / ('smoke_state.json' if smoke else 'campaign_state.json')
    if not state_path.exists(): return 0
    run = molab_read(state_path).get('attack_runs',{}).get(job)
    if not run: return 0
    p=Path(run)/variant/'manifest.jsonl'
    return len([line for line in p.read_text().splitlines() if line.strip()]) if p.exists() else 0


def molab_progress(stage, **details):
    path=MOLAB_ROOT/'progress.json'
    value={'stage':stage,'updated_unix':time.time(),'controller_pid':os.getpid(),**details}
    molab_write(path,value)
    backup=molab_module('embedded_attack',ATTACK_RUNTIME_SOURCE).DriveBackup(MOLAB_DRIVE_PREFIX,'control')
    backup.put(path,'progress.json')


def molab_call(stage, *args, deadline):
    remaining=(deadline-time.time())/3600
    if remaining<=0:return 75
    os.environ['KLTN_DEADLINE_UNIX']=str(deadline)
    log=MOLAB_ROOT/'logs'/('_'.join((stage,*args))+'.log')
    log.parent.mkdir(exist_ok=True)
    command=[str(molab_interpreter(stage)),'-u','-c',MOLAB_LOADER,str(Path(__file__).resolve()),stage,*args]
    with log.open('a',encoding='utf-8') as output:
        child=subprocess.Popen(command,stdout=output,stderr=subprocess.STDOUT,start_new_session=False)
        molab_progress(stage,task=list(args),worker_pid=child.pid,log=str(log))
        print('START',stage,*args,'PID',child.pid,flush=True)
        code=child.wait()
    print('END',stage,*args,'EXIT',code,flush=True)
    return code


def molab_dispatch(mode, *args):
    attack,evaluation=MOLAB_ROOT/'attack',MOLAB_ROOT/'evaluation'
    deadline=float(os.environ.get('KLTN_DEADLINE_UNIX',time.time()+MOLAB_MAX_HOURS*3600))
    if mode=='control':return molab_control(deadline)
    plan=molab_read(attack/'molab_hypotheses_n200.json')
    if mode=='attack':
        name,variant=args
        smoke=name=='smoke_full_gradient'
        if smoke:
            plan['num_samples']=2
            job={'name':name,'variants':['hybrid_plain'],'overrides':{'MANUAL_TEXT_OVERRIDES':molab_read(attack/'provenance/factual_captions.json')}}
        else:job=next(j for j in plan['attack_jobs'] if j['name']==name)
        state=attack/('smoke_state.json' if smoke else 'campaign_state.json')
        if not state.exists():molab_write(state,{'attack_runs':{},'evaluations':{}})
        os.environ['KLTN_TARGET_VARIANT']=variant
        driver=molab_module('embedded_attack',ATTACK_RUNTIME_SOURCE)
        code=driver.attack_worker(attack,plan,job,state,MOLAB_DRIVE_PREFIX+('/smoke' if smoke else '/results'),max(.001,(deadline-time.time())/3600))
        completed=molab_completed(name,variant,smoke)
        if completed==plan['num_samples']:
            print('CONFIGURATION COMPLETE',name,variant,completed,flush=True);return 0
        return code or 1
    if mode in ('eval','eval_smoke'):
        smoke = mode == 'eval_smoke'
        source_root = attack
        backup_prefix = MOLAB_DRIVE_PREFIX + '/evaluation'
        if smoke:
            evaluation = MOLAB_ROOT / 'smoke_evaluation'
            evaluation.mkdir(exist_ok=True)
            source_root = MOLAB_ROOT / 'smoke_source'
            source_root.mkdir(exist_ok=True)
            molab_write(source_root/'campaign_state.json',molab_read(attack/'smoke_state.json'))
            smoke_plan=json.loads(json.dumps(plan))
            smoke_plan['num_samples']=2
            smoke_plan['attack_jobs']=[{'name':'smoke_full_gradient','variants':['hybrid_plain']}]
            molab_write(evaluation/'molab_hypotheses_n200.json',smoke_plan)
            shutil.copyfile(MOLAB_ROOT/'evaluation/text_gen_eval_v3.1.ipynb',evaluation/'text_gen_eval_v3.1.ipynb')
            cache=evaluation/'hf_cache'
            if not cache.exists():cache.symlink_to(MOLAB_ROOT/'evaluation/hf_cache',target_is_directory=True)
            backup_prefix=MOLAB_DRIVE_PREFIX+'/smoke_evaluation'
            if not (evaluation/'evaluation_state.json').exists():
                state=molab_read(MOLAB_ROOT/'evaluation/evaluation_state.json')
                molab_write(evaluation/'evaluation_state.json',{'tasks':{},'model_revisions':state['model_revisions'],'backup_prefix':backup_prefix})
        config,model,kind=args
        evaluator=molab_module('embedded_eval',EVALUATION_RUNTIME_SOURCE)
        evaluator.prepare(evaluation,source_root,MOLAB_DRIVE_PREFIX+'/results')
        evaluator.restore_evaluations(evaluation,backup_prefix)
        from dotenv import load_dotenv
        load_dotenv('/marimo/.env',override=False)
        judge=kind=='judge'
        if judge and not os.environ.get('OPENAI_API_KEY'):
            print('MISSING OPENAI_API_KEY; caption checkpoints preserved',flush=True);return 75
        pause=evaluation/'judge_quota_pause.json'
        if judge and pause.exists() and molab_read(pause).get('not_before_unix',0)>time.time():return 75
        code=evaluator.worker(evaluation,config,model,MOLAB_CAPTION_BATCH_SIZE,judge,max(.001,(deadline-time.time())/3600),True)
        evaluator.report(evaluation)
        return code
    raise ValueError('Unknown notebook stage')


def molab_control(deadline):
    import fcntl
    MOLAB_ROOT.mkdir(parents=True,exist_ok=True)
    lock=(MOLAB_ROOT/'controller.lock').open('w')
    try:fcntl.flock(lock,fcntl.LOCK_EX|fcntl.LOCK_NB)
    except BlockingIOError:print('Existing notebook controller owns the campaign');return 0
    plan=molab_initialize()
    if molab_completed('smoke_full_gradient','hybrid_plain',True)!=2:
        code=molab_call('attack','smoke_full_gradient','hybrid_plain',deadline=deadline)
        if code:
            molab_progress('smoke_failed_or_paused',exit_code=code);return code
    molab_write(MOLAB_ROOT/'smoke_passed.json',{'pairs':2,'steps_per_pair':300,'notebook_sha256':molab_sha(__file__)})
    if not (MOLAB_ROOT/'evaluation_smoke_passed.json').exists():
        for model in plan['evaluation']['models']:
            code=molab_call('eval_smoke','smoke_full_gradient__hybrid_plain',model,'judge',deadline=deadline)
            if code:
                molab_progress('evaluation_smoke_failed_or_paused',model=model,exit_code=code);return code
        molab_write(MOLAB_ROOT/'evaluation_smoke_passed.json',{'pairs':2,'models':plan['evaluation']['models'],'diagnostic_only':True})
    print('GPU SMOKE PASSED: 2 original pairs x 300 steps. Starting eight configurations.',flush=True)
    # Finish plain/H4 controls first; each complete cohort is evaluated immediately.
    jobs=plan['attack_jobs']
    tasks=[(jobs[0],'hybrid_plain'),(jobs[1],'hybrid_plain'),(jobs[2],'hybrid_plain')]+[(jobs[0],v) for v in jobs[0]['variants'][1:]]
    for job,variant in tasks:
        name=job['name'];config=name+'__'+variant
        if time.time()>=deadline:molab_progress('paused_time_budget',next_config=config);return 75
        if molab_completed(name,variant)!=200:
            code=molab_call('attack',name,variant,deadline=deadline)
            if code:
                molab_progress('attack_failed_or_paused',configuration=config,exit_code=code);return code
        for model in plan['evaluation']['models']:
            code=molab_call('eval',config,model,'captions',deadline=deadline)
            if code:
                molab_progress('caption_failed_or_paused',configuration=config,model=model,exit_code=code);return code
        for model in plan['evaluation']['models']:
            code=molab_call('eval',config,model,'judge',deadline=deadline)
            if code not in (0,75):
                molab_progress('judge_failed',configuration=config,model=model,exit_code=code);return code
            # A provider quota pause remains explicit; never mark unscored work complete.
        molab_progress('configuration_processed',configuration=config)
    report=molab_module('embedded_eval',EVALUATION_RUNTIME_SOURCE).report(MOLAB_ROOT/'evaluation')
    complete=len(report)==16 and all(r.get('judge_status')=='complete' and r.get('judged_pairs')==200 for r in report)
    molab_progress('complete' if complete else 'attacks_complete_judge_pending')
    return 0 if complete else 75


def molab_launch():
    MOLAB_ROOT.mkdir(parents=True,exist_ok=True)
    pid_file=MOLAB_ROOT/'controller_pid.json'
    if pid_file.exists():
        saved=molab_read(pid_file)
        proc=Path('/proc')/str(saved['pid'])/'stat'
        if proc.exists():
            raw=proc.read_text();fields=raw[raw.rfind(')')+2:].split()
            if fields[0]!='Z' and fields[19]==saved.get('start_ticks'):
                print('Notebook controller already running:',saved['pid']);return
    os.environ['KLTN_DEADLINE_UNIX']=str(time.time()+MOLAB_MAX_HOURS*3600)
    script=Path(globals().get('__file__','/marimo/molab/kltn-attack-v4.3.3-h1-cfvlm.ipynb')).resolve()
    if script.suffix!='.ipynb':
        script=Path('/marimo/molab/kltn-attack-v4.3.3-h1-cfvlm.ipynb')
    if script.suffix!='.ipynb' or not script.exists():
        raise RuntimeError('Save both notebooks under /marimo/molab before Run all')
    log=MOLAB_ROOT/'controller.log'
    with log.open('a',encoding='utf-8') as stream:
        child=subprocess.Popen([sys.executable,'-u','-c',MOLAB_LOADER,str(script),'control'],stdout=stream,stderr=subprocess.STDOUT,start_new_session=True)
    stat=Path(f'/proc/{child.pid}/stat').read_text();fields=stat[stat.rfind(')')+2:].split()
    molab_write(pid_file,{'pid':child.pid,'start_ticks':fields[19],'notebook':str(script)})
    print('MOLAB STARTED',child.pid,'LOG',str(log),'PROGRESS',str(MOLAB_ROOT/'progress.json'))


# KLTN attack v4.3.3-h1 — CF-VLM negative caption policy

Corrects v4.3.2's noun-only counterfactual prompt using CF-VLM §2.2 and
Appendix A, page 17. Allows one object attribute or one causal intervention
(including its dependent outcome). See [paper](https://arxiv.org/abs/2506.17267)
and [official code](https://github.com/HCP-AI-Research-Lab/CF-VLM).

This attack adapts the text edit policy; it does not reproduce CF-VLM's SDXL
counterfactual images or fine-tuning losses. JSON traces and structural checks
are additions for auditing and cannot prove visual falsity/semantic atomicity.
New policy, bank and run identity: do not resume a v4.3.2 result directory.
Generate four candidates (official code default K=4); mine/select K_NEG=2 as
before. A second image-conditioned Qwen pass checks semantic minimality,
unchanged context, coherent causal effects and visible contradiction. Failed
or uncertain candidates are rejected; retries receive rejection feedback.
The same Qwen model performs this automated audit: it is not a human quality
certificate. All corrected negative variants share the audited cached bank.
Batched Qwen generation/auditing defaults to 16 with OOM splitting. FOA retains
its ordered state. The audited v2 campaign excludes the earlier test bank.


In [ ]:
# Cell 1 — User configuration. All budgets below use pixel units [0,255].
if globals().get('_NOTEBOOK_STAGE') == 'attack':
    from pathlib import Path
    
    DATASET_ROOT = Path('/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset')
    WORK_ROOT = Path('/kaggle/working')
    OUTPUT_ROOT = WORK_ROOT / 'kltn_attack_v4_3_3_h1_cfvlm_outputs'
    SOURCE_DIR = None
    TARGET_DIR = None
    PAIR_MANIFEST = None  # Previous attack manifest; source/target paths relative to its parent.
    NUM_SAMPLES = 1
    SAMPLE_START = 0
    RUN_ATTACK = False  # Set False for setup/preflight-only run
    INSTALL_DEPENDENCIES = False  # Installs for RUN_ATTACK or SETUP_ONLY; fresh kernel required.
    SETUP_ONLY = False  # True: install pinned packages without downloading pretrained weights.
    RUN_SELF_CHECKS = True
    GPU_SMOKE = False  # Tiny synthetic encoders only; no model download.
    
    RESUME_RUN_DIR = None
    RUN_VARIANTS = ['hybrid_dynamic']  # To compare baselines: ['foa_reference', 'mpc_clip', 'hybrid_dynamic']
    SEED = 2023
    ATTACK_DEVICE = 'cuda:0'
    ATTENTION_DEVICE = 'cuda:1'
    CAPTION_DEVICE = 'cuda:0'
    BACKBONES = ('B16', 'B32', 'Laion')
    INPUT_RES = 224
    STEPS = 300
    EPSILON_255 = 16.0
    ALPHA_255 = 1.0
    CLUSTER_NUMBER = 10
    PAPER_INPUT_RES = INPUT_RES
    PAPER_CROP_SCALE = (0.5, 0.9)
    USE_SOURCE_CROP = True
    USE_TARGET_CROP = True
    FUSION_LAMBDA = 0.6
    TEMPERATURE = 0.2
    POSITIVE_OMEGA = 2.0
    MPC_MOMENTUM = 0.9
    HYBRID_MOMENTUM = 0.0
    LOSS_WEIGHTS = {'foa': 1.0, 'mpc': 1.0, 'negative': 0.5}
    NEGATIVE_LOSS = 'infonce'  # infonce | margin | hard_margin | repulsion
    NEGATIVE_TEMPERATURE = 0.2
    NEGATIVE_MARGIN = 0.2
    K_NEG = 2
    MAX_NEGATIVES = 12
    CF_NEGATIVE_CANDIDATES = 4
    NEGATIVE_GENERATION_BATCH_SIZE = 16
    MINE_EVERY = 10
    REFRESH_EVERY = 30
    NUM_PHASES = 10
    HOTSPOT_TOP_K = 3
    HOTSPOT_IOU = 0.3
    HOTSPOT_PROB = 1.0  # v3.2 uses the always-hotspot crop primitive.
    ATTENTION_MODEL_ID = 'Qwen/Qwen3-VL-8B-Instruct'
    ATTENTION_REVISION = 'main'  # Resolved once to immutable SHA; restored on resume.
    ATTENTION_LAYER = 29
    ATTENTION_INPUT_RES = 448
    ATTENTION_PROMPT = 'Describe this image, no longer than 25 words.'
    MAX_CONCEPT_TOKENS = 128
    CONTENT_WORDS_ONLY = True
    CAPTION_MODEL_ID = 'OpenGVLab/InternVL3-1B-hf'
    MODEL_REVISION = 'main'
    TEXT_RETRIES = 5
    CONCEPT_MAX_NEW_TOKENS = 2048
    MANUAL_TEXT_OVERRIDES = {}  # {'00000': {'source': '...', 'target': '...', 'negatives': [...]}}
    HYBRID_AMP = False  # Stable FP32 default for dynamic. Optional AMP has logged FP32 fallback.
    GRADIENT_SCALE = 1.0  # No manual FP16 amplification by default; 1024 triggered overflow risk.
    HYBRID_AMP_FP32_RETRY = True  # Only hybrid; restore FOA history + RNG before a retry.
    SAVED_TENSORS_ON_CPU = True  # Explicit activation offload; unchanged model/dtype/objective.
    OFFLOAD_CLIP_DURING_QWEN = True  # Release GPU 0 between attention refreshes when needed.
    LOG_EVERY = 10  # Keep machine-readable steps/events trace every 10 updates.
    SHOW_ATTACK_PROGRESS = True  # v3.2-style live tqdm bar; display-only (not part of experiment identity).
    SAVE_EVERY = 30
    CONTINUE_ON_SAMPLE_ERROR = False  # Stateful references stop their variant at first attack error.
    EXPORT_ZIP = True
    
    VARIANTS = {
        'foa_reference': dict(foa=True, mpc=False, negatives='none', region='none', full_mpc=False, reference=True),
        'mpc_clip': dict(foa=False, mpc=True, negatives='none', region='none', full_mpc=False, reference=False),
        'mpc_full_reference': dict(foa=False, mpc=True, negatives='none', region='none', full_mpc=True, reference=True),
        'hybrid_plain': dict(foa=True, mpc=True, negatives='none', region='none', full_mpc=False, reference=False),
        'hybrid_negative': dict(foa=True, mpc=True, negatives='hard', region='none', full_mpc=False, reference=False),
        'hybrid_fixed': dict(foa=True, mpc=True, negatives='hard', region='fixed', full_mpc=False, reference=False),
        'hybrid_dynamic': dict(foa=True, mpc=True, negatives='hard', region='dynamic', full_mpc=False, reference=False),
        'hybrid_random_region': dict(foa=True, mpc=True, negatives='hard', region='random', full_mpc=False, reference=False),
        'hybrid_random_neg': dict(foa=True, mpc=True, negatives='random', region='dynamic', full_mpc=False, reference=False),
    }
    
    def validate_configuration():
        if not RUN_VARIANTS or len(set(RUN_VARIANTS)) != len(RUN_VARIANTS) or set(RUN_VARIANTS) - VARIANTS.keys():
            raise ValueError('Select unique known RUN_VARIANTS.')
        if tuple(BACKBONES) != ('B16', 'B32', 'Laion'):
            raise ValueError('v4.3 locks the requested three-model ensemble; edit and relabel an ablation explicitly.')
        for name in ('STEPS', 'INPUT_RES', 'K_NEG', 'MAX_NEGATIVES', 'MINE_EVERY', 'REFRESH_EVERY', 'NUM_PHASES', 'HOTSPOT_TOP_K', 'TEXT_RETRIES', 'CF_NEGATIVE_CANDIDATES', 'NEGATIVE_GENERATION_BATCH_SIZE', 'LOG_EVERY', 'SAVE_EVERY'):
            if type(globals()[name]) is not int or globals()[name] < 1:
                raise ValueError(f'{name} must be a positive integer.')
        if type(SAMPLE_START) is not int or SAMPLE_START < 0 or (NUM_SAMPLES is not None and (type(NUM_SAMPLES) is not int or NUM_SAMPLES < 1)):
            raise ValueError('Invalid sample selection.')
        if INPUT_RES != 224 or CLUSTER_NUMBER != 10 or not 1 <= K_NEG <= MAX_NEGATIVES <= 12:
            raise ValueError('Expected resolution=224, K=10 and 1 <= k_neg <= bank_size <= 12.')
        if not K_NEG <= CF_NEGATIVE_CANDIDATES <= MAX_NEGATIVES:
            raise ValueError('Expected k_neg <= CF candidates <= MAX_NEGATIVES')
        if NEGATIVE_LOSS not in {'infonce', 'margin', 'hard_margin', 'repulsion'}:
            raise ValueError('Unknown negative loss.')
        if not 0 <= EPSILON_255 <= 255 or ALPHA_255 <= 0 or min(TEMPERATURE, NEGATIVE_TEMPERATURE, GRADIENT_SCALE) <= 0:
            raise ValueError('Invalid budget/temperature/gradient scale.')
        if not 0 <= FUSION_LAMBDA <= 1 or not 0 <= HOTSPOT_IOU <= 1 or not 0 <= HOTSPOT_PROB <= 1:
            raise ValueError('Invalid fusion/hotspot parameters.')
        if set(LOSS_WEIGHTS) != {'foa', 'mpc', 'negative'} or any(v < 0 for v in LOSS_WEIGHTS.values()):
            raise ValueError('Loss weights must have foa/mpc/negative nonnegative entries.')
        if not 0 <= MPC_MOMENTUM < 1 or not 0 <= HYBRID_MOMENTUM < 1:
            raise ValueError('Invalid momentum.')
        if any(VARIANTS[v]['region'] != 'none' for v in RUN_VARIANTS) and STEPS < NUM_PHASES * HOTSPOT_TOP_K:
            raise ValueError('Region sequencing needs at least NUM_PHASES * HOTSPOT_TOP_K updates.')
    
    validate_configuration()
    GRADIENT_ENGINE = 'full_backward_serial_engine'


In [ ]:
# Cell 2 — A single pinned runtime; preflight never pip-installs or downloads weights.
if globals().get('_NOTEBOOK_STAGE') == 'attack':
    import ast, base64, contextlib, gc, hashlib, inspect, io, json, math, os, random, re
    import shutil, string, subprocess, sys, threading, time, traceback
    from abc import abstractmethod
    from contextlib import contextmanager
    from datetime import datetime, timezone
    from functools import partial
    from importlib import metadata as package_metadata
    from types import SimpleNamespace
    from typing import Any, Dict, List, Optional, Sequence
    
    PINNED_PACKAGES = [
        'transformers==4.57.1', 'tokenizers==0.22.1', 'huggingface-hub==0.34.4',
        'accelerate==1.11.0', 'bitsandbytes==0.48.1', 'ftfy==6.3.1',
        'tqdm==4.67.1', 'loguru==0.7.3', 'sentencepiece==0.2.1',
    ]
    if (RUN_ATTACK or SETUP_ONLY) and INSTALL_DEPENDENCIES:
        if any(p in sys.modules for p in ('transformers', 'tokenizers')):
            raise RuntimeError('Restart kernel before installing pinned packages, or disable INSTALL_DEPENDENCIES.')
        WORK_ROOT.mkdir(parents=True, exist_ok=True)
        with (WORK_ROOT / 'v43_setup.log').open('w', encoding='utf-8') as stream:
            result = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINNED_PACKAGES], stdout=stream, stderr=subprocess.STDOUT)
        if result.returncode:
            raise RuntimeError(f'Setup failed; read {WORK_ROOT / "v43_setup.log"}.')
    
    # Keep Torch/Torchvision supplied by Kaggle. K-means is embedded, not pip-resolved.
    import numpy as np
    import torch
    import torch.nn.functional as F
    import torchvision
    from PIL import Image
    from torch import Tensor, nn
    from torchvision import datasets, transforms
    from torchvision.transforms import functional as TF
    from tqdm.auto import tqdm
    try:
        from loguru import logger
        logger.remove()  # Only progress bar + warnings/errors in notebook output.
        logger.add(sys.stderr, level='WARNING')
    except ImportError:
        import logging
        logger = logging.getLogger('v43_attention')
        logger.setLevel(logging.WARNING)
    import transformers
    if transformers.__version__ != '4.57.1':
        raise RuntimeError('This notebook requires transformers==4.57.1. In a fresh kernel set SETUP_ONLY=True and INSTALL_DEPENDENCIES=True; keep RUN_ATTACK=False for setup/preflight.')
    from transformers import (
        AutoConfig, AutoProcessor, AutoTokenizer, AutoModel, AutoModelForImageTextToText,
        BitsAndBytesConfig, CLIPModel as HFCLIPModel, CLIPProcessor,
    )
    from huggingface_hub import HfApi
    
    BUILD_CELL_HASHES = {'upstream_snapshot': '851588e400880055492fcf8258bfbf223eb3087b9bbb0fe9d486b4452c88c07d', 'reference_support': 'f8b88cde966ff9f7b7d8a974b0233b7c6d3917c1ea42319a44da093e31576c1b', 'foa_author_loss': '0ad594a56eaeae0e04cd83087e344e4b0d9f52ffc2903be3fbdce3c7fbeb6869', 'models': '306ee3e63033878287df58000cf3bdd11d3e6ac550398d9cddf9c4e7ef03dc68', 'data_artifacts': '14d45e93a982dbd151e250c31e45241ce37f1722a65f4e827d288c68d0f91380', 'losses': '76df71591a4119dfb7faad4ecc1a04ffa3eb59edf785cb6538f070393457f27e', 'text_generation': '96775a36f7a55a19a7793536a4ac4d8314fb5a3a6ac52cb90e436087b2fda8af', 'hotspots_from_v32': '3a1c90f96f1cb192f0e3d097bf7724825b7a83668226a111325ea0b74456b8c2', 'qwen_attention_from_v32': '58f9a8d9a4ba954070da9abe876cec21c954566041dbd8101fbc53e23f58d888', 'attack_loops': '6a7e9a1a7b9c3a4e6d97322f97edeb0d6054785e05932007d733fd0f3e6fe7a8', 'checks': '0f1909bfad42618dfc92046d0f1e96ccd3e14a6c6275caac55623c947fde56b1', 'runner': '3d89cb68f0089c71aab280f1093e2db6eed19d31c9c0ea31647c582ffca2850c'}
    BUILD_CELL_FIRST_LINES = {'UPSTREAM = {}': 'upstream_snapshot', '# Cell 3 — Compile pinned author definitions into isolated namespaces, without imports/entrypoints.': 'reference_support', '# FOA: exact author definitions. Edit experimental behavior in the hybrid cells.': 'foa_author_loss', "# Cell 5 — Author model adapters share CLIP weights, but keep each method's own preprocessing.": 'models', '# Cell 6 — Pair once, export evaluation-compatible PNGs and JSON checkpoints.': 'data_artifacts', '# Cell 7 — Editable objectives. Scores are maximized; negative losses are minimized.': 'losses', '# Cell 8 — Cache InternVL captions and image-conditioned Qwen counterfactuals.': 'text_generation', '"""Fixed-area hotspot ranking with the original IoU fallback."""': 'hotspots_from_v32', 'DEBUG = os.environ.get("DEBUG", "0") == "1"': 'qwen_attention_from_v32', '# Cell 11 — Reference-preserving update order and editable hybrid orchestration.': 'attack_loops', '# Cell 12 — Meaningful checks, no pretrained weights or remote model calls.': 'checks', '# Cell 13 — Preflight, immutable run identity, per-variant manifests and boundary resume.': 'runner'}
    RUN_DIR = None
    MODEL_REVISIONS = {}
    MODEL_METADATA = {}
    MODEL_STORE = {}
    ATTENTION_BACKEND = None
    
    def sha256_file(path):
        digest = hashlib.sha256()
        with Path(path).open('rb') as stream:
            for block in iter(lambda: stream.read(1024 * 1024), b''):
                digest.update(block)
        return digest.hexdigest()
    
    def json_hash(value):
        return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False, default=str).encode()).hexdigest()
    
    def write_json(path, value):
        path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
        pending = path.with_suffix(path.suffix + '.tmp')
        pending.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False, default=str) + '\n', encoding='utf-8')
        pending.replace(path)
    
    def append_jsonl(path, value):
        path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
        with path.open('a', encoding='utf-8') as stream:
            stream.write(json.dumps(value, ensure_ascii=False, allow_nan=False, default=str) + '\n')
            stream.flush()
    
    def runtime_code_hashes():
        # In IPython, hash executed cells, so edits to a function invalidate resume.
        try:
            history = get_ipython().history_manager.input_hist_raw
        except (NameError, AttributeError):
            return BUILD_CELL_HASHES
        latest = dict(BUILD_CELL_HASHES)
        for source in history:
            first = source.strip().splitlines()[0] if source.strip() else ''
            name = BUILD_CELL_FIRST_LINES.get(first)
            if name is not None:
                latest[name] = hashlib.sha256(source.strip().encode()).hexdigest()
        return latest
    
    def seed_all(seed):
        random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    
    @contextlib.contextmanager
    def preserve_rng():
        state = capture_rng()
        try:
            yield
        finally:
            restore_rng(state)
    
    def capture_rng():
        np_state = np.random.get_state()
        return {'python': random.getstate(),
                'numpy': [np_state[0], np_state[1].tolist(), int(np_state[2]), int(np_state[3]), float(np_state[4])],
                'torch': torch.get_rng_state().tolist(),
                'cuda': [s.tolist() for s in torch.cuda.get_rng_state_all()] if torch.cuda.is_available() else []}
    
    def restore_rng(state):
        def tuples(value):
            return tuple(tuples(v) for v in value) if isinstance(value, (tuple, list)) else value
        random.setstate(tuples(state['python']))
        p = state['numpy']; np.random.set_state((p[0], np.asarray(p[1], dtype=np.uint32), p[2], p[3], p[4]))
        torch.set_rng_state(torch.tensor(state['torch'], dtype=torch.uint8))
        if state['cuda']:
            if len(state['cuda']) != torch.cuda.device_count():
                raise RuntimeError('Resume requires the same visible CUDA device count.')
            torch.cuda.set_rng_state_all([torch.tensor(s, dtype=torch.uint8) for s in state['cuda']])
    
    def event(stage, action, **fields):
        row = {'utc': datetime.now(timezone.utc).isoformat(), 'stage': stage, 'action': action, **fields}
        if RUN_DIR is not None:
            append_jsonl(RUN_DIR / 'events.jsonl', row)
        # Events are ALWAYS saved, but frequent/internal events do not flood Kaggle output.
        if (stage, action) in {('attack', 'progress'), ('text', 'captions_cached'),
                               ('text', 'negative_bank_cached'), ('resume', 'skip_complete')} and SHOW_ATTACK_PROGRESS:
            return
        if (stage, action) == ('attack', 'sample_start') and SHOW_ATTACK_PROGRESS:
            print(f"Sample {fields.get('position', '?')}/{fields.get('total', '?')} | "
                  f"variant={fields['variant']} | id={fields['sample_id']}")
            return
        print(f'[{stage}] {action}', ' '.join(f'{k}={v}' for k, v in fields.items()))
    
    def cuda_cleanup():
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    
    def saved_tensor_context():
        return torch.autograd.graph.save_on_cpu(pin_memory=True) if SAVED_TENSORS_ON_CPU else contextlib.nullcontext()
    
    def amp_context(enabled):
        return torch.autocast('cuda', dtype=torch.float16, enabled=enabled and str(ATTACK_DEVICE).startswith('cuda'))


## Nguồn và giới hạn reproduction

- [FOA author code](https://github.com/jiaxiaojunQAQ/FOA-Attack): entrypoint
  `generate_adversarial_samples_foa_attack.py`, K=10. Không dùng entrypoint tự chọn K trong `FOAttack.py`.
- [MPC author code](https://github.com/LiYuanBoJNU/MPCAttack): CLIP B16/B32/Laion,
  InternVL3-1B-hf và DINOv2-base. Caption generator là InternVL; **text encoder là CLIP**.
- [MPC paper §3.2](https://arxiv.org/html/2603.04846v1#S3.SS2): Eq.5 mô tả normalization từng block;
  code chính thức concatenate các feature đã xử lý rồi normalize toàn vector trong InfoNCE.
  Notebook lấy **code làm chuẩn**: không normalize lại fused CLIP anchor trước concatenate.
  Code dùng positive logit `omega * cosine / tau`, trả **negative CE** và gradient ascent;
  không chép lại cách đặt omega trong biểu thức Eq.6.
- [SAGA](https://arxiv.org/html/2602.04356v2): map cố định trước attack. `hybrid_dynamic`
  là phần mở rộng refresh map trên ảnh adversarial, không gọi là SAGA nguyên bản.

Các source tối thiểu và SHA-256 được nhúng bên dưới để chạy không cần checkout repository.
Chỉ compile các định nghĩa cần thiết, giữ nguyên thân hàm/class và autograd của tác giả.
K-means nhúng từ commit đã dùng trong v3.2, nhánh Euclidean; không thay bằng custom cluster means.
FOA/MPC reference giữ zero/random initialization, lịch sử loss, crop/update order và dtype upstream.
`STEPS != 300` được ghi là **short-run diagnostic**, không phải reproduction kết quả paper.

In [ ]:
UPSTREAM = {}
if globals().get('_NOTEBOOK_STAGE') == 'attack':
    
    UPSTREAM['foa'] = {'repo': 'jiaxiaojunQAQ/FOA-Attack', 'commit': '6aa9ad463a1e812b7f77ff2e5c4b1712e7c78c4b'}
    
    UPSTREAM['foa']["sources"] = {}
    
    UPSTREAM['foa']["sources"]['generate_adversarial_samples_foa_attack.py'] = r'''import os
import json
import hashlib
import random
import torchvision.transforms as transforms
import numpy as np
import torch
import torchvision
from PIL import Image
import hydra
from omegaconf import DictConfig
import os
from config_schema import MainConfig
from functools import partial
from typing import List, Dict, Optional
from torch import nn
from pytorch_lightning import seed_everything
import wandb
from omegaconf import OmegaConf
from tqdm import tqdm

from surrogates import (
    ClipB16FeatureExtractor,
    ClipL336FeatureExtractor,
    ClipB32FeatureExtractor,
    ClipLaionFeatureExtractor,
    EnsembleFeatureLoss,
    EnsembleFeatureLoss_OT_Auto,
    EnsembleFeatureExtractor,
    EnsembleFeatureExtractor_ot,
    EnsembleFeatureLoss_OT_foa_attack
)

from utils import hash_training_config, setup_wandb, ensure_dir

# Mapping from backbone names to model classes
BACKBONE_MAP: Dict[str, type] = {
    "L336": ClipL336FeatureExtractor,
    "B16": ClipB16FeatureExtractor,
    "B32": ClipB32FeatureExtractor,
    "Laion": ClipLaionFeatureExtractor,
}


def get_models_ot(cfg: MainConfig):
    """Get models based on configuration.

    Args:
        cfg: Configuration object containing model settings

    Returns:
        Tuple of (feature_extractor, list of models)

    Raises:
        ValueError: If ensemble=False but multiple backbones specified
    """
    if not cfg.model.ensemble and len(cfg.model.backbone) > 1:
        raise ValueError("When ensemble=False, only one backbone can be specified")

    models = []
    for backbone_name in cfg.model.backbone:
        if backbone_name not in BACKBONE_MAP:
            raise ValueError(
                f"Unknown backbone: {backbone_name}. Valid options are: {list(BACKBONE_MAP.keys())}"
            )
        model_class = BACKBONE_MAP[backbone_name]
        model = model_class().eval().to(cfg.model.device).requires_grad_(False)
        models.append(model)

    if cfg.model.ensemble:
        ensemble_extractor = EnsembleFeatureExtractor_ot(models,cluster_number=10)
    else:
        ensemble_extractor = models[0]  # Use single model directly

    return ensemble_extractor, models


def get_ensemble_loss(cfg: MainConfig, models: List[nn.Module]):
    ensemble_loss = EnsembleFeatureLoss(models)
    return ensemble_loss

def get_ensemble_loss_ot(cfg: MainConfig, models: List[nn.Module]):
    ensemble_loss_ot = EnsembleFeatureLoss_OT_foa_attack(models,cluster_number=10)
    return ensemble_loss_ot


def set_environment(seed=2023):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# Transform PIL.Image to PyTorch Tensor
def to_tensor(pic):
    mode_to_nptype = {"I": np.int32, "I;16": np.int16, "F": np.float32}
    img = torch.from_numpy(
        np.array(pic, mode_to_nptype.get(pic.mode, np.uint8), copy=True)
    )
    img = img.view(pic.size[1], pic.size[0], len(pic.getbands()))
    img = img.permute((2, 0, 1)).contiguous()
    return img.to(dtype=torch.get_default_dtype())


# Dataset with image paths
class ImageFolderWithPaths(torchvision.datasets.ImageFolder):
    def __getitem__(self, index):
        original_tuple = super().__getitem__(index)
        path, _ = self.samples[index]
        return original_tuple + (path,)


@hydra.main(version_base=None, config_path="config", config_name="ensemble_3models")
def main(cfg: MainConfig):
    set_environment()

    # Initialize wandb using shared utility
    setup_wandb(cfg, tags=["image_generation"])
    # Define metrics relationship for logging multiple images
    wandb.define_metric("epoch")
    wandb.define_metric("*", step_metric="epoch")

    ensemble_extractor_ot, models = get_models_ot(cfg)
    ensemble_loss_ot = get_ensemble_loss_ot(cfg, models)

    transform_fn = transforms.Compose(
        [
            transforms.Resize(
                cfg.model.input_res,
                interpolation=torchvision.transforms.InterpolationMode.BICUBIC,
            ),
            transforms.CenterCrop(cfg.model.input_res),
            transforms.Lambda(lambda img: img.convert("RGB")),
            transforms.Lambda(lambda img: to_tensor(img)),
        ]
    )

    clean_data = ImageFolderWithPaths(cfg.data.cle_data_path, transform=transform_fn)
    target_data = ImageFolderWithPaths(cfg.data.tgt_data_path, transform=transform_fn)

    data_loader_imagenet = torch.utils.data.DataLoader(
        clean_data, batch_size=cfg.data.batch_size, shuffle=False
    )
    data_loader_target = torch.utils.data.DataLoader(
        target_data, batch_size=cfg.data.batch_size, shuffle=False
    )

    print("Using source crop:", cfg.model.use_source_crop)
    print("Using target crop:", cfg.model.use_target_crop)

    source_crop = (
        transforms.RandomResizedCrop(cfg.model.input_res, scale=cfg.model.crop_scale)
        if cfg.model.use_source_crop
        else torch.nn.Identity()
    )
    target_crop = (
        transforms.RandomResizedCrop(cfg.model.input_res, scale=cfg.model.crop_scale)
        if cfg.model.use_target_crop
        else torch.nn.Identity()
    )

    for i, ((image_org, _, path_org), (image_tgt, _, path_tgt)) in enumerate(
        zip(data_loader_imagenet, data_loader_target)
    ):
        if cfg.data.batch_size * (i + 1) > cfg.data.num_samples:
            break

        print(f"\nProcessing image {i+1}/{cfg.data.num_samples//cfg.data.batch_size}")

        attack_imgpair(
            cfg=cfg,
            ensemble_extractor=ensemble_extractor_ot,
            ensemble_loss=ensemble_loss_ot,
            source_crop=source_crop,
            img_index=i,
            image_org=image_org,
            path_org=path_org,
            image_tgt=image_tgt,
            target_crop=target_crop,
        )

    wandb.finish()


def attack_imgpair(
    cfg: MainConfig,
    ensemble_extractor: nn.Module,
    ensemble_loss: nn.Module,
    source_crop: Optional[transforms.RandomResizedCrop],
    target_crop: Optional[transforms.RandomResizedCrop],
    img_index: int,
    image_org: torch.Tensor,
    path_org: List[str],
    image_tgt: torch.Tensor,
):
    image_org, image_tgt = image_org.to(cfg.model.device), image_tgt.to(
        cfg.model.device
    )
    attack_type = cfg.attack
    attack_fn = {
        "fgsm": fgsm_attack,
        "mifgsm": mifgsm_attack,
        "pgd": pgd_attack,
    }[attack_type]
    adv_image = attack_fn(
        cfg=cfg,
        ensemble_extractor=ensemble_extractor,
        ensemble_loss=ensemble_loss,
        source_crop=source_crop,
        target_crop=target_crop,
        img_index=img_index,
        image_org=image_org,
        image_tgt=image_tgt,
    )

    # Get config hash for output directory
    config_hash = hash_training_config(cfg)

    # Save images
    for path_idx in range(len(path_org)):
        folder, name = (
            path_org[path_idx].split("/")[-2],
            path_org[path_idx].split("/")[-1],
        )
        # Use config hash in output path
        folder_to_save = os.path.join(cfg.data.output, "img", config_hash, folder)
        ensure_dir(folder_to_save)

        if "JPEG" in name:
            torchvision.utils.save_image(
                adv_image[path_idx], os.path.join(folder_to_save, name[:-4]) + "png"
            )
        elif "png" in name:
            torchvision.utils.save_image(
                adv_image[path_idx], os.path.join(folder_to_save, name)
            )


def log_metrics(pbar, metrics, img_index, epoch=None):
    """
    Log metrics to progress bar and wandb.

    Args:
        pbar: tqdm progress bar to update
        metrics: Dictionary of metrics to log
        img_index: Index of the image (for wandb logging)
        epoch: Optional epoch number for logging
    """
    # Format metrics for progress bar
    pbar_metrics = {
        k: f"{v:.5f}" if "sim" in k else f"{v:.3f}" for k, v in metrics.items()
    }
    pbar.set_postfix(pbar_metrics)

    # Prepare wandb metrics with image index
    wandb_metrics = {f"img{img_index}_{k}": v for k, v in metrics.items()}
    if epoch is not None:
        wandb_metrics["epoch"] = epoch

    # Log to wandb
    wandb.log(wandb_metrics)


def fgsm_attack(
    cfg: MainConfig,
    ensemble_extractor: nn.Module,
    ensemble_loss: nn.Module,
    source_crop: Optional[transforms.RandomResizedCrop],
    target_crop: Optional[transforms.RandomResizedCrop],
    img_index: int,
    image_org: torch.Tensor,
    image_tgt: torch.Tensor,
):
    """
    Perform FGSM attack on the image to generate adversarial examples.

    Args:
        cfg: Configuration parameters
        ensemble_extractor: Ensemble feature extractor model
        ensemble_loss: Ensemble loss function
        source_crop: Optional transform for cropping source images
        target_crop: Optional transform for cropping target images
        i: Index of the image (for logging)
        image_org: Original source image tensor
        image_tgt: Target image tensor to match features with

    Returns:
        torch.Tensor: Generated adversarial image
    """
    # Initialize perturbation
    delta = torch.zeros_like(image_org, requires_grad=True)

    # Progress bar for optimization
    pbar = tqdm(range(cfg.optim.steps), desc=f"Attack progress")

    # Main optimization loop
    for epoch in pbar:

        with torch.no_grad():
            ensemble_loss.set_ground_truth(target_crop(image_tgt))

        # Forward pass
        adv_image = image_org + delta

        adv_features,adv_features_local = ensemble_extractor(adv_image)

        # Calculate metrics
        metrics = {
            "max_delta": torch.max(torch.abs(delta)).item(),
            "mean_delta": torch.mean(torch.abs(delta)).item(),
        }

        # Calculate loss based on configuration
        global_sim = ensemble_loss(adv_features,adv_features_local)
        metrics["global_similarity"] = global_sim.item()

        if cfg.model.use_source_crop:
            # If using source crop, calculate additional local similarity
            local_cropped = source_crop(adv_image)
            local_features,local_features_local = ensemble_extractor(local_cropped)
            local_sim = ensemble_loss(local_features,local_features_local)
            loss = local_sim
            metrics["local_similarity"] = local_sim.item()
        else:
            # Otherwise use global similarity as loss
            loss = global_sim

        # Log current metrics
        log_metrics(pbar, metrics, img_index, epoch)

        grad = torch.autograd.grad(loss, delta, create_graph=False)[0]

        # print("loss",loss)
        # print(grad)
        # Update delta using FGSM
        delta.data = torch.clamp(
            delta + cfg.optim.alpha * torch.sign(grad),
            min=-cfg.optim.epsilon,
            max=cfg.optim.epsilon,
        )

    # Create final adversarial image
    adv_image = image_org + delta
    adv_image = torch.clamp(adv_image / 255.0, 0.0, 1.0)

    # Log final perturbation metrics
    final_metrics = {
        "max_delta": torch.max(torch.abs(delta)).item(),
        "mean_delta": torch.mean(torch.abs(delta)).item(),
    }
    log_metrics(pbar, final_metrics, img_index)


    # print(delta)

    return adv_image


def mifgsm_attack(
    cfg: MainConfig,
    ensemble_extractor: nn.Module,
    ensemble_loss: nn.Module,
    source_crop: Optional[transforms.RandomResizedCrop],
    target_crop: Optional[transforms.RandomResizedCrop],
    img_index: int,
    image_org: torch.Tensor,
    image_tgt: torch.Tensor,
):
    """
    Perform MI-FGSM attack on the image to generate adversarial examples.

    Args:
        cfg: Configuration parameters
        ensemble_extractor: Ensemble feature extractor model
        ensemble_loss: Ensemble loss function
        source_crop: Optional transform for cropping source images
        target_crop: Optional transform for cropping target images
        i: Index of the image (for logging)
        image_org: Original source image tensor
        image_tgt: Target image tensor to match features with

    Returns:
        torch.Tensor: Generated adversarial image
    """
    # Initialize perturbation and momentum
    delta = torch.zeros_like(image_org, requires_grad=True)
    momentum = torch.zeros_like(image_org, requires_grad=False)

    # Progress bar for optimization
    pbar = tqdm(range(cfg.optim.steps), desc=f"Attack progress")

    # Main optimization loop
    for epoch in pbar:

        with torch.no_grad():
            ensemble_loss.set_ground_truth(target_crop(image_tgt))

        # Forward pass
        adv_image = image_org + delta
        adv_features = ensemble_extractor(adv_image)

        # Calculate metrics
        metrics = {
            "max_delta": torch.max(torch.abs(delta)).item(),
            "mean_delta": torch.mean(torch.abs(delta)).item(),
        }

        # Calculate loss based on configuration
        global_sim = ensemble_loss(adv_features)
        metrics["global_similarity"] = global_sim.item()

        if cfg.model.use_source_crop:
            # If using source crop, calculate additional local similarity
            local_cropped = source_crop(adv_image)
            local_features = ensemble_extractor(local_cropped)
            local_sim = ensemble_loss(local_features)
            loss = local_sim
            metrics["local_similarity"] = local_sim.item()
        else:
            # Otherwise use global similarity as loss
            loss = global_sim

        log_metrics(pbar, metrics, img_index, epoch)

        grad = torch.autograd.grad(loss, delta, create_graph=False)[0]

        # MI-FGSM update
        momentum = momentum * 0.9 + grad
        delta.data = torch.clamp(
            delta + cfg.optim.alpha * torch.sign(momentum),
            min=-cfg.optim.epsilon,
            max=cfg.optim.epsilon,
        )

    # Create final adversarial image
    adv_image = image_org + delta
    adv_image = torch.clamp(adv_image / 255.0, 0.0, 1.0)

    # Log final perturbation metrics
    final_metrics = {
        "max_delta": torch.max(torch.abs(delta)).item(),
        "mean_delta": torch.mean(torch.abs(delta)).item(),
    }
    log_metrics(pbar, final_metrics, img_index)

    return adv_image


def pgd_attack(
    cfg: MainConfig,
    ensemble_extractor: nn.Module,
    ensemble_loss: nn.Module,
    source_crop: Optional[transforms.RandomResizedCrop],
    target_crop: Optional[transforms.RandomResizedCrop],
    img_index: int,
    image_org: torch.Tensor,
    image_tgt: torch.Tensor,
):
    """
    Perform PGD attack on the image to generate adversarial examples.

    Args:
        cfg: Configuration parameters
        ensemble_extractor: Ensemble feature extractor model
        ensemble_loss: Ensemble loss function
        source_crop: Optional transform for cropping source images
        target_crop: Optional transform for cropping target images
        i: Index of the image (for logging)
        image_org: Original source image tensor
        image_tgt: Target image tensor to match features with

    Returns:
        torch.Tensor: Generated adversarial image
    """
    # Initialize perturbation and momentum
    delta = torch.zeros_like(image_org, requires_grad=True)
    optimizer = torch.optim.Adam([delta], lr=cfg.optim.alpha)

    # Progress bar for optimization
    pbar = tqdm(range(cfg.optim.steps), desc=f"Attack progress")

    # Main optimization loop
    for epoch in pbar:

        with torch.no_grad():
            ensemble_loss.set_ground_truth(target_crop(image_tgt))

        # Forward pass
        adv_image = image_org + delta
        adv_features = ensemble_extractor(adv_image)

        # Calculate metrics
        metrics = {
            "max_delta": torch.max(torch.abs(delta)).item(),
            "mean_delta": torch.mean(torch.abs(delta)).item(),
        }

        # Calculate loss based on configuration
        global_sim = ensemble_loss(adv_features)
        metrics["global_similarity"] = global_sim.item()

        if cfg.model.use_source_crop:
            # If using source crop, calculate additional local similarity
            local_cropped = source_crop(adv_image)
            local_features = ensemble_extractor(local_cropped)
            local_sim = ensemble_loss(local_features)
            loss = -local_sim # since we want to maximize the loss
            metrics["local_similarity"] = local_sim.item()
        else:
            # Otherwise use global similarity as loss
            loss = -global_sim

        log_metrics(pbar, metrics, img_index, epoch)

        optimizer.zero_grad()
        loss.backward()

        # PGD update
        optimizer.step()
        delta.data = torch.clamp(
            delta,
            min=-cfg.optim.epsilon,
            max=cfg.optim.epsilon,
        )

    # Create final adversarial image
    adv_image = image_org + delta
    adv_image = torch.clamp(adv_image / 255.0, 0.0, 1.0)

    # Log final perturbation metrics
    final_metrics = {
        "max_delta": torch.max(torch.abs(delta)).item(),
        "mean_delta": torch.mean(torch.abs(delta)).item(),
    }
    log_metrics(pbar, final_metrics, img_index)

    return adv_image


if __name__ == "__main__":
    main()
'''
    
    UPSTREAM['foa']["sources"]['surrogates/FeatureExtractors/Base.py'] = r'''import torch
from torch import nn, Tensor
from abc import abstractmethod
from typing import List, Any, Callable, Dict
from kmeans_pytorch import kmeans
import torch.nn.functional as F
from torchvision import transforms as T
import sys
import os
from contextlib import contextmanager
import numpy as np
@contextmanager
def suppress_output():
    with open(os.devnull, 'w') as fnull:
        old_stdout = sys.stdout
        old_stderr = sys.stderr
        sys.stdout = fnull
        sys.stderr = fnull
        try:
            yield
        finally:
            sys.stdout = old_stdout
            sys.stderr = old_stderr

class BaseFeatureExtractor(nn.Module):
    def __init__(self):
        super(BaseFeatureExtractor, self).__init__()
        pass

    @abstractmethod
    def forward(self, x: Tensor) -> Tensor:
        pass


class EnsembleFeatureExtractor(BaseFeatureExtractor):
    def __init__(self, extractors: List[BaseFeatureExtractor]):
        super(EnsembleFeatureExtractor, self).__init__()
        self.extractors = nn.ModuleList(extractors)

    def forward(self, x: Tensor) -> Tensor:
        # features = []
        # for model in self.extractors:
        #     features.append(model(x).squeeze())
        # features = torch.cat(features, dim=0)
        features = {}  # 不拼接，改为字典存储
        for i, model in enumerate(self.extractors):
            features[i] = model(x).squeeze()
        return features

class EnsembleFeatureExtractor_ot(BaseFeatureExtractor):
    def __init__(self, extractors: List[BaseFeatureExtractor],cluster_number=5):
        super(EnsembleFeatureExtractor_ot, self).__init__()
        self.extractors = nn.ModuleList(extractors)
        self.cluster_number = cluster_number

    def forward(self, x: Tensor) -> Tensor:
        # features = []
        # for model in self.extractors:
        #     features.append(model(x).squeeze())
        # features = torch.cat(features, dim=0)
        features = {}  # 不拼接，改为字典存储
        features_local = {}
        for i, model in enumerate(self.extractors):
            # features[i] = model(x).squeeze()
            x_tensor, x_embedding = model.global_local_features(x.to(x.device))
            features[i] = x_tensor.squeeze()
            cluster_center = self.get_cluster_center(x_embedding[0],x.device).unsqueeze(0)
            features_local[i]=cluster_center

        return features,features_local

    def get_cluster_center(self, embedding_img,device):
        # self.setup_seed(20)
        # for i in range(10):
        # np.random.seed(20)
        with suppress_output():
            cluster_ids_x, cluster_center = kmeans(
                X=embedding_img,
                num_clusters=self.cluster_number,
                distance='euclidean',
                device=device
            )
            cluster_center = cluster_center.to(device)
        # print(cluster_ids_x)
        return cluster_center
    
class EnsembleFeatureExtractor_ot3(BaseFeatureExtractor):
    def __init__(self, extractors: List[BaseFeatureExtractor]):
        super(EnsembleFeatureExtractor_ot3, self).__init__()
        self.extractors = nn.ModuleList(extractors)

    def forward(self, x: Tensor) -> Tensor:
        # features = []
        # for model in self.extractors:
        #     features.append(model(x).squeeze())
        # features = torch.cat(features, dim=0)
        features = {}  # 不拼接，改为字典存储
        features_local = {}
        for i, model in enumerate(self.extractors):
            # features[i] = model(x).squeeze()
            x_tensor, x_embedding = model.global_local_features(x.to(x.device))
            features[i] = x_tensor.squeeze()
            cluster_center = self.get_cluster_center(x_embedding[0],x.device).unsqueeze(0)
            features_local[i]=cluster_center

        return features,features_local

    def get_cluster_center(self, embedding_img,device):
        # self.setup_seed(20)
        # for i in range(10):
        # np.random.seed(20)
        with suppress_output():
            cluster_ids_x, cluster_center = kmeans(
                X=embedding_img,
                num_clusters=3,
                distance='euclidean',
                device=device
            )
            cluster_center = cluster_center.to(device)
        # print(cluster_ids_x)
        return cluster_center


class EnsembleFeatureLoss(nn.Module):
    def __init__(self, extractors: List[BaseFeatureExtractor]):
        super(EnsembleFeatureLoss, self).__init__()
        self.extractors = nn.ModuleList(extractors)
        self.ground_truth = []

    @torch.no_grad()
    def set_ground_truth(self, x: Tensor):
        self.ground_truth.clear()
        for model in self.extractors:
            self.ground_truth.append(model(x).to(x.device))

    def __call__(self, feature_dict: Dict[int, Tensor], y: Any = None) -> Tensor:
        loss = 0
        for index, model in enumerate(self.extractors):
            gt = self.ground_truth[index]
            feature = feature_dict[index]
            loss += torch.mean(torch.sum(feature * gt, dim=1))
            
        loss = loss / len(self.extractors)

        return loss


class EnsembleFeatureLoss_OT(nn.Module):
    def __init__(self, extractors: List[BaseFeatureExtractor]):
        super(EnsembleFeatureLoss_OT, self).__init__()
        self.extractors = nn.ModuleList(extractors)
        self.ground_truth = []
        self.ground_truth_local = []



    @torch.no_grad()
    def set_ground_truth(self, x: Tensor):
        self.ground_truth.clear()
        self.ground_truth_local.clear()
        for model in self.extractors:
            x_tensor, x_embedding = model.global_local_features(x.to(x.device))
            x_embedding = x_embedding.squeeze(0)
            cluster_center = self.get_cluster_center(x_embedding).unsqueeze(0)
            self.ground_truth.append(x_tensor)
            self.ground_truth_local.append(cluster_center)



    def __call__(self, feature_dict: Dict[int, Tensor],feature_local_dict: Dict[int, Tensor], y: Any = None) -> Tensor:
        loss = 0
        loss_local=0
        for index, model in enumerate(self.extractors):
            gt_local = self.ground_truth_local[index].squeeze(0)
            gt = self.ground_truth[index]
            feature = feature_dict[index]
            feature_local = feature_local_dict[index].squeeze(0)
            # print("gt_local",gt_local.shape)
            # print("feature_local", feature_local.shape)
            loss_local += self.OT(gt_local, feature_local) * 2

            loss += torch.mean(torch.sum(feature * gt, dim=1))

        loss = loss / len(self.extractors)
        loss_local=loss_local / len(self.extractors)

        # print("loss",loss)
        # print("loss_local", loss_local)
        loss=loss+loss_local*0.1
        return loss

    def get_cluster_center(self, embedding_img):
        # self.setup_seed(20)
        # for i in range(10):
        # np.random.seed(20)
        with suppress_output():
            cluster_ids_x, cluster_center = kmeans(
                X=embedding_img,
                num_clusters=5,
                distance='euclidean',
                device=torch.device('cuda'),
                iter_limit=100,  # 最大迭代次数，默认是 10
            )
            cluster_center = cluster_center.to('cuda')
        # print(cluster_ids_x)
        return cluster_center

    def OT(self, src_dis, tgt_dis):
        # print(src_dis.shape, tgt_dis.shape)
        src_dis_norm = F.normalize(src_dis, dim=1)
        tgt_dis_norm = F.normalize(tgt_dis, dim=1)
        sim = torch.einsum('md,nd->mn', src_dis_norm, tgt_dis_norm).contiguous()
        wdist = 1 - sim
        xx = torch.full((src_dis.shape[0],), 1.0 / src_dis.shape[0], dtype=sim.dtype, device=sim.device)
        yy = torch.full((tgt_dis.shape[0],), 1.0 / tgt_dis.shape[0], dtype=sim.dtype, device=sim.device)
        with torch.no_grad():
            KK = torch.exp(-wdist / 0.1)
            T = self.Sinkhorn(KK, xx, yy)
        if torch.isnan(T).any():
            return None
        sim_op = torch.sum(T * sim, dim=(0, 1))
        loss = torch.sum(sim_op)
        return loss

    def Sinkhorn(self, K, u, v):
        r = torch.ones_like(u)
        c = torch.ones_like(v)
        thresh = 1e-2
        for i in range(100):
            r0 = r
            r = u / (K @ c.unsqueeze(-1)).squeeze(-1)
            c = v / (K.t() @ r.unsqueeze(-1)).squeeze(-1)
            err = (r - r0).abs().mean()
            if err.item() < thresh:
                break
        T = torch.outer(r, c) * K
        return T


class EnsembleFeatureLoss_OT_Auto(nn.Module):
    def __init__(self, extractors: List[BaseFeatureExtractor]):
        super(EnsembleFeatureLoss_OT_Auto, self).__init__()
        self.extractors = nn.ModuleList(extractors)
        self.ground_truth = []
        self.ground_truth_local = []
        self.previous_loss_list=[]
        self.previous_loss_local_list = []



    @torch.no_grad()
    def set_ground_truth(self, x: Tensor):
        self.ground_truth.clear()
        self.ground_truth_local.clear()
        for model in self.extractors:
            x_tensor, x_embedding = model.global_local_features(x.to(x.device))
            x_embedding = x_embedding.squeeze(0)
            cluster_center = self.get_cluster_center(x_embedding).unsqueeze(0)
            self.ground_truth.append(x_tensor)
            self.ground_truth_local.append(cluster_center)



    def __call__(self, feature_dict: Dict[int, Tensor],feature_local_dict: Dict[int, Tensor], y: Any = None) -> Tensor:
        loss_list = []
        loss_local_list = []
        for index, model in enumerate(self.extractors):
            gt_local = self.ground_truth_local[index].squeeze(0)
            gt = self.ground_truth[index]
            feature = feature_dict[index]
            feature_local = feature_local_dict[index].squeeze(0)
            # print("gt_local",gt_local.shape)
            # print("feature_local", feature_local.shape)
            local_loss = self.OT(gt_local, feature_local) * 2
            feat_loss = torch.mean(torch.sum(feature * gt, dim=1))

            loss_list.append(feat_loss)
            loss_local_list.append(local_loss)

        total_losses = [
            loss_list[i] + 0.1 * loss_local_list[i]
            for i in range(len(self.extractors))
        ]
        # 初始化 previous_loss_list（首次）
        if len(self.previous_loss_list) == 0:
            self.previous_loss_list = [l.detach() for l in total_losses]

        weights = []
        for i in range(len(self.extractors)):
            ratio = total_losses[i].item() / (self.previous_loss_list[i].item() + 1e-8)
            weights.append(ratio)
            # 归一化 softmax 计算动态权重
        T = 1.0
        K = len(weights)
        weights_np = np.array(weights)
        weights_softmax = np.exp(weights_np / T)
        weights_softmax /= np.sum(weights_softmax)
        weights_softmax *= K  # 可选：缩放为 K


        # 初始化 previous_loss_list（首次）
        for i in range(len(self.extractors)):
            self.previous_loss_list[i] = total_losses[i].detach()

        # 加权总损失
        total_loss = sum(
            weights_softmax[i] * total_losses[i]
            for i in range(len(self.extractors))
        )
        return total_loss

    def get_cluster_center(self, embedding_img):
        # self.setup_seed(20)
        # for i in range(10):
        # np.random.seed(20)
        with suppress_output():
            cluster_ids_x, cluster_center = kmeans(
                X=embedding_img,
                num_clusters=5,
                distance='euclidean',
                device=torch.device('cuda'),
                iter_limit=100,  # 最大迭代次数，默认是 10
            )
            cluster_center = cluster_center.to('cuda')
        # print(cluster_ids_x)
        return cluster_center

    def OT(self, src_dis, tgt_dis):
        # print(src_dis.shape, tgt_dis.shape)
        src_dis_norm = F.normalize(src_dis, dim=1)
        tgt_dis_norm = F.normalize(tgt_dis, dim=1)
        sim = torch.einsum('md,nd->mn', src_dis_norm, tgt_dis_norm).contiguous()
        wdist = 1 - sim
        xx = torch.full((src_dis.shape[0],), 1.0 / src_dis.shape[0], dtype=sim.dtype, device=sim.device)
        yy = torch.full((tgt_dis.shape[0],), 1.0 / tgt_dis.shape[0], dtype=sim.dtype, device=sim.device)
        with torch.no_grad():
            KK = torch.exp(-wdist / 0.1)
            T = self.Sinkhorn(KK, xx, yy)
        if torch.isnan(T).any():
            return None
        sim_op = torch.sum(T * sim, dim=(0, 1))
        loss = torch.sum(sim_op)
        return loss

    def Sinkhorn(self, K, u, v):
        r = torch.ones_like(u)
        c = torch.ones_like(v)
        thresh = 1e-2
        for i in range(100):
            r0 = r
            r = u / (K @ c.unsqueeze(-1)).squeeze(-1)
            c = v / (K.t() @ r.unsqueeze(-1)).squeeze(-1)
            err = (r - r0).abs().mean()
            if err.item() < thresh:
                break
        T = torch.outer(r, c) * K
        return T
    
class EnsembleFeatureLoss_OT_foa_attack(nn.Module):
    def __init__(self, extractors: List[BaseFeatureExtractor],cluster_number=5):
        super(EnsembleFeatureLoss_OT_foa_attack, self).__init__()
        self.extractors = nn.ModuleList(extractors)
        self.ground_truth = []
        self.ground_truth_local = []
        self.previous_loss_list=[]
        self.previous_loss_local_list = []
        self.cluster_number = cluster_number


    @torch.no_grad()
    def set_ground_truth(self, x: Tensor):
        self.ground_truth.clear()
        self.ground_truth_local.clear()
        for model in self.extractors:
            x_tensor, x_embedding = model.global_local_features(x.to(x.device))
            x_embedding = x_embedding.squeeze(0)
            cluster_center = self.get_cluster_center(x_embedding,x.device).unsqueeze(0)
            self.ground_truth.append(x_tensor)
            self.ground_truth_local.append(cluster_center)

    def __call__(self, feature_dict: Dict[int, Tensor],feature_local_dict: Dict[int, Tensor], y: Any = None) -> Tensor:
        loss_list = []
        loss_local_list = []
        for index, model in enumerate(self.extractors):
            gt_local = self.ground_truth_local[index].squeeze(0)
            gt = self.ground_truth[index]
            feature = feature_dict[index].unsqueeze(0)
            feature_local = feature_local_dict[index].squeeze(0)
            # print("gt_local",gt_local.shape)
            # print("feature_local", feature_local.shape)
            local_loss = self.OT(gt_local, feature_local)
            
            # feat_loss = torch.mean(torch.sum(feature * gt, dim=1))
            feat_loss = self.OT(gt,feature)

            loss_list.append(feat_loss)
            loss_local_list.append(local_loss)

        total_losses = [
            loss_list[i] + 0.2 * loss_local_list[i]
            for i in range(len(self.extractors))
        ]
        # 初始化 previous_loss_list（首次）
        if len(self.previous_loss_list) == 0:
            self.previous_loss_list = [l.detach() for l in total_losses]

        weights = []
        for i in range(len(self.extractors)):
            ratio = total_losses[i].item() / (self.previous_loss_list[i].item() + 1e-8)
            weights.append(ratio)
            # 归一化 softmax 计算动态权重
        
        T = 1.0
        K = len(weights)
        weights_np = np.array(weights)
        weights_softmax = np.exp(weights_np / T)
        weights_softmax /= np.sum(weights_softmax)
        weights_softmax *= K  # 可选：缩放为 K


        # 初始化 previous_loss_list（首次）
        for i in range(len(self.extractors)):
            self.previous_loss_list[i] = total_losses[i].detach()

        # 加权总损失
        total_loss = sum(
            weights_softmax[i] * total_losses[i]
            for i in range(len(self.extractors))
        )
        return total_loss

    def get_cluster_center(self, embedding_img,device):
        # self.setup_seed(20)
        # for i in range(10):
        # np.random.seed(20)
        with suppress_output():
            cluster_ids_x, cluster_center = kmeans(
                X=embedding_img,
                num_clusters=self.cluster_number, # TODO
                distance='euclidean',
                device=device,
                iter_limit=100,  # 最大迭代次数，默认是 10
            )
            cluster_center = cluster_center.to(device)
        # print(cluster_ids_x)
        return cluster_center

    def OT(self, src_dis, tgt_dis):
        # print(src_dis.shape, tgt_dis.shape)
        src_dis_norm = F.normalize(src_dis, dim=1)
        tgt_dis_norm = F.normalize(tgt_dis, dim=1)
        sim = torch.einsum('md,nd->mn', src_dis_norm, tgt_dis_norm).contiguous()
        wdist = 1 - sim
        xx = torch.full((src_dis.shape[0],), 1.0 / src_dis.shape[0], dtype=sim.dtype, device=sim.device)
        yy = torch.full((tgt_dis.shape[0],), 1.0 / tgt_dis.shape[0], dtype=sim.dtype, device=sim.device)
        with torch.no_grad():
            KK = torch.exp(-wdist / 0.1)
            T = self.Sinkhorn(KK, xx, yy)
        if torch.isnan(T).any():
            return None
        sim_op = torch.sum(T * sim, dim=(0, 1))
        loss = torch.sum(sim_op)
        return loss

    def Sinkhorn(self, K, u, v):
        r = torch.ones_like(u)
        c = torch.ones_like(v)
        thresh = 1e-2
        for i in range(100):
            r0 = r
            r = u / (K @ c.unsqueeze(-1)).squeeze(-1)
            c = v / (K.t() @ r.unsqueeze(-1)).squeeze(-1)
            err = (r - r0).abs().mean()
            if err.item() < thresh:
                break
        T = torch.outer(r, c) * K
        return T

class EnsembleFeatureLoss_OT_ablation_wo_global(nn.Module):
    def __init__(self, extractors: List[BaseFeatureExtractor], cluster_number=5):
        super(EnsembleFeatureLoss_OT_ablation_wo_global, self).__init__()
        self.extractors = nn.ModuleList(extractors)
        self.ground_truth = []
        self.ground_truth_local = []
        self.previous_loss_list=[]
        self.previous_loss_local_list = []
        self.cluster_number = cluster_number


    @torch.no_grad()
    def set_ground_truth(self, x: Tensor):
        self.ground_truth.clear()
        self.ground_truth_local.clear()
        for model in self.extractors:
            x_tensor, x_embedding = model.global_local_features(x.to(x.device))
            x_embedding = x_embedding.squeeze(0)
            cluster_center = self.get_cluster_center(x_embedding,x.device).unsqueeze(0)
            self.ground_truth.append(x_tensor)
            self.ground_truth_local.append(cluster_center)

    def __call__(self, feature_dict: Dict[int, Tensor],feature_local_dict: Dict[int, Tensor], y: Any = None) -> Tensor:
        loss_list = []
        loss_local_list = []
        for index, model in enumerate(self.extractors):
            gt_local = self.ground_truth_local[index].squeeze(0)
            gt = self.ground_truth[index]
            feature = feature_dict[index]
            feature_local = feature_local_dict[index].squeeze(0)
            local_loss = self.OT(gt_local, feature_local)
            feat_loss = torch.mean(torch.sum(feature * gt, dim=1))

            loss_list.append(feat_loss)
            loss_local_list.append(local_loss)

        total_losses = [
            loss_list[i] + 0.2 * loss_local_list[i]
            for i in range(len(self.extractors))
        ]
        # 初始化 previous_loss_list（首次）
        if len(self.previous_loss_list) == 0:
            self.previous_loss_list = [l.detach() for l in total_losses]

        weights = []
        for i in range(len(self.extractors)):
            ratio = total_losses[i].item() / (self.previous_loss_list[i].item() + 1e-8)
            weights.append(ratio)
            # 归一化 softmax 计算动态权重
        
        T = 1.0
        K = len(weights)
        weights_np = np.array(weights)
        weights_softmax = np.exp(weights_np / T)
        weights_softmax /= np.sum(weights_softmax)
        weights_softmax *= K  # 可选：缩放为 K


        # 初始化 previous_loss_list（首次）
        for i in range(len(self.extractors)):
            self.previous_loss_list[i] = total_losses[i].detach()

        # 加权总损失
        total_loss = sum(
            weights_softmax[i] * total_losses[i]
            for i in range(len(self.extractors))
        )
        return total_loss

    def get_cluster_center(self, embedding_img,device):
        with suppress_output():
            cluster_ids_x, cluster_center = kmeans(
                X=embedding_img,
                num_clusters=self.cluster_number, # TODO
                distance='euclidean',
                device=device,
                iter_limit=100,  # 最大迭代次数，默认是 10
            )
            cluster_center = cluster_center.to(device)
        return cluster_center

    def OT(self, src_dis, tgt_dis):
        # print(src_dis.shape, tgt_dis.shape)
        src_dis_norm = F.normalize(src_dis, dim=1)
        tgt_dis_norm = F.normalize(tgt_dis, dim=1)
        sim = torch.einsum('md,nd->mn', src_dis_norm, tgt_dis_norm).contiguous()
        wdist = 1 - sim
        xx = torch.full((src_dis.shape[0],), 1.0 / src_dis.shape[0], dtype=sim.dtype, device=sim.device)
        yy = torch.full((tgt_dis.shape[0],), 1.0 / tgt_dis.shape[0], dtype=sim.dtype, device=sim.device)
        with torch.no_grad():
            KK = torch.exp(-wdist / 0.1)
            T = self.Sinkhorn(KK, xx, yy)
        if torch.isnan(T).any():
            return None
        sim_op = torch.sum(T * sim, dim=(0, 1))
        loss = torch.sum(sim_op)
        return loss

    def Sinkhorn(self, K, u, v):
        r = torch.ones_like(u)
        c = torch.ones_like(v)
        thresh = 1e-2
        for i in range(100):
            r0 = r
            r = u / (K @ c.unsqueeze(-1)).squeeze(-1)
            c = v / (K.t() @ r.unsqueeze(-1)).squeeze(-1)
            err = (r - r0).abs().mean()
            if err.item() < thresh:
                break
        T = torch.outer(r, c) * K
        return T
    
class EnsembleFeatureLoss_OT_ablation_wo_local(nn.Module):
    def __init__(self, extractors: List[BaseFeatureExtractor], cluster_number=5):
        super(EnsembleFeatureLoss_OT_ablation_wo_local, self).__init__()
        self.extractors = nn.ModuleList(extractors)
        self.ground_truth = []
        self.ground_truth_local = []
        self.previous_loss_list=[]
        self.previous_loss_local_list = []
        self.cluster_number = cluster_number


    @torch.no_grad()
    def set_ground_truth(self, x: Tensor):
        self.ground_truth.clear()
        self.ground_truth_local.clear()
        for model in self.extractors:
            x_tensor, x_embedding = model.global_local_features(x.to(x.device))
            x_embedding = x_embedding.squeeze(0)
            cluster_center = self.get_cluster_center(x_embedding,x.device).unsqueeze(0)
            self.ground_truth.append(x_tensor)
            self.ground_truth_local.append(cluster_center)

    def __call__(self, feature_dict: Dict[int, Tensor],feature_local_dict: Dict[int, Tensor], y: Any = None) -> Tensor:
        loss_list = []
        loss_local_list = []
        for index, model in enumerate(self.extractors):
            
            gt = self.ground_truth[index]
            feature = feature_dict[index].unsqueeze(0)
            feat_loss = self.OT(gt,feature)

            loss_list.append(feat_loss)

        total_losses = [
            loss_list[i]
            for i in range(len(self.extractors))
        ]
        # 初始化 previous_loss_list（首次）
        if len(self.previous_loss_list) == 0:
            self.previous_loss_list = [l.detach() for l in total_losses]

        weights = []
        for i in range(len(self.extractors)):
            ratio = total_losses[i].item() / (self.previous_loss_list[i].item() + 1e-8)
            weights.append(ratio)
            # 归一化 softmax 计算动态权重
        
        T = 1.0
        K = len(weights)
        weights_np = np.array(weights)
        weights_softmax = np.exp(weights_np / T)
        weights_softmax /= np.sum(weights_softmax)
        weights_softmax *= K  # 可选：缩放为 K


        # 初始化 previous_loss_list（首次）
        for i in range(len(self.extractors)):
            self.previous_loss_list[i] = total_losses[i].detach()

        # 加权总损失
        total_loss = sum(
            weights_softmax[i] * total_losses[i]
            for i in range(len(self.extractors))
        )
        return total_loss

    def get_cluster_center(self, embedding_img,device):
        with suppress_output():
            cluster_ids_x, cluster_center = kmeans(
                X=embedding_img,
                num_clusters=self.cluster_number, # TODO
                distance='euclidean',
                device=device,
                iter_limit=100,  # 最大迭代次数，默认是 10
            )
            cluster_center = cluster_center.to(device)
        return cluster_center

    def OT(self, src_dis, tgt_dis):
        # print(src_dis.shape, tgt_dis.shape)
        src_dis_norm = F.normalize(src_dis, dim=1)
        tgt_dis_norm = F.normalize(tgt_dis, dim=1)
        sim = torch.einsum('md,nd->mn', src_dis_norm, tgt_dis_norm).contiguous()
        wdist = 1 - sim
        xx = torch.full((src_dis.shape[0],), 1.0 / src_dis.shape[0], dtype=sim.dtype, device=sim.device)
        yy = torch.full((tgt_dis.shape[0],), 1.0 / tgt_dis.shape[0], dtype=sim.dtype, device=sim.device)
        with torch.no_grad():
            KK = torch.exp(-wdist / 0.1)
            T = self.Sinkhorn(KK, xx, yy)
        if torch.isnan(T).any():
            return None
        sim_op = torch.sum(T * sim, dim=(0, 1))
        loss = torch.sum(sim_op)
        return loss

    def Sinkhorn(self, K, u, v):
        r = torch.ones_like(u)
        c = torch.ones_like(v)
        thresh = 1e-2
        for i in range(100):
            r0 = r
            r = u / (K @ c.unsqueeze(-1)).squeeze(-1)
            c = v / (K.t() @ r.unsqueeze(-1)).squeeze(-1)
            err = (r - r0).abs().mean()
            if err.item() < thresh:
                break
        T = torch.outer(r, c) * K
        return T
    
class EnsembleFeatureLoss_OT_ablation_wo_dynamic(nn.Module):
    def __init__(self, extractors: List[BaseFeatureExtractor], cluster_number=5):
        super(EnsembleFeatureLoss_OT_ablation_wo_dynamic, self).__init__()
        self.extractors = nn.ModuleList(extractors)
        self.ground_truth = []
        self.ground_truth_local = []
        self.previous_loss_list=[]
        self.previous_loss_local_list = []
        self.cluster_number = cluster_number


    @torch.no_grad()
    def set_ground_truth(self, x: Tensor):
        self.ground_truth.clear()
        self.ground_truth_local.clear()
        for model in self.extractors:
            x_tensor, x_embedding = model.global_local_features(x.to(x.device))
            x_embedding = x_embedding.squeeze(0)
            cluster_center = self.get_cluster_center(x_embedding,x.device).unsqueeze(0)
            self.ground_truth.append(x_tensor)
            self.ground_truth_local.append(cluster_center)

    def __call__(self, feature_dict: Dict[int, Tensor],feature_local_dict: Dict[int, Tensor], y: Any = None) -> Tensor:
        loss_list = []
        loss_local_list = []
        for index, model in enumerate(self.extractors):
            
            gt = self.ground_truth[index]
            feature = feature_dict[index].unsqueeze(0)
            feat_loss = self.OT(gt,feature)

            loss_list.append(feat_loss)

        total_losses = [
            loss_list[i]
            for i in range(len(self.extractors))
        ]

        # 加权总损失
        total_loss = sum(total_losses)/len(total_losses)
        return total_loss

    def get_cluster_center(self, embedding_img,device):
        with suppress_output():
            cluster_ids_x, cluster_center = kmeans(
                X=embedding_img,
                num_clusters=self.cluster_number, # TODO
                distance='euclidean',
                device=device,
                iter_limit=100,  # 最大迭代次数，默认是 10
            )
            cluster_center = cluster_center.to(device)
        return cluster_center

    def OT(self, src_dis, tgt_dis):
        # print(src_dis.shape, tgt_dis.shape)
        src_dis_norm = F.normalize(src_dis, dim=1)
        tgt_dis_norm = F.normalize(tgt_dis, dim=1)
        sim = torch.einsum('md,nd->mn', src_dis_norm, tgt_dis_norm).contiguous()
        wdist = 1 - sim
        xx = torch.full((src_dis.shape[0],), 1.0 / src_dis.shape[0], dtype=sim.dtype, device=sim.device)
        yy = torch.full((tgt_dis.shape[0],), 1.0 / tgt_dis.shape[0], dtype=sim.dtype, device=sim.device)
        with torch.no_grad():
            KK = torch.exp(-wdist / 0.1)
            T = self.Sinkhorn(KK, xx, yy)
        if torch.isnan(T).any():
            return None
        sim_op = torch.sum(T * sim, dim=(0, 1))
        loss = torch.sum(sim_op)
        return loss

    def Sinkhorn(self, K, u, v):
        r = torch.ones_like(u)
        c = torch.ones_like(v)
        thresh = 1e-2
        for i in range(100):
            r0 = r
            r = u / (K @ c.unsqueeze(-1)).squeeze(-1)
            c = v / (K.t() @ r.unsqueeze(-1)).squeeze(-1)
            err = (r - r0).abs().mean()
            if err.item() < thresh:
                break
        T = torch.outer(r, c) * K
        return T'''
    
    UPSTREAM['foa']["sources"]['surrogates/FeatureExtractors/ClipB16.py'] = r'''import torch
from transformers import CLIPVisionModel, CLIPProcessor, CLIPModel
from .Base import BaseFeatureExtractor
from torchvision import transforms






class ClipB16FeatureExtractor(BaseFeatureExtractor):
    def __init__(self):
        super(ClipB16FeatureExtractor, self).__init__()
        self.model = CLIPModel.from_pretrained("openai/clip-vit-base-patch16")
        self.processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch16")
        self.normalizer = transforms.Compose(
        [
            transforms.Resize(224, interpolation=transforms.InterpolationMode.BICUBIC, antialias=True),
            transforms.Lambda(lambda img: torch.clamp(img, 0.0, 255.0) / 255.0),
            transforms.CenterCrop(224),
            transforms.Normalize((0.48145466, 0.4578275, 0.40821073), (0.26862954, 0.26130258, 0.27577711)), # CLIP imgs mean and std.
        ]
    )

    def forward(self, x):
        # x = torch.clamp(x, min=0, max=1)
        inputs = dict(pixel_values=self.normalizer(x))
        image_features = self.model.get_image_features(**inputs)
        image_features = image_features / image_features.norm(dim=1, keepdim=True)
        return image_features

    def global_local_features(self, x):
        # x = torch.clamp(x, min=0, max=1)
        inputs = dict(pixel_values=self.normalizer(x))
        # image_features = self.model.get_image_features(**inputs)
        # image_features = image_features / image_features.norm(dim=1, keepdim=True)

        inputs["pixel_values"] = inputs["pixel_values"]
        outputs = self.model.vision_model(pixel_values=inputs['pixel_values'])
        features = outputs.last_hidden_state
        global_feature = features[:, 0, :]
        global_feature = global_feature / global_feature.norm(dim=1, keepdim=True)
        local_feature = features[:, 1:, :]
        local_feature = local_feature / local_feature.norm(dim=1, keepdim=True)
        # features = self.model.get_image_embedding(inputs["pixel_values"])
        # global_feature = features[:, 0, :]
        # local_feature = features[:, 1:, :]
        return global_feature, local_feature


class ClipB16FeatureExtractorOT(BaseFeatureExtractor):
    def __init__(self):
        super(ClipB16FeatureExtractorOT, self).__init__()
        self.model = CLIPModel.from_pretrained("openai/clip-vit-base-patch16")
        self.processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch16")
        self.normalizer = transforms.Compose(
            [
                transforms.Resize(224, interpolation=transforms.InterpolationMode.BICUBIC, antialias=True),
                transforms.Lambda(lambda img: torch.clamp(img, 0.0, 255.0) / 255.0),
                transforms.CenterCrop(224),
                transforms.Normalize((0.48145466, 0.4578275, 0.40821073), (0.26862954, 0.26130258, 0.27577711)),
                # CLIP imgs mean and std.
            ]
        )

    def forward(self, x):
        x = torch.clamp(x, min=0, max=1)
        inputs = dict(pixel_values=self.normalizer(x))
        inputs["pixel_values"] = inputs["pixel_values"].to(self.device)
        features = self.model.get_image_embedding(inputs["pixel_values"])
        global_feature = features[:,0,:]
        local_feature = features[:,1:,:]
        return global_feature, local_feature'''
    
    UPSTREAM['foa']["sources"]['surrogates/FeatureExtractors/ClipB32.py'] = r'''import torch
from transformers import CLIPVisionModel, CLIPProcessor, CLIPModel
from .Base import BaseFeatureExtractor
from torchvision import transforms


class ClipB32FeatureExtractor(BaseFeatureExtractor):
    def __init__(self):
        super(ClipB32FeatureExtractor, self).__init__()
        self.model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
        self.processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
        self.normalizer = transforms.Compose(
        [
            transforms.Resize(224, interpolation=transforms.InterpolationMode.BICUBIC, antialias=True),
            transforms.Lambda(lambda img: torch.clamp(img, 0.0, 255.0) / 255.0),
            transforms.CenterCrop(224),
            transforms.Normalize((0.48145466, 0.4578275, 0.40821073), (0.26862954, 0.26130258, 0.27577711)), # CLIP imgs mean and std.
        ]
    )

    def forward(self, x):
        # x = torch.clamp(x, min=0, max=1)
        inputs = dict(pixel_values=self.normalizer(x))
        image_features = self.model.get_image_features(**inputs)
        image_features = image_features / image_features.norm(dim=1, keepdim=True)
        return image_features

    def global_local_features(self, x):
        # x = torch.clamp(x, min=0, max=1)
        inputs = dict(pixel_values=self.normalizer(x))
        # image_features = self.model.get_image_features(**inputs)
        # image_features = image_features / image_features.norm(dim=1, keepdim=True)

        inputs["pixel_values"] = inputs["pixel_values"]
        outputs = self.model.vision_model(pixel_values=inputs['pixel_values'])
        features = outputs.last_hidden_state
        global_feature = features[:, 0, :]
        global_feature = global_feature / global_feature.norm(dim=1, keepdim=True)
        local_feature = features[:, 1:, :]
        local_feature = local_feature / local_feature.norm(dim=1, keepdim=True)
        # features = self.model.get_image_embedding(inputs["pixel_values"])
        # global_feature = features[:, 0, :]
        # local_feature = features[:, 1:, :]
        return global_feature, local_feature
class ClipB32FeatureExtractorOT(BaseFeatureExtractor):
    def __init__(self):
        super(ClipB32FeatureExtractorOT, self).__init__()
        self.model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
        self.processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
        self.normalizer = transforms.Compose(
            [
                transforms.Resize(224, interpolation=transforms.InterpolationMode.BICUBIC, antialias=True),
                transforms.Lambda(lambda img: torch.clamp(img, 0.0, 255.0) / 255.0),
                transforms.CenterCrop(224),
                transforms.Normalize((0.48145466, 0.4578275, 0.40821073), (0.26862954, 0.26130258, 0.27577711)),
                # CLIP imgs mean and std.
            ]
        )

    def forward(self, x):
        x = torch.clamp(x, min=0, max=1)
        inputs = dict(pixel_values=self.normalizer(x))
        inputs["pixel_values"] = inputs["pixel_values"].to(self.device)
        features = self.model.get_image_embedding(inputs["pixel_values"])
        global_feature = features[:,0,:]
        local_feature = features[:,1:,:]
        return global_feature, local_feature










'''
    
    UPSTREAM['foa']["sources"]['surrogates/FeatureExtractors/ClipLaion.py'] = r'''import torch
from transformers import CLIPVisionModel, CLIPProcessor, CLIPModel
from .Base import BaseFeatureExtractor
from torchvision import transforms


class ClipLaionFeatureExtractor(BaseFeatureExtractor):
    def __init__(self):
        super(ClipLaionFeatureExtractor, self).__init__()
        self.model = CLIPModel.from_pretrained("laion/CLIP-ViT-G-14-laion2B-s12B-b42K")
        # self.processor = CLIPProcessor.from_pretrained("openai/clip-vit-large-patch14-336")
        self.normalizer = transforms.Compose(
        [
            transforms.Resize(224, interpolation=transforms.InterpolationMode.BICUBIC, antialias=True),
            transforms.Lambda(lambda img: torch.clamp(img, 0.0, 255.0) / 255.0),
            transforms.CenterCrop(224),
            transforms.Normalize((0.48145466, 0.4578275, 0.40821073), (0.26862954, 0.26130258, 0.27577711)), # CLIP imgs mean and std.
        ]
    )

    def forward(self, x):
        # x = torch.clamp(x, min=0, max=1)
        inputs = dict(pixel_values=self.normalizer(x))
        image_features = self.model.get_image_features(**inputs)
        image_features = image_features / image_features.norm(dim=1, keepdim=True)
        return image_features

    def global_local_features(self, x):
        # x = torch.clamp(x, min=0, max=1)
        inputs = dict(pixel_values=self.normalizer(x))
        # image_features = self.model.get_image_features(**inputs)
        # image_features = image_features / image_features.norm(dim=1, keepdim=True)

        inputs["pixel_values"] = inputs["pixel_values"]
        outputs = self.model.vision_model(pixel_values=inputs['pixel_values'])
        features = outputs.last_hidden_state
        global_feature = features[:, 0, :]
        global_feature = global_feature / global_feature.norm(dim=1, keepdim=True)
        local_feature = features[:, 1:, :]
        local_feature = local_feature / local_feature.norm(dim=1, keepdim=True)
        # features = self.model.get_image_embedding(inputs["pixel_values"])
        # global_feature = features[:, 0, :]
        # local_feature = features[:, 1:, :]
        return global_feature, local_feature

class ClipLaionFeatureExtractorOT(BaseFeatureExtractor):
    def __init__(self):
        super(ClipLaionFeatureExtractorOT, self).__init__()
        self.model = CLIPModel.from_pretrained("laion/CLIP-ViT-G-14-laion2B-s12B-b42K")
        # self.processor = CLIPProcessor.from_pretrained("openai/clip-vit-large-patch14-336")
        self.normalizer = transforms.Compose(
            [
                transforms.Resize(224, interpolation=transforms.InterpolationMode.BICUBIC, antialias=True),
                transforms.Lambda(lambda img: torch.clamp(img, 0.0, 255.0) / 255.0),
                transforms.CenterCrop(224),
                transforms.Normalize((0.48145466, 0.4578275, 0.40821073), (0.26862954, 0.26130258, 0.27577711)),
                # CLIP imgs mean and std.
            ]
        )

    def forward(self, x):
        x = torch.clamp(x, min=0, max=1)
        inputs = dict(pixel_values=self.normalizer(x))
        inputs["pixel_values"] = inputs["pixel_values"].to(self.device)
        features = self.model.get_image_embedding(inputs["pixel_values"])
        global_feature = features[:,0,:]
        local_feature = features[:,1:,:]
        return global_feature, local_feature
'''
    
    UPSTREAM['foa']["sources"]['config/ensemble_3models.yaml'] = r'''# Configuration for ensemble_3models.py

defaults:
  - _self_

data:
  batch_size: 1
  num_samples: 100
  output: "./foa_attack_output"
  cle_data_path: "resources/images/bigscale"
  tgt_data_path: "resources/images/target_images"

optim:
  alpha: 1.0
  epsilon: 16
  steps: 300 # 300

model:
  input_res: 224
  use_source_crop: true
  use_target_crop: true
  crop_scale: [0.5, 0.9] 
  ensemble: true
  device: "cuda:0"# Using GPU 1
  backbone: ["B16", "B32", "Laion"]  # Use all models in ensemble

wandb:
  project: "local_adversarial_attack"
  entity: ??? # fill your wandb entity

blackbox:
  model_name: "gemini"  # Can be gpt4o, claude, gemini, gpt_4v
  batch_size: 1
  timeout: 30

attack: 'fgsm' # [fgsm, mifgsm, pgd]
'''
    
    UPSTREAM['mpc'] = {'repo': 'LiYuanBoJNU/MPCAttack', 'commit': '7b195a5bcbaeaf1e8534d101199810a2782caad3', 'license': 'MIT License\n\nCopyright (c) 2026 yb_li\n\nPermission is hereby granted, free of charge, to any person obtaining a copy\nof this software and associated documentation files (the "Software"), to deal\nin the Software without restriction, including without limitation the rights\nto use, copy, modify, merge, publish, distribute, sublicense, and/or sell\ncopies of the Software, and to permit persons to whom the Software is\nfurnished to do so, subject to the following conditions:\n\nThe above copyright notice and this permission notice shall be included in all\ncopies or substantial portions of the Software.\n\nTHE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR\nIMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,\nFITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE\nAUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER\nLIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,\nOUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE\nSOFTWARE.\n'}
    
    UPSTREAM['mpc']["sources"] = {}
    
    UPSTREAM['mpc']["sources"]['generate_adversarial_examples_MPCAttack.py'] = r'''import os
import argparse
import random
from typing import List, Dict

import numpy as np
import torch
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch import nn
from tqdm import tqdm
from PIL import Image

from surrogates import (
    DINOv2FeatureExtractor,
    InternVL3_1B_FeatureExtractor,
    ClipB16FeatureExtractor,
    ClipL336FeatureExtractor,
    ClipB32FeatureExtractor,
    ClipLaionFeatureExtractor,
    EnsembleFeatureExtractor_our,
)

from utils import ensure_dir, info_nce_loss

CLIP_BACKBONE_MAP: Dict[str, type] = {
    "L336": ClipL336FeatureExtractor,
    "B16": ClipB16FeatureExtractor,
    "B32": ClipB32FeatureExtractor,
    "Laion": ClipLaionFeatureExtractor,
}


def set_seed(seed=2023):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def to_tensor(pic):
    img = torch.from_numpy(np.array(pic, np.uint8, copy=True))
    img = img.view(pic.size[1], pic.size[0], len(pic.getbands()))
    img = img.permute((2, 0, 1)).contiguous()
    return img.float()


class ImageFolderWithPaths(torchvision.datasets.ImageFolder):
    def __getitem__(self, index):
        img, label = super().__getitem__(index)
        path, _ = self.samples[index]
        return img, label, path


def build_clip_models(args):
    if not args.ensemble and len(args.clip_backbone) > 1:
        raise ValueError("When ensemble=False, only one backbone allowed")

    models = []
    for name in args.clip_backbone:
        if name not in CLIP_BACKBONE_MAP:
            raise ValueError(f"Unknown clip_backbone: {name}")

        model = CLIP_BACKBONE_MAP[name]().eval().to(args.device)
        model.requires_grad_(False)
        models.append(model)

    if args.ensemble:
        extractor = EnsembleFeatureExtractor_our(models)
    else:
        extractor = models[0]

    return extractor, models



def extract_feature(img, clip_models, internvl_model, dino_model):
    clip_feats = clip_models(img)
    clip_feats = torch.cat(list(clip_feats.values()), dim=1)

    intern_feat = internvl_model(img).mean(dim=1)
    dino_feat = dino_model(img)

    z = torch.cat([clip_feats, intern_feat, dino_feat], dim=1)
    return z



def run_attack(args):
    device = args.device

    transform_fn = transforms.Compose([
        transforms.Resize(args.input_res,
                          interpolation=torchvision.transforms.InterpolationMode.BICUBIC),
        transforms.CenterCrop(args.input_res),
        transforms.Lambda(lambda img: img.convert("RGB")),
        transforms.Lambda(lambda img: to_tensor(img)),
    ])

    # input transformation
    source_crop = (transforms.RandomResizedCrop(args.input_res, scale=args.crop_scale))

    source_dataset = ImageFolderWithPaths(args.cle_data_path, transform=transform_fn)
    target_dataset = ImageFolderWithPaths(args.tgt_data_path, transform=transform_fn)

    source_loader = torch.utils.data.DataLoader(source_dataset, batch_size=args.batch_size, shuffle=False)
    target_loader = torch.utils.data.DataLoader(target_dataset, batch_size=args.batch_size, shuffle=False)

    # Loading models
    clip_models, _ = build_clip_models(args)
    internvl_model = InternVL3_1B_FeatureExtractor().eval().to(device)
    internvl_model.requires_grad_(False)
    dino_model = DINOv2FeatureExtractor().eval().to(device)
    dino_model.requires_grad_(False)

    for i, ((image_org, _, path_org), (image_tgt, _, path_tgt)) in enumerate(zip(source_loader, target_loader)):
        print(i, path_org, path_tgt)

        src_text = internvl_model.generate_description(image_org)
        tar_text = internvl_model.generate_description(image_tgt)
        print(src_text)
        print(tar_text)

        # Initialize perturbation
        delta = 16 / 255 * torch.randn_like(image_org).to(device)
        delta.requires_grad_(True)

        img_src = image_org.to(device)
        img_tgt = image_tgt.to(device)

        with torch.no_grad():
            text_feature_src = clip_models.get_text_features(src_text)  # list
            text_feature_tar = clip_models.get_text_features(tar_text)

            src_teats = clip_models(img_src)  # list [1, 512]
            tgt_teats = clip_models(img_tgt)

            src_feat_internvl = internvl_model(img_src)  # torch.Size([1, 256, 896])
            tgt_feat_internvl = internvl_model(img_tgt)

            src_feat_dino = dino_model(img_src)  # torch.Size([1, 768])
            tgt_feat_dino = dino_model(img_tgt)

            clip_src = []
            clip_tgt = []

            for index in range(len(src_teats)):
                src_feat_clip = args.lam * src_teats[index] + (1 - args.lam) * text_feature_src[index]
                tgt_feat_clip = args.lam * tgt_teats[index] + (1 - args.lam) * text_feature_tar[index]
                clip_src.append(src_feat_clip)
                clip_tgt.append(tgt_feat_clip)
            clip_src = torch.cat(clip_src, dim=1)
            clip_tgt = torch.cat(clip_tgt, dim=1)

            internvl_src = src_feat_internvl.mean(dim=1).float()
            dino_src = src_feat_dino
            z_src = torch.cat([clip_src, internvl_src, dino_src], dim=1)

            internvl_tgt = tgt_feat_internvl.mean(dim=1).float()
            dino_tgt = tgt_feat_dino
            z_tgt = torch.cat([clip_tgt, internvl_tgt, dino_tgt], dim=1)


        momentum = torch.zeros_like(delta, requires_grad=False)
        pbar = tqdm(range(300), desc=f"Attack Progress")
        for epoch in pbar:

            adv_img = img_src + delta
            local_cropped = source_crop(adv_img)

            adv_feats = clip_models(local_cropped)
            adv_feat_internvl = internvl_model(local_cropped)
            adv_feat_dino = dino_model(local_cropped)

            clip_adv = torch.cat(list(adv_feats.values()), dim=1)
            internvl_adv = adv_feat_internvl.mean(dim=1).float()
            dino_adv = adv_feat_dino
            z_adv = torch.cat([clip_adv, internvl_adv, dino_adv], dim=1)
            z_sim_src = torch.mean(F.cosine_similarity(z_adv, z_src.detach(), dim=-1))
            z_sim_tgt = torch.mean(F.cosine_similarity(z_adv, z_tgt.detach(), dim=-1))

            loss = info_nce_loss(z_adv, z_tgt.detach(), z_src.detach(), temperature=args.tau, omega=args.omega)  # infonce loss


            if epoch % 10 == 0:
                print(f'epoch:{epoch}', f'loss:{loss.item()}', f'z_sim_src:{z_sim_src.item()}', f'z_sim_tgt:{z_sim_tgt.item()}')


            grad = torch.autograd.grad(loss, delta, create_graph=False)[0]

            # MI-FGSM update
            momentum = momentum * 0.9 + grad
            delta.data = torch.clamp(
                delta + args.alpha * torch.sign(momentum),
                min=-args.epsilon,
                max=args.epsilon,
            )
            # torchvision.utils.save_image(delta.detach().cpu(), f"delta1/delta_{epoch}.png")

        # Create final adversarial image
        adv_image = image_org + delta.detach().cpu()
        adv_image = torch.clamp(adv_image / 255.0, 0.0, 1.0)

        adv_text = internvl_model.generate_description(adv_image * 255)
        print(tar_text)
        print(adv_text)
        # sys.exit()


        # Save images
        for path_idx in range(len(path_org)):
            folder, name = (
                os.path.basename(os.path.dirname(path_org[path_idx])),
                os.path.basename(path_org[path_idx]),
            )
            folder_to_save = os.path.join(args.output, "img", folder)
            ensure_dir(folder_to_save)

            if "JPEG" in name:
                torchvision.utils.save_image(
                    adv_image[path_idx], os.path.join(folder_to_save, name[:-4]) + "png"
                )
            elif "png" in name:
                torchvision.utils.save_image(
                    adv_image[path_idx], os.path.join(folder_to_save, name)
                )




def get_args():
    parser = argparse.ArgumentParser()

    # Data
    parser.add_argument("--cle_data_path", type=str, default="resources/images/bigscale")
    parser.add_argument("--tgt_data_path", type=str, default="resources/images/target_images")
    parser.add_argument("--output", type=str, default="./MPCAttack")
    parser.add_argument("--batch_size", type=int, default=1)

    # Model
    parser.add_argument("--input_res", type=int, default=224)
    parser.add_argument("--crop_scale", nargs=2, type=float, default=[0.5, 0.9])
    parser.add_argument("--ensemble", default=True)
    parser.add_argument("--device", type=str, default="cuda:0")
    parser.add_argument("--clip_backbone", nargs="+", default=["B16", "B32", "Laion"])

    # Optim
    parser.add_argument("--alpha", type=float, default=1.0)
    parser.add_argument("--epsilon", type=float, default=16)
    parser.add_argument("--steps", type=int, default=300)

    # MPCAttack Parameter
    parser.add_argument("--lam", type=float, default=0.6)
    parser.add_argument("--tau", type=float, default=0.2)
    parser.add_argument("--omega", type=float, default=2.0)

    return parser.parse_args()


def main():
    args = get_args()
    set_seed(2023)
    run_attack(args)


if __name__ == "__main__":
    main()
'''.replace('\n', '\r\n')
    
    UPSTREAM['mpc']["sources"]['utils.py'] = r'''"""Shared utilities for adversarial attack and text generation models."""

import os
import json
import sys

import yaml
import hashlib
import base64
from typing import Dict, Any, List, Union
from omegaconf import OmegaConf
# import wandb
from config_schema import MainConfig
import torch
import torch.nn.functional as F



def info_nce_loss(feat_adv, feat_tgt, feat_src, temperature=0.2, mode='img', omega=2.0):
    """
    InfoNCE loss for pulling adv->tgt close, pushing adv->src away.

    Args:
        feat_adv: Tensor [B, D] adversarial features
        feat_tgt: Tensor [B, D] target features (positive)
        feat_src: Tensor [B, D] source features (negative)
        temperature: float, softmax temperature

    Returns:
        loss: scalar tensor
    """
    # L2 normalize
    feat_adv = F.normalize(feat_adv, dim=-1)
    feat_tgt = F.normalize(feat_tgt, dim=-1)
    feat_src = F.normalize(feat_src, dim=-1)
    # Cosine similarities
    if mode == 'text':
        sim_pos = torch.sum(feat_adv * feat_tgt, dim=-1).mean(dim=1) * omega / temperature  # [B]
        sim_neg = torch.sum(feat_adv * feat_src, dim=-1).mean(dim=1) / temperature  # [B]
    else:
        sim_pos = torch.sum(feat_adv * feat_tgt, dim=-1) * omega / temperature  # [B]
        sim_neg = torch.sum(feat_adv * feat_src, dim=-1) / temperature  # [B]
    # Stack positive and negative: shape [B, 2]
    logits = torch.stack([sim_pos, sim_neg], dim=1)
    labels = torch.zeros(feat_adv.size(0), dtype=torch.long, device=feat_adv.device)  # positives index=0
    # Cross-entropy loss
    loss = F.cross_entropy(logits, labels)
    return - loss


def load_api_keys() -> Dict[str, str]:
    """Load API keys from the api_keys file.
    
    Returns:
        Dict[str, str]: Dictionary containing API keys for different models
        
    Raises:
        FileNotFoundError: If no api_keys file is found
    """
    for ext in ['yaml', 'yml', 'json']:
        file_path = f'api_keys.{ext}'
        if os.path.exists(file_path):
            with open(file_path, 'r') as f:
                if ext in ['yaml', 'yml']:
                    return yaml.safe_load(f)
                else:
                    return json.load(f)
    
    raise FileNotFoundError(
        "API keys file not found. Please create api_keys.yaml, api_keys.yml, or api_keys.json "
        "in the root directory with your API keys."
    )


def get_api_key(model_name: str) -> str:
    """Get API key for specified model.
    
    Args:
        model_name: Name of the model to get API key for
        
    Returns:
        str: API key for the specified model
        
    Raises:
        KeyError: If API key for model is not found
    """
    api_keys = load_api_keys()
    if model_name not in api_keys:
        raise KeyError(
            f"API key for {model_name} not found in api_keys file. "
            f"Available models: {list(api_keys.keys())}"
        )
    return api_keys[model_name]


def hash_training_config(cfg: MainConfig) -> str:
    """Create a deterministic hash of training-relevant config parameters.
    
    Args:
        cfg: Configuration object containing model settings
        
    Returns:
        str: MD5 hash of the config parameters
    """
    # Convert backbone list to plain Python list
    if isinstance(cfg.model.backbone, (list, tuple)):
        backbone = list(cfg.model.backbone)
    else:
        backbone = OmegaConf.to_container(cfg.model.backbone)
        
    # Create config dict with converted values
    train_config = {
        "data": {
            "batch_size": int(cfg.data.batch_size),
            "num_samples": int(cfg.data.num_samples),
            "cle_data_path": str(cfg.data.cle_data_path),
            "tgt_data_path": str(cfg.data.tgt_data_path),
        },
        "optim": {
            "alpha": float(cfg.optim.alpha),
            "epsilon": int(cfg.optim.epsilon),
            "steps": int(cfg.optim.steps),
        },
        "model": {
            "input_res": int(cfg.model.input_res),
            "use_source_crop": bool(cfg.model.use_source_crop),
            "use_target_crop": bool(cfg.model.use_target_crop),
            "crop_scale": tuple(float(x) for x in cfg.model.crop_scale),
            "ensemble": bool(cfg.model.ensemble),
            "backbone": backbone,
        },
        "attack": cfg.attack,
    }
    
    # Convert to JSON string with sorted keys
    json_str = json.dumps(train_config, sort_keys=True)
    return hashlib.md5(json_str.encode()).hexdigest()


# def setup_wandb(cfg: MainConfig, tags=None) -> None:
#     """Initialize Weights & Biases logging.
#
#     Args:
#         cfg: Configuration object containing wandb settings
#     """
#     config_dict = OmegaConf.to_container(cfg, resolve=True)
#     wandb.init(
#         project=cfg.wandb.project,
#         config=config_dict,
#         tags=tags,
#     )


def encode_image(image_path: str) -> str:
    """Encode image file to base64 string.
    
    Args:
        image_path: Path to image file
        
    Returns:
        str: Base64 encoded image string
    """
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")


def ensure_dir(path: str) -> None:
    """Ensure directory exists, create if it doesn't.
    
    Args:
        path: Directory path to ensure exists
    """
    os.makedirs(path, exist_ok=True)


def get_output_paths(cfg: MainConfig, config_hash: str) -> Dict[str, str]:
    """Get dictionary of output paths based on config.
    
    Args:
        cfg: Configuration object
        config_hash: Hash of training config
        
    Returns:
        Dict[str, str]: Dictionary containing output paths
    """
    return {
        'output_dir': os.path.join(cfg.data.output, "img", config_hash),
        'desc_output_dir': os.path.join(cfg.data.output, "description", config_hash)
    } '''
    
    UPSTREAM['mpc']["sources"]['surrogates/FeatureExtractors/Base.py'] = r'''import torch
from torch import nn, Tensor
from abc import abstractmethod
from typing import List, Any, Callable, Dict
from kmeans_pytorch import kmeans
import torch.nn.functional as F
from torchvision import transforms as T
import sys
import os
from contextlib import contextmanager
import numpy as np
@contextmanager
def suppress_output():
    with open(os.devnull, 'w') as fnull:
        old_stdout = sys.stdout
        old_stderr = sys.stderr
        sys.stdout = fnull
        sys.stderr = fnull
        try:
            yield
        finally:
            sys.stdout = old_stdout
            sys.stderr = old_stderr

class BaseFeatureExtractor(nn.Module):
    def __init__(self):
        super(BaseFeatureExtractor, self).__init__()
        pass

    @abstractmethod
    def forward(self, x: Tensor) -> Tensor:
        pass


class EnsembleFeatureExtractor_smolvlm(BaseFeatureExtractor):
    def __init__(self, extractors: List[BaseFeatureExtractor]):
        super(EnsembleFeatureExtractor_smolvlm, self).__init__()
        self.extractors = nn.ModuleList(extractors)

    def forward(self, x: Tensor) -> Tensor:
        # features = []
        # for model in self.extractors:
        #     features.append(model(x).squeeze())
        # features = torch.cat(features, dim=0)
        output = {}  # 不拼接，改为字典存储
        for i, model in enumerate(self.extractors):
            output[i] = model(x)
        return output

    def get_text_features(self, text):
        text_features = {}  # 不拼接，改为字典存储
        for i, model in enumerate(self.extractors):
            text_features[i] = model.text_features_get(text).squeeze()
        return text_features

class EnsembleFeatureExtractor_our(BaseFeatureExtractor):
    def __init__(self, extractors: List[BaseFeatureExtractor]):
        super(EnsembleFeatureExtractor_our, self).__init__()
        self.extractors = nn.ModuleList(extractors)

    def forward(self, x: Tensor) -> Tensor:
        # features = []
        # for model in self.extractors:
        #     features.append(model(x).squeeze())
        # features = torch.cat(features, dim=0)
        features = {}
        for i, model in enumerate(self.extractors):
            # features[i] = model(x).squeeze()
            features[i] = model(x)
        return features

    def get_text_features(self, text):
        text_features = {}  # 不拼接，改为字典存储
        for i, model in enumerate(self.extractors):
            # text_features[i] = model.text_features_get(text).squeeze()
            text_features[i] = model.text_features_get(text)
        return text_features

class EnsembleFeatureExtractor(BaseFeatureExtractor):
    def __init__(self, extractors: List[BaseFeatureExtractor]):
        super(EnsembleFeatureExtractor, self).__init__()
        self.extractors = nn.ModuleList(extractors)

    def forward(self, x: Tensor) -> Tensor:
        # features = []
        # for model in self.extractors:
        #     features.append(model(x).squeeze())
        # features = torch.cat(features, dim=0)
        features = {}  # 不拼接，改为字典存储
        for i, model in enumerate(self.extractors):
            features[i] = model(x).squeeze()

        return features

    def get_text_features(self, text):
        text_features = {}  # 不拼接，改为字典存储
        for i, model in enumerate(self.extractors):
            text_features[i] = model.text_features_get(text).squeeze()

        return text_features



class EnsembleFeatureExtractor_ot(BaseFeatureExtractor):
    def __init__(self, extractors: List[BaseFeatureExtractor],cluster_number=5):
        super(EnsembleFeatureExtractor_ot, self).__init__()
        self.extractors = nn.ModuleList(extractors)
        self.cluster_number = cluster_number

    def forward(self, x: Tensor) -> Tensor:
        # features = []
        # for model in self.extractors:
        #     features.append(model(x).squeeze())
        # features = torch.cat(features, dim=0)
        features = {}  # 不拼接，改为字典存储
        features_local = {}
        for i, model in enumerate(self.extractors):
            # features[i] = model(x).squeeze()
            x_tensor, x_embedding = model.global_local_features(x.to(x.device))
            features[i] = x_tensor.squeeze()
            cluster_center = self.get_cluster_center(x_embedding[0],x.device).unsqueeze(0)
            features_local[i]=cluster_center

        return features,features_local

    def get_cluster_center(self, embedding_img,device):
        # self.setup_seed(20)
        # for i in range(10):
        # np.random.seed(20)
        with suppress_output():
            cluster_ids_x, cluster_center = kmeans(
                X=embedding_img,
                num_clusters=self.cluster_number,
                distance='euclidean',
                device=device
            )
            cluster_center = cluster_center.to(device)
        # print(cluster_ids_x)
        return cluster_center
    
class EnsembleFeatureExtractor_ot3(BaseFeatureExtractor):
    def __init__(self, extractors: List[BaseFeatureExtractor]):
        super(EnsembleFeatureExtractor_ot3, self).__init__()
        self.extractors = nn.ModuleList(extractors)

    def forward(self, x: Tensor) -> Tensor:
        # features = []
        # for model in self.extractors:
        #     features.append(model(x).squeeze())
        # features = torch.cat(features, dim=0)
        features = {}  # 不拼接，改为字典存储
        features_local = {}
        for i, model in enumerate(self.extractors):
            # features[i] = model(x).squeeze()
            x_tensor, x_embedding = model.global_local_features(x.to(x.device))
            features[i] = x_tensor.squeeze()
            cluster_center = self.get_cluster_center(x_embedding[0],x.device).unsqueeze(0)
            features_local[i]=cluster_center

        return features,features_local

    def get_cluster_center(self, embedding_img,device):
        # self.setup_seed(20)
        # for i in range(10):
        # np.random.seed(20)
        with suppress_output():
            cluster_ids_x, cluster_center = kmeans(
                X=embedding_img,
                num_clusters=3,
                distance='euclidean',
                device=device
            )
            cluster_center = cluster_center.to(device)
        # print(cluster_ids_x)
        return cluster_center


class EnsembleFeatureLoss(nn.Module):
    def __init__(self, extractors: List[BaseFeatureExtractor]):
        super(EnsembleFeatureLoss, self).__init__()
        self.extractors = nn.ModuleList(extractors)
        self.ground_truth = []

    @torch.no_grad()
    def set_ground_truth(self, x: Tensor):
        self.ground_truth.clear()
        for model in self.extractors:
            self.ground_truth.append(model(x).to(x.device))

    def __call__(self, feature_dict: Dict[int, Tensor], y: Any = None) -> Tensor:
        loss = 0
        for index, model in enumerate(self.extractors):
            gt = self.ground_truth[index]
            feature = feature_dict[index]
            loss += torch.mean(torch.sum(feature * gt, dim=1))
            
        loss = loss / len(self.extractors)

        return loss


class EnsembleFeatureLoss_MPCO(nn.Module):
    def __init__(self, extractors: List[BaseFeatureExtractor]):
        super(EnsembleFeatureLoss_MPCO, self).__init__()
        self.extractors = nn.ModuleList(extractors)
        self.ground_truth = []
        self.ground_src = []

    @torch.no_grad()
    def set_ground_truth(self, x: Tensor):
        self.ground_truth.clear()
        for model in self.extractors:
            self.ground_truth.append(model(x).to(x.device))

    @torch.no_grad()
    def set_ground_src(self, x: Tensor):
        self.ground_src.clear()
        for model in self.extractors:
            self.ground_src.append(model(x).to(x.device))

    def __call__(self, feature_dict: Dict[int, Tensor], y: Any = None) -> Tensor:

        tgt_l = []
        for index in range(len(self.ground_truth)):
            if len(self.ground_truth[index].size()) > 2:
                gt = self.ground_truth[index].mean(dim=1)
            else:
                gt = self.ground_truth[index]
            tgt_l.append(gt)

        src_l = []
        for index in range(len(self.ground_src)):
            if len(self.ground_src[index].size()) > 2:
                src = self.ground_src[index].mean(dim=1)
            else:
                src = self.ground_src[index]
            src_l.append(src)

        adv_l = []
        for index, model in enumerate(self.extractors):
            feature = feature_dict[index].unsqueeze(0)
            if len(feature.size()) > 2:
                adv = feature.mean(dim=1)
            else:
                adv = feature
            adv_l.append(adv)

        feat_tgt = torch.cat(tgt_l, dim=1)
        feat_src = torch.cat(src_l, dim=1)
        feat_adv = torch.cat(adv_l, dim=1)
        # print(feat_adv.size())
        feat_adv = F.normalize(feat_adv, dim=-1)
        feat_tgt = F.normalize(feat_tgt, dim=-1)
        feat_src = F.normalize(feat_src, dim=-1)
        loss = torch.mean(torch.sum(feat_adv * feat_tgt, dim=1)) - torch.mean(torch.sum(feat_adv * feat_src, dim=1))

        # omega = 0.2
        # temperature = 2.0
        # feat_adv = F.normalize(feat_adv, dim=-1)
        # feat_tgt = F.normalize(feat_tgt, dim=-1)
        # feat_src = F.normalize(feat_src, dim=-1)
        # sim_pos = torch.sum(feat_adv * feat_tgt, dim=-1) * omega / temperature  # [B]
        # sim_neg = torch.sum(feat_adv * feat_src, dim=-1) / temperature  # [B]
        # logits = torch.stack([sim_pos, sim_neg], dim=1)
        # labels = torch.zeros(feat_adv.size(0), dtype=torch.long, device=feat_adv.device)  # positives index=0
        # loss = F.cross_entropy(logits, labels)
        # print(torch.mean(torch.sum(feat_adv * feat_tgt, dim=1)))

        return loss


class EnsembleFeatureLoss_OT(nn.Module):
    def __init__(self, extractors: List[BaseFeatureExtractor]):
        super(EnsembleFeatureLoss_OT, self).__init__()
        self.extractors = nn.ModuleList(extractors)
        self.ground_truth = []
        self.ground_truth_local = []



    @torch.no_grad()
    def set_ground_truth(self, x: Tensor):
        self.ground_truth.clear()
        self.ground_truth_local.clear()
        for model in self.extractors:
            x_tensor, x_embedding = model.global_local_features(x.to(x.device))
            x_embedding = x_embedding.squeeze(0)
            cluster_center = self.get_cluster_center(x_embedding).unsqueeze(0)
            self.ground_truth.append(x_tensor)
            self.ground_truth_local.append(cluster_center)



    def __call__(self, feature_dict: Dict[int, Tensor],feature_local_dict: Dict[int, Tensor], y: Any = None) -> Tensor:
        loss = 0
        loss_local=0
        for index, model in enumerate(self.extractors):
            gt_local = self.ground_truth_local[index].squeeze(0)
            gt = self.ground_truth[index]
            feature = feature_dict[index]
            feature_local = feature_local_dict[index].squeeze(0)
            # print("gt_local",gt_local.shape)
            # print("feature_local", feature_local.shape)
            loss_local += self.OT(gt_local, feature_local) * 2

            loss += torch.mean(torch.sum(feature * gt, dim=1))

        loss = loss / len(self.extractors)
        loss_local=loss_local / len(self.extractors)

        # print("loss",loss)
        # print("loss_local", loss_local)
        loss=loss+loss_local*0.1
        return loss

    def get_cluster_center(self, embedding_img):
        # self.setup_seed(20)
        # for i in range(10):
        # np.random.seed(20)
        with suppress_output():
            cluster_ids_x, cluster_center = kmeans(
                X=embedding_img,
                num_clusters=5,
                distance='euclidean',
                device=torch.device('cuda'),
                iter_limit=100,  # 最大迭代次数，默认是 10
            )
            cluster_center = cluster_center.to('cuda')
        # print(cluster_ids_x)
        return cluster_center

    def OT(self, src_dis, tgt_dis):
        # print(src_dis.shape, tgt_dis.shape)
        src_dis_norm = F.normalize(src_dis, dim=1)
        tgt_dis_norm = F.normalize(tgt_dis, dim=1)
        sim = torch.einsum('md,nd->mn', src_dis_norm, tgt_dis_norm).contiguous()
        wdist = 1 - sim
        xx = torch.full((src_dis.shape[0],), 1.0 / src_dis.shape[0], dtype=sim.dtype, device=sim.device)
        yy = torch.full((tgt_dis.shape[0],), 1.0 / tgt_dis.shape[0], dtype=sim.dtype, device=sim.device)
        with torch.no_grad():
            KK = torch.exp(-wdist / 0.1)
            T = self.Sinkhorn(KK, xx, yy)
        if torch.isnan(T).any():
            return None
        sim_op = torch.sum(T * sim, dim=(0, 1))
        loss = torch.sum(sim_op)
        return loss

    def Sinkhorn(self, K, u, v):
        r = torch.ones_like(u)
        c = torch.ones_like(v)
        thresh = 1e-2
        for i in range(100):
            r0 = r
            r = u / (K @ c.unsqueeze(-1)).squeeze(-1)
            c = v / (K.t() @ r.unsqueeze(-1)).squeeze(-1)
            err = (r - r0).abs().mean()
            if err.item() < thresh:
                break
        T = torch.outer(r, c) * K
        return T


class EnsembleFeatureLoss_OT_Auto(nn.Module):
    def __init__(self, extractors: List[BaseFeatureExtractor]):
        super(EnsembleFeatureLoss_OT_Auto, self).__init__()
        self.extractors = nn.ModuleList(extractors)
        self.ground_truth = []
        self.ground_truth_local = []
        self.previous_loss_list=[]
        self.previous_loss_local_list = []



    @torch.no_grad()
    def set_ground_truth(self, x: Tensor):
        self.ground_truth.clear()
        self.ground_truth_local.clear()
        for model in self.extractors:
            x_tensor, x_embedding = model.global_local_features(x.to(x.device))
            x_embedding = x_embedding.squeeze(0)
            cluster_center = self.get_cluster_center(x_embedding).unsqueeze(0)
            self.ground_truth.append(x_tensor)
            self.ground_truth_local.append(cluster_center)



    def __call__(self, feature_dict: Dict[int, Tensor],feature_local_dict: Dict[int, Tensor], y: Any = None) -> Tensor:
        loss_list = []
        loss_local_list = []
        for index, model in enumerate(self.extractors):
            gt_local = self.ground_truth_local[index].squeeze(0)
            gt = self.ground_truth[index]
            feature = feature_dict[index]
            feature_local = feature_local_dict[index].squeeze(0)
            # print("gt_local",gt_local.shape)
            # print("feature_local", feature_local.shape)
            local_loss = self.OT(gt_local, feature_local) * 2
            feat_loss = torch.mean(torch.sum(feature * gt, dim=1))

            loss_list.append(feat_loss)
            loss_local_list.append(local_loss)

        total_losses = [
            loss_list[i] + 0.1 * loss_local_list[i]
            for i in range(len(self.extractors))
        ]
        # 初始化 previous_loss_list（首次）
        if len(self.previous_loss_list) == 0:
            self.previous_loss_list = [l.detach() for l in total_losses]

        weights = []
        for i in range(len(self.extractors)):
            ratio = total_losses[i].item() / (self.previous_loss_list[i].item() + 1e-8)
            weights.append(ratio)
            # 归一化 softmax 计算动态权重
        T = 1.0
        K = len(weights)
        weights_np = np.array(weights)
        weights_softmax = np.exp(weights_np / T)
        weights_softmax /= np.sum(weights_softmax)
        weights_softmax *= K  # 可选：缩放为 K


        # 初始化 previous_loss_list（首次）
        for i in range(len(self.extractors)):
            self.previous_loss_list[i] = total_losses[i].detach()

        # 加权总损失
        total_loss = sum(
            weights_softmax[i] * total_losses[i]
            for i in range(len(self.extractors))
        )
        return total_loss

    def get_cluster_center(self, embedding_img):
        # self.setup_seed(20)
        # for i in range(10):
        # np.random.seed(20)
        with suppress_output():
            cluster_ids_x, cluster_center = kmeans(
                X=embedding_img,
                num_clusters=5,
                distance='euclidean',
                device=torch.device('cuda'),
                iter_limit=100,  # 最大迭代次数，默认是 10
            )
            cluster_center = cluster_center.to('cuda')
        # print(cluster_ids_x)
        return cluster_center

    def OT(self, src_dis, tgt_dis):
        # print(src_dis.shape, tgt_dis.shape)
        src_dis_norm = F.normalize(src_dis, dim=1)
        tgt_dis_norm = F.normalize(tgt_dis, dim=1)
        sim = torch.einsum('md,nd->mn', src_dis_norm, tgt_dis_norm).contiguous()
        wdist = 1 - sim
        xx = torch.full((src_dis.shape[0],), 1.0 / src_dis.shape[0], dtype=sim.dtype, device=sim.device)
        yy = torch.full((tgt_dis.shape[0],), 1.0 / tgt_dis.shape[0], dtype=sim.dtype, device=sim.device)
        with torch.no_grad():
            KK = torch.exp(-wdist / 0.1)
            T = self.Sinkhorn(KK, xx, yy)
        if torch.isnan(T).any():
            return None
        sim_op = torch.sum(T * sim, dim=(0, 1))
        loss = torch.sum(sim_op)
        return loss

    def Sinkhorn(self, K, u, v):
        r = torch.ones_like(u)
        c = torch.ones_like(v)
        thresh = 1e-2
        for i in range(100):
            r0 = r
            r = u / (K @ c.unsqueeze(-1)).squeeze(-1)
            c = v / (K.t() @ r.unsqueeze(-1)).squeeze(-1)
            err = (r - r0).abs().mean()
            if err.item() < thresh:
                break
        T = torch.outer(r, c) * K
        return T
    
class EnsembleFeatureLoss_OT_foa_attack(nn.Module):
    def __init__(self, extractors: List[BaseFeatureExtractor],cluster_number=5):
        super(EnsembleFeatureLoss_OT_foa_attack, self).__init__()
        self.extractors = nn.ModuleList(extractors)
        self.ground_truth = []
        self.ground_truth_local = []
        self.previous_loss_list=[]
        self.previous_loss_local_list = []
        self.cluster_number = cluster_number


    @torch.no_grad()
    def set_ground_truth(self, x: Tensor):
        self.ground_truth.clear()
        self.ground_truth_local.clear()
        for model in self.extractors:
            x_tensor, x_embedding = model.global_local_features(x.to(x.device))
            x_embedding = x_embedding.squeeze(0)
            cluster_center = self.get_cluster_center(x_embedding,x.device).unsqueeze(0)
            self.ground_truth.append(x_tensor)
            self.ground_truth_local.append(cluster_center)

    def __call__(self, feature_dict: Dict[int, Tensor],feature_local_dict: Dict[int, Tensor], y: Any = None) -> Tensor:
        loss_list = []
        loss_local_list = []
        for index, model in enumerate(self.extractors):
            gt_local = self.ground_truth_local[index].squeeze(0)
            gt = self.ground_truth[index]
            feature = feature_dict[index].unsqueeze(0)
            feature_local = feature_local_dict[index].squeeze(0)
            # print("gt_local",gt_local.shape)
            # print("feature_local", feature_local.shape)
            local_loss = self.OT(gt_local, feature_local)
            
            # feat_loss = torch.mean(torch.sum(feature * gt, dim=1))
            feat_loss = self.OT(gt,feature)

            loss_list.append(feat_loss)
            loss_local_list.append(local_loss)

        total_losses = [
            loss_list[i] + 0.2 * loss_local_list[i]
            for i in range(len(self.extractors))
        ]
        # 初始化 previous_loss_list（首次）
        if len(self.previous_loss_list) == 0:
            self.previous_loss_list = [l.detach() for l in total_losses]

        weights = []
        for i in range(len(self.extractors)):
            ratio = total_losses[i].item() / (self.previous_loss_list[i].item() + 1e-8)
            weights.append(ratio)
            # 归一化 softmax 计算动态权重
        
        T = 1.0
        K = len(weights)
        weights_np = np.array(weights)
        weights_softmax = np.exp(weights_np / T)
        weights_softmax /= np.sum(weights_softmax)
        weights_softmax *= K  # 可选：缩放为 K


        # 初始化 previous_loss_list（首次）
        for i in range(len(self.extractors)):
            self.previous_loss_list[i] = total_losses[i].detach()

        # 加权总损失
        total_loss = sum(
            weights_softmax[i] * total_losses[i]
            for i in range(len(self.extractors))
        )
        return total_loss

    def get_cluster_center(self, embedding_img,device):
        # self.setup_seed(20)
        # for i in range(10):
        # np.random.seed(20)
        with suppress_output():
            cluster_ids_x, cluster_center = kmeans(
                X=embedding_img,
                num_clusters=self.cluster_number, # TODO
                distance='euclidean',
                device=device,
                iter_limit=100,  # 最大迭代次数，默认是 10
            )
            cluster_center = cluster_center.to(device)
        # print(cluster_ids_x)
        return cluster_center

    def OT(self, src_dis, tgt_dis):
        # print(src_dis.shape, tgt_dis.shape)
        src_dis_norm = F.normalize(src_dis, dim=1)
        tgt_dis_norm = F.normalize(tgt_dis, dim=1)
        sim = torch.einsum('md,nd->mn', src_dis_norm, tgt_dis_norm).contiguous()
        wdist = 1 - sim
        xx = torch.full((src_dis.shape[0],), 1.0 / src_dis.shape[0], dtype=sim.dtype, device=sim.device)
        yy = torch.full((tgt_dis.shape[0],), 1.0 / tgt_dis.shape[0], dtype=sim.dtype, device=sim.device)
        with torch.no_grad():
            KK = torch.exp(-wdist / 0.1)
            T = self.Sinkhorn(KK, xx, yy)
        if torch.isnan(T).any():
            return None
        sim_op = torch.sum(T * sim, dim=(0, 1))
        loss = torch.sum(sim_op)
        return loss

    def Sinkhorn(self, K, u, v):
        r = torch.ones_like(u)
        c = torch.ones_like(v)
        thresh = 1e-2
        for i in range(100):
            r0 = r
            r = u / (K @ c.unsqueeze(-1)).squeeze(-1)
            c = v / (K.t() @ r.unsqueeze(-1)).squeeze(-1)
            err = (r - r0).abs().mean()
            if err.item() < thresh:
                break
        T = torch.outer(r, c) * K
        return T

class EnsembleFeatureLoss_OT_ablation_wo_global(nn.Module):
    def __init__(self, extractors: List[BaseFeatureExtractor], cluster_number=5):
        super(EnsembleFeatureLoss_OT_ablation_wo_global, self).__init__()
        self.extractors = nn.ModuleList(extractors)
        self.ground_truth = []
        self.ground_truth_local = []
        self.previous_loss_list=[]
        self.previous_loss_local_list = []
        self.cluster_number = cluster_number


    @torch.no_grad()
    def set_ground_truth(self, x: Tensor):
        self.ground_truth.clear()
        self.ground_truth_local.clear()
        for model in self.extractors:
            x_tensor, x_embedding = model.global_local_features(x.to(x.device))
            x_embedding = x_embedding.squeeze(0)
            cluster_center = self.get_cluster_center(x_embedding,x.device).unsqueeze(0)
            self.ground_truth.append(x_tensor)
            self.ground_truth_local.append(cluster_center)

    def __call__(self, feature_dict: Dict[int, Tensor],feature_local_dict: Dict[int, Tensor], y: Any = None) -> Tensor:
        loss_list = []
        loss_local_list = []
        for index, model in enumerate(self.extractors):
            gt_local = self.ground_truth_local[index].squeeze(0)
            gt = self.ground_truth[index]
            feature = feature_dict[index]
            feature_local = feature_local_dict[index].squeeze(0)
            local_loss = self.OT(gt_local, feature_local)
            feat_loss = torch.mean(torch.sum(feature * gt, dim=1))

            loss_list.append(feat_loss)
            loss_local_list.append(local_loss)

        total_losses = [
            loss_list[i] + 0.2 * loss_local_list[i]
            for i in range(len(self.extractors))
        ]
        # 初始化 previous_loss_list（首次）
        if len(self.previous_loss_list) == 0:
            self.previous_loss_list = [l.detach() for l in total_losses]

        weights = []
        for i in range(len(self.extractors)):
            ratio = total_losses[i].item() / (self.previous_loss_list[i].item() + 1e-8)
            weights.append(ratio)
            # 归一化 softmax 计算动态权重
        
        T = 1.0
        K = len(weights)
        weights_np = np.array(weights)
        weights_softmax = np.exp(weights_np / T)
        weights_softmax /= np.sum(weights_softmax)
        weights_softmax *= K  # 可选：缩放为 K


        # 初始化 previous_loss_list（首次）
        for i in range(len(self.extractors)):
            self.previous_loss_list[i] = total_losses[i].detach()

        # 加权总损失
        total_loss = sum(
            weights_softmax[i] * total_losses[i]
            for i in range(len(self.extractors))
        )
        return total_loss

    def get_cluster_center(self, embedding_img,device):
        with suppress_output():
            cluster_ids_x, cluster_center = kmeans(
                X=embedding_img,
                num_clusters=self.cluster_number, # TODO
                distance='euclidean',
                device=device,
                iter_limit=100,  # 最大迭代次数，默认是 10
            )
            cluster_center = cluster_center.to(device)
        return cluster_center

    def OT(self, src_dis, tgt_dis):
        # print(src_dis.shape, tgt_dis.shape)
        src_dis_norm = F.normalize(src_dis, dim=1)
        tgt_dis_norm = F.normalize(tgt_dis, dim=1)
        sim = torch.einsum('md,nd->mn', src_dis_norm, tgt_dis_norm).contiguous()
        wdist = 1 - sim
        xx = torch.full((src_dis.shape[0],), 1.0 / src_dis.shape[0], dtype=sim.dtype, device=sim.device)
        yy = torch.full((tgt_dis.shape[0],), 1.0 / tgt_dis.shape[0], dtype=sim.dtype, device=sim.device)
        with torch.no_grad():
            KK = torch.exp(-wdist / 0.1)
            T = self.Sinkhorn(KK, xx, yy)
        if torch.isnan(T).any():
            return None
        sim_op = torch.sum(T * sim, dim=(0, 1))
        loss = torch.sum(sim_op)
        return loss

    def Sinkhorn(self, K, u, v):
        r = torch.ones_like(u)
        c = torch.ones_like(v)
        thresh = 1e-2
        for i in range(100):
            r0 = r
            r = u / (K @ c.unsqueeze(-1)).squeeze(-1)
            c = v / (K.t() @ r.unsqueeze(-1)).squeeze(-1)
            err = (r - r0).abs().mean()
            if err.item() < thresh:
                break
        T = torch.outer(r, c) * K
        return T
    
class EnsembleFeatureLoss_OT_ablation_wo_local(nn.Module):
    def __init__(self, extractors: List[BaseFeatureExtractor], cluster_number=5):
        super(EnsembleFeatureLoss_OT_ablation_wo_local, self).__init__()
        self.extractors = nn.ModuleList(extractors)
        self.ground_truth = []
        self.ground_truth_local = []
        self.previous_loss_list=[]
        self.previous_loss_local_list = []
        self.cluster_number = cluster_number


    @torch.no_grad()
    def set_ground_truth(self, x: Tensor):
        self.ground_truth.clear()
        self.ground_truth_local.clear()
        for model in self.extractors:
            x_tensor, x_embedding = model.global_local_features(x.to(x.device))
            x_embedding = x_embedding.squeeze(0)
            cluster_center = self.get_cluster_center(x_embedding,x.device).unsqueeze(0)
            self.ground_truth.append(x_tensor)
            self.ground_truth_local.append(cluster_center)

    def __call__(self, feature_dict: Dict[int, Tensor],feature_local_dict: Dict[int, Tensor], y: Any = None) -> Tensor:
        loss_list = []
        loss_local_list = []
        for index, model in enumerate(self.extractors):
            
            gt = self.ground_truth[index]
            feature = feature_dict[index].unsqueeze(0)
            feat_loss = self.OT(gt,feature)

            loss_list.append(feat_loss)

        total_losses = [
            loss_list[i]
            for i in range(len(self.extractors))
        ]
        # 初始化 previous_loss_list（首次）
        if len(self.previous_loss_list) == 0:
            self.previous_loss_list = [l.detach() for l in total_losses]

        weights = []
        for i in range(len(self.extractors)):
            ratio = total_losses[i].item() / (self.previous_loss_list[i].item() + 1e-8)
            weights.append(ratio)
            # 归一化 softmax 计算动态权重
        
        T = 1.0
        K = len(weights)
        weights_np = np.array(weights)
        weights_softmax = np.exp(weights_np / T)
        weights_softmax /= np.sum(weights_softmax)
        weights_softmax *= K  # 可选：缩放为 K


        # 初始化 previous_loss_list（首次）
        for i in range(len(self.extractors)):
            self.previous_loss_list[i] = total_losses[i].detach()

        # 加权总损失
        total_loss = sum(
            weights_softmax[i] * total_losses[i]
            for i in range(len(self.extractors))
        )
        return total_loss

    def get_cluster_center(self, embedding_img,device):
        with suppress_output():
            cluster_ids_x, cluster_center = kmeans(
                X=embedding_img,
                num_clusters=self.cluster_number, # TODO
                distance='euclidean',
                device=device,
                iter_limit=100,  # 最大迭代次数，默认是 10
            )
            cluster_center = cluster_center.to(device)
        return cluster_center

    def OT(self, src_dis, tgt_dis):
        # print(src_dis.shape, tgt_dis.shape)
        src_dis_norm = F.normalize(src_dis, dim=1)
        tgt_dis_norm = F.normalize(tgt_dis, dim=1)
        sim = torch.einsum('md,nd->mn', src_dis_norm, tgt_dis_norm).contiguous()
        wdist = 1 - sim
        xx = torch.full((src_dis.shape[0],), 1.0 / src_dis.shape[0], dtype=sim.dtype, device=sim.device)
        yy = torch.full((tgt_dis.shape[0],), 1.0 / tgt_dis.shape[0], dtype=sim.dtype, device=sim.device)
        with torch.no_grad():
            KK = torch.exp(-wdist / 0.1)
            T = self.Sinkhorn(KK, xx, yy)
        if torch.isnan(T).any():
            return None
        sim_op = torch.sum(T * sim, dim=(0, 1))
        loss = torch.sum(sim_op)
        return loss

    def Sinkhorn(self, K, u, v):
        r = torch.ones_like(u)
        c = torch.ones_like(v)
        thresh = 1e-2
        for i in range(100):
            r0 = r
            r = u / (K @ c.unsqueeze(-1)).squeeze(-1)
            c = v / (K.t() @ r.unsqueeze(-1)).squeeze(-1)
            err = (r - r0).abs().mean()
            if err.item() < thresh:
                break
        T = torch.outer(r, c) * K
        return T
    
class EnsembleFeatureLoss_OT_ablation_wo_dynamic(nn.Module):
    def __init__(self, extractors: List[BaseFeatureExtractor], cluster_number=5):
        super(EnsembleFeatureLoss_OT_ablation_wo_dynamic, self).__init__()
        self.extractors = nn.ModuleList(extractors)
        self.ground_truth = []
        self.ground_truth_local = []
        self.previous_loss_list=[]
        self.previous_loss_local_list = []
        self.cluster_number = cluster_number


    @torch.no_grad()
    def set_ground_truth(self, x: Tensor):
        self.ground_truth.clear()
        self.ground_truth_local.clear()
        for model in self.extractors:
            x_tensor, x_embedding = model.global_local_features(x.to(x.device))
            x_embedding = x_embedding.squeeze(0)
            cluster_center = self.get_cluster_center(x_embedding,x.device).unsqueeze(0)
            self.ground_truth.append(x_tensor)
            self.ground_truth_local.append(cluster_center)

    def __call__(self, feature_dict: Dict[int, Tensor],feature_local_dict: Dict[int, Tensor], y: Any = None) -> Tensor:
        loss_list = []
        loss_local_list = []
        for index, model in enumerate(self.extractors):
            
            gt = self.ground_truth[index]
            feature = feature_dict[index].unsqueeze(0)
            feat_loss = self.OT(gt,feature)

            loss_list.append(feat_loss)

        total_losses = [
            loss_list[i]
            for i in range(len(self.extractors))
        ]

        # 加权总损失
        total_loss = sum(total_losses)/len(total_losses)
        return total_loss

    def get_cluster_center(self, embedding_img,device):
        with suppress_output():
            cluster_ids_x, cluster_center = kmeans(
                X=embedding_img,
                num_clusters=self.cluster_number, # TODO
                distance='euclidean',
                device=device,
                iter_limit=100,  # 最大迭代次数，默认是 10
            )
            cluster_center = cluster_center.to(device)
        return cluster_center

    def OT(self, src_dis, tgt_dis):
        # print(src_dis.shape, tgt_dis.shape)
        src_dis_norm = F.normalize(src_dis, dim=1)
        tgt_dis_norm = F.normalize(tgt_dis, dim=1)
        sim = torch.einsum('md,nd->mn', src_dis_norm, tgt_dis_norm).contiguous()
        wdist = 1 - sim
        xx = torch.full((src_dis.shape[0],), 1.0 / src_dis.shape[0], dtype=sim.dtype, device=sim.device)
        yy = torch.full((tgt_dis.shape[0],), 1.0 / tgt_dis.shape[0], dtype=sim.dtype, device=sim.device)
        with torch.no_grad():
            KK = torch.exp(-wdist / 0.1)
            T = self.Sinkhorn(KK, xx, yy)
        if torch.isnan(T).any():
            return None
        sim_op = torch.sum(T * sim, dim=(0, 1))
        loss = torch.sum(sim_op)
        return loss

    def Sinkhorn(self, K, u, v):
        r = torch.ones_like(u)
        c = torch.ones_like(v)
        thresh = 1e-2
        for i in range(100):
            r0 = r
            r = u / (K @ c.unsqueeze(-1)).squeeze(-1)
            c = v / (K.t() @ r.unsqueeze(-1)).squeeze(-1)
            err = (r - r0).abs().mean()
            if err.item() < thresh:
                break
        T = torch.outer(r, c) * K
        return T'''
    
    UPSTREAM['mpc']["sources"]['surrogates/FeatureExtractors/ClipB16.py'] = r'''import torch
from transformers import CLIPVisionModel, CLIPProcessor, CLIPModel, AutoTokenizer
from .Base import BaseFeatureExtractor
from torchvision import transforms
import sys






class ClipB16FeatureExtractor(BaseFeatureExtractor):
    def __init__(self):
        super(ClipB16FeatureExtractor, self).__init__()
        self.model = CLIPModel.from_pretrained("openai/clip-vit-base-patch16")
        self.processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch16")
        self.tokenizer = AutoTokenizer.from_pretrained("openai/clip-vit-base-patch16")
        self.normalizer = transforms.Compose(
        [
            transforms.Resize((224, 224), interpolation=transforms.InterpolationMode.BICUBIC, antialias=True),
            transforms.Lambda(lambda img: torch.clamp(img, 0.0, 255.0) / 255.0),
            # transforms.CenterCrop(224),
            transforms.Normalize((0.48145466, 0.4578275, 0.40821073), (0.26862954, 0.26130258, 0.27577711)), # CLIP imgs mean and std.
        ]
    )

    def forward(self, x):
        # x = torch.clamp(x, min=0, max=1)
        inputs = dict(pixel_values=self.normalizer(x))
        image_features = self.model.get_image_features(**inputs)
        image_features = image_features / image_features.norm(dim=1, keepdim=True)
        return image_features

    def text_features_get(self, text):
        inputs = self.tokenizer(text, padding=True, return_tensors="pt").to(self.model.device)
        # print(inputs)
        text_features = self.model.get_text_features(**inputs)
        text_features = text_features / text_features.norm(dim=1, keepdim=True)
        return text_features

    def get_token_features(self, x):
        # x = torch.clamp(x, min=0, max=1)
        inputs = dict(pixel_values=self.normalizer(x))
        # image_features = self.model.get_image_features(**inputs)
        # image_features = image_features / image_features.norm(dim=1, keepdim=True)

        inputs["pixel_values"] = inputs["pixel_values"]
        outputs = self.model.vision_model(pixel_values=inputs['pixel_values'], output_hidden_states=True)
        # print(outputs.hidden_states[0].size())
        # sys.exit()
        features = outputs.last_hidden_state
        features / features.norm(dim=1, keepdim=True)
        mid_features = outputs.hidden_states
        return features, mid_features

    def global_local_features(self, x):
        # x = torch.clamp(x, min=0, max=1)
        inputs = dict(pixel_values=self.normalizer(x))
        # image_features = self.model.get_image_features(**inputs)
        # image_features = image_features / image_features.norm(dim=1, keepdim=True)

        inputs["pixel_values"] = inputs["pixel_values"]
        outputs = self.model.vision_model(pixel_values=inputs['pixel_values'])
        features = outputs.last_hidden_state
        # print(features.size())
        # sys.exit()
        global_feature = features[:, 0, :]
        global_feature = global_feature / global_feature.norm(dim=1, keepdim=True)
        local_feature = features[:, 1:, :]
        local_feature = local_feature / local_feature.norm(dim=1, keepdim=True)
        # features = self.model.get_image_embedding(inputs["pixel_values"])
        # global_feature = features[:, 0, :]
        # local_feature = features[:, 1:, :]
        return global_feature, local_feature


class ClipB16FeatureExtractorOT(BaseFeatureExtractor):
    def __init__(self):
        super(ClipB16FeatureExtractorOT, self).__init__()
        self.model = CLIPModel.from_pretrained("openai/clip-vit-base-patch16")
        self.processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch16")
        self.normalizer = transforms.Compose(
            [
                transforms.Resize(224, interpolation=transforms.InterpolationMode.BICUBIC, antialias=True),
                transforms.Lambda(lambda img: torch.clamp(img, 0.0, 255.0) / 255.0),
                transforms.CenterCrop(224),
                transforms.Normalize((0.48145466, 0.4578275, 0.40821073), (0.26862954, 0.26130258, 0.27577711)),
                # CLIP imgs mean and std.
            ]
        )

    def forward(self, x):
        x = torch.clamp(x, min=0, max=1)
        inputs = dict(pixel_values=self.normalizer(x))
        inputs["pixel_values"] = inputs["pixel_values"].to(self.device)
        features = self.model.get_image_embedding(inputs["pixel_values"])
        global_feature = features[:,0,:]
        local_feature = features[:,1:,:]
        return global_feature, local_feature'''
    
    UPSTREAM['mpc']["sources"]['surrogates/FeatureExtractors/ClipB32.py'] = r'''import sys

import torch
from transformers import CLIPVisionModel, CLIPProcessor, CLIPModel, AutoTokenizer
from .Base import BaseFeatureExtractor
from torchvision import transforms


class ClipB32FeatureExtractor(BaseFeatureExtractor):
    def __init__(self):
        super(ClipB32FeatureExtractor, self).__init__()
        self.model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
        self.processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
        self.tokenizer = AutoTokenizer.from_pretrained("openai/clip-vit-base-patch32")
        self.normalizer = transforms.Compose(
        [
            transforms.Resize((224, 224), interpolation=transforms.InterpolationMode.BICUBIC, antialias=True),
            transforms.Lambda(lambda img: torch.clamp(img, 0.0, 255.0) / 255.0),
            # transforms.CenterCrop(224),
            transforms.Normalize((0.48145466, 0.4578275, 0.40821073), (0.26862954, 0.26130258, 0.27577711)), # CLIP imgs mean and std.
        ]
    )

    def forward(self, x):
        # x = torch.clamp(x, min=0, max=1)
        inputs = dict(pixel_values=self.normalizer(x))
        image_features = self.model.get_image_features(**inputs)
        image_features = image_features / image_features.norm(dim=1, keepdim=True)
        return image_features

    def text_features_get(self, text):
        inputs = self.tokenizer(text, padding=True, return_tensors="pt").to(self.model.device)
        text_features = self.model.get_text_features(**inputs)
        text_features = text_features / text_features.norm(dim=1, keepdim=True)
        return text_features

    def get_token_features(self, x):
        # x = torch.clamp(x, min=0, max=1)
        inputs = dict(pixel_values=self.normalizer(x))
        # image_features = self.model.get_image_features(**inputs)
        # image_features = image_features / image_features.norm(dim=1, keepdim=True)

        inputs["pixel_values"] = inputs["pixel_values"]
        outputs = self.model.vision_model(pixel_values=inputs['pixel_values'], output_hidden_states=True)
        # print(outputs.hidden_states[0].size())
        features = outputs.last_hidden_state
        features / features.norm(dim=1, keepdim=True)
        mid_features = outputs.hidden_states
        return features, mid_features

    def global_local_features(self, x):
        # x = torch.clamp(x, min=0, max=1)
        inputs = dict(pixel_values=self.normalizer(x))
        # image_features = self.model.get_image_features(**inputs)
        # image_features = image_features / image_features.norm(dim=1, keepdim=True)

        inputs["pixel_values"] = inputs["pixel_values"]
        outputs = self.model.vision_model(pixel_values=inputs['pixel_values'])
        features = outputs.last_hidden_state
        # print(features.size())
        # sys.exit()
        global_feature = features[:, 0, :]
        global_feature = global_feature / global_feature.norm(dim=1, keepdim=True)
        local_feature = features[:, 1:, :]
        local_feature = local_feature / local_feature.norm(dim=1, keepdim=True)
        # features = self.model.get_image_embedding(inputs["pixel_values"])
        # global_feature = features[:, 0, :]
        # local_feature = features[:, 1:, :]
        return global_feature, local_feature
class ClipB32FeatureExtractorOT(BaseFeatureExtractor):
    def __init__(self):
        super(ClipB32FeatureExtractorOT, self).__init__()
        self.model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
        self.processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
        self.normalizer = transforms.Compose(
            [
                transforms.Resize(224, interpolation=transforms.InterpolationMode.BICUBIC, antialias=True),
                transforms.Lambda(lambda img: torch.clamp(img, 0.0, 255.0) / 255.0),
                transforms.CenterCrop(224),
                transforms.Normalize((0.48145466, 0.4578275, 0.40821073), (0.26862954, 0.26130258, 0.27577711)),
                # CLIP imgs mean and std.
            ]
        )

    def forward(self, x):
        x = torch.clamp(x, min=0, max=1)
        inputs = dict(pixel_values=self.normalizer(x))
        inputs["pixel_values"] = inputs["pixel_values"].to(self.device)
        features = self.model.get_image_embedding(inputs["pixel_values"])
        global_feature = features[:,0,:]
        local_feature = features[:,1:,:]
        return global_feature, local_feature










'''
    
    UPSTREAM['mpc']["sources"]['surrogates/FeatureExtractors/ClipLaion.py'] = r'''import sys

import torch
from transformers import CLIPVisionModel, CLIPProcessor, CLIPModel, AutoTokenizer
# from .Base import BaseFeatureExtractor
from surrogates.FeatureExtractors.Base import BaseFeatureExtractor
from torchvision import transforms



class ClipLaionFeatureExtractor(BaseFeatureExtractor):
    def __init__(self):
        super(ClipLaionFeatureExtractor, self).__init__()
        self.model = CLIPModel.from_pretrained("laion/CLIP-ViT-G-14-laion2B-s12B-b42K")
        # self.processor = CLIPProcessor.from_pretrained("openai/clip-vit-large-patch14-336")
        self.tokenizer = AutoTokenizer.from_pretrained("laion/CLIP-ViT-G-14-laion2B-s12B-b42K")
        self.normalizer = transforms.Compose(
        [
            transforms.Resize((224, 224), interpolation=transforms.InterpolationMode.BICUBIC, antialias=True),
            transforms.Lambda(lambda img: torch.clamp(img, 0.0, 255.0) / 255.0),
            # transforms.CenterCrop(224),
            transforms.Normalize((0.48145466, 0.4578275, 0.40821073), (0.26862954, 0.26130258, 0.27577711)), # CLIP imgs mean and std.
        ]
    )

    def forward(self, x):
        # x = torch.clamp(x, min=0, max=1)
        inputs = dict(pixel_values=self.normalizer(x))
        image_features = self.model.get_image_features(**inputs) #torch.Size([1, 1024])
        image_features = image_features / image_features.norm(dim=1, keepdim=True)
        return image_features

    def text_features_get(self, text):
        inputs = self.tokenizer(text, padding=True, return_tensors="pt").to(self.model.device)
        text_features = self.model.get_text_features(**inputs)
        text_features = text_features / text_features.norm(dim=1, keepdim=True)
        return text_features

    def get_token_features(self, x):
        # x = torch.clamp(x, min=0, max=1)
        inputs = dict(pixel_values=self.normalizer(x))
        # image_features = self.model.get_image_features(**inputs)
        # image_features = image_features / image_features.norm(dim=1, keepdim=True)

        inputs["pixel_values"] = inputs["pixel_values"]
        outputs = self.model.vision_model(pixel_values=inputs['pixel_values'], output_hidden_states=True)
        # print(outputs.hidden_states[0].size())
        features = outputs.last_hidden_state
        features / features.norm(dim=1, keepdim=True)
        mid_features = outputs.hidden_states
        return features, mid_features


    def global_local_features(self, x):
        # x = torch.clamp(x, min=0, max=1)
        inputs = dict(pixel_values=self.normalizer(x))
        # image_features = self.model.get_image_features(**inputs)
        # image_features = image_features / image_features.norm(dim=1, keepdim=True)

        inputs["pixel_values"] = inputs["pixel_values"]
        outputs = self.model.vision_model(pixel_values=inputs['pixel_values'], output_attentions=True)
        # print(outputs)
        # print(len(outputs.attentions))
        # for attention in outputs.attentions:
        #     print(attention.size())
        features = outputs.last_hidden_state
        # print(features.size())
        # sys.exit()
        global_feature = features[:, 0, :]
        global_feature = global_feature / global_feature.norm(dim=1, keepdim=True)
        local_feature = features[:, 1:, :]
        local_feature = local_feature / local_feature.norm(dim=1, keepdim=True)
        # features = self.model.get_image_embedding(inputs["pixel_values"])
        # global_feature = features[:, 0, :]
        # local_feature = features[:, 1:, :]
        return global_feature, local_feature
'''
    
    UPSTREAM['mpc']["sources"]['surrogates/FeatureExtractors/InternVL3_1B.py'] = r'''import sys

import torch
from transformers import AutoProcessor, AutoModel, AutoModelForImageTextToText, AutoTokenizer, InternVLModel, InternVLProcessor
from transformers.image_utils import load_images

from torchvision import transforms
# from .Base import BaseFeatureExtractor
from surrogates.FeatureExtractors.Base import BaseFeatureExtractor

import torchvision.transforms as T
from torchvision.transforms.functional import InterpolationMode
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

import torch.nn.functional as F

def dynamic_preprocess_tensor(images, min_num=1, max_num=12, image_size=448, use_thumbnail=False):
    """
    images: torch.Tensor of shape [B, C, H, W]
    return: list of torch.Tensor, each shape [num_tiles, C, image_size, image_size]
    """
    B, C, H, W = images.shape
    output_list = []

    # for b in range(B):
    img = images[0]  # [C,H,W]
    aspect_ratio = W / H


    target_ratios = sorted(
        [(i, j) for n in range(min_num, max_num+1) for i in range(1,n+1) for j in range(1,n+1)
         if min_num <= i*j <= max_num], key=lambda x: x[0]*x[1])


    target_aspect_ratio = min(target_ratios, key=lambda r: abs((r[0]/r[1]) - aspect_ratio))
    target_width = image_size * target_aspect_ratio[0]
    target_height = image_size * target_aspect_ratio[1]
    blocks = target_aspect_ratio[0] * target_aspect_ratio[1]


    img_resized = F.interpolate(img.unsqueeze(0), size=(int(target_height), int(target_width)), mode='bilinear', align_corners=False)[0]


    tiles = []
    n_w = int(target_width // image_size)
    n_h = int(target_height // image_size)
    for i in range(n_h):
        for j in range(n_w):
            tile = img_resized[:, i*image_size:(i+1)*image_size, j*image_size:(j+1)*image_size]
            tiles.append(tile)
    tiles = torch.stack(tiles, dim=0)  # [num_tiles, C, image_size, image_size]


    if use_thumbnail and tiles.shape[0] != 1:
        thumb = F.interpolate(img.unsqueeze(0), size=(image_size, image_size), mode='bilinear', align_corners=False)[0]
        tiles = torch.cat([tiles, thumb.unsqueeze(0)], dim=0)

    # output_list.append(tiles)

    return tiles

class InternVL3_1B_FeatureExtractor(BaseFeatureExtractor):
    def __init__(self):
        super(InternVL3_1B_FeatureExtractor, self).__init__()
        self.normalizer = transforms.Compose(
        [
            transforms.Resize((448, 448), interpolation=transforms.InterpolationMode.BICUBIC, antialias=True),
            transforms.Lambda(lambda img: torch.clamp(img, 0.0, 255.0) / 255.0),
            # transforms.CenterCrop(224),
            transforms.Normalize((0.48145466, 0.4578275, 0.40821073), (0.26862954, 0.26130258, 0.27577711)), # CLIP imgs mean and std.
        ]
    )
        # path = "OpenGVLab/InternVL3-1B"
        # self.processor = AutoProcessor.from_pretrained(path)
        # self.tokenizer = AutoTokenizer.from_pretrained(path)
        # self.model = AutoModel.from_pretrained(
        #     path,
        #     torch_dtype=torch.bfloat16,
        #     low_cpu_mem_usage=True,
        #     use_flash_attn=True,
        #     trust_remote_code=True)
        self.processor = AutoProcessor.from_pretrained("OpenGVLab/InternVL3-1B-hf")
        self.model = AutoModelForImageTextToText.from_pretrained("OpenGVLab/InternVL3-1B-hf", torch_dtype=torch.bfloat16)
        self.tokenizer = AutoTokenizer.from_pretrained("OpenGVLab/InternVL3-1B-hf")



    def get_input_pixel_values(self, x):
        re_x = transforms.Resize((224, 224), interpolation=transforms.InterpolationMode.BICUBIC, antialias=True)(x)
        pixel_values = self.normalizer(dynamic_preprocess_tensor(re_x, use_thumbnail=True))
        return pixel_values

    # def prepare_inputs(self, x):

    def generate_description(self, image):
        if image.max() == 1.0:
            image = image * 255
        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image"},
                    {"type": "text", "text": "Describe this image in one concise sentence, no longer than 20 words."},
                ],
            }
        ]
        prompt = self.processor.apply_chat_template(messages, add_generation_prompt=True)
        inputs = self.processor(text=prompt, images=[image], return_tensors="pt")

        inputs = inputs.to(self.model.device, dtype=torch.bfloat16)
        generate_ids = self.model.generate(**inputs, max_new_tokens=50)
        decoded_output = self.processor.decode(generate_ids[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        # print('generated description:', decoded_output)
        return decoded_output


    def forward(self, x):
        pixel_values = self.get_input_pixel_values(x)
        pixel_values = pixel_values.to(self.model.device, dtype=torch.bfloat16)
        inputs = dict(pixel_values=pixel_values)
        outputs = self.model.get_image_features(**inputs)
        # print(outputs.size())

        image_features = outputs / outputs.norm(dim=1, keepdim=True)
        # print(image_features.size())
        return image_features

    def global_local_features(self, x):
        pixel_values = self.get_input_pixel_values(x)
        pixel_values = pixel_values.to(self.model.device, dtype=torch.bfloat16)
        # inputs = dict(pixel_values=pixel_values)
        vision_features = self.model.vision_tower(pixel_values=pixel_values).last_hidden_state
        # print(vision_features.size())
        # vision_features = vision_features[:, 1:, :]
        # print(vision_features.size())
        # sys.exit()
        global_feature = vision_features[:, 0, :]
        global_feature = global_feature / global_feature.norm(dim=1, keepdim=True)
        local_feature = vision_features[:, 1:, :]
        local_feature = local_feature / local_feature.norm(dim=1, keepdim=True)
        # features = self.model.get_image_embedding(inputs["pixel_values"])
        # global_feature = features[:, 0, :]
        # local_feature = features[:, 1:, :]
        return global_feature, local_feature



    def get_text_embeddings(self, text):

        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": text},
                ],
            }
        ]

        prompt = self.processor.apply_chat_template(messages, add_generation_prompt=True)
        inputs = self.processor(text=prompt, return_tensors="pt")
        inputs = inputs.to(self.model.device, dtype=torch.bfloat16)
        # print(inputs)
        inputs_embeds = self.model.get_input_embeddings()(inputs.input_ids)  # torch.Size([1, 940, 896])
        # print(inputs_embeds)
        print(inputs_embeds.size())

        return inputs_embeds

'''.replace('\n', '\r\n')
    
    UPSTREAM['mpc']["sources"]['surrogates/FeatureExtractors/DINOv2_Base.py'] = r'''import sys

import torch

from transformers import AutoImageProcessor, AutoModel, AutoTokenizer, Dinov2Model
# from .Base import BaseFeatureExtractor
from surrogates.FeatureExtractors.Base import BaseFeatureExtractor
from torchvision import transforms
import torch.nn.functional as F

class DINOv2FeatureExtractor(BaseFeatureExtractor):
    def __init__(self):
        super(DINOv2FeatureExtractor, self).__init__()
        self.model = AutoModel.from_pretrained('facebook/dinov2-base')  # facebook/dinov2-giant
        self.processor = AutoImageProcessor.from_pretrained('facebook/dinov2-base')
        # self.tokenizer = AutoTokenizer.from_pretrained('facebook/dinov2-base')
        self.normalizer = transforms.Compose(
        [
            transforms.Resize((224, 224), interpolation=transforms.InterpolationMode.BICUBIC, antialias=True),
            transforms.Lambda(lambda img: torch.clamp(img, 0.0, 255.0) / 255.0),
            # transforms.CenterCrop(224),
            transforms.Normalize((0.48145466, 0.4578275, 0.40821073), (0.26862954, 0.26130258, 0.27577711)), # CLIP imgs mean and std.
        ]
    )

    def forward(self, x):
        # x = torch.clamp(x, min=0, max=1)
        pixel_values = self.normalizer(x)
        outputs = self.model(pixel_values=pixel_values)
        last_hidden_states = outputs.last_hidden_state
        image_features = last_hidden_states[:, 0, :]  # torch.Size([1, 768])
        image_features = image_features / image_features.norm(dim=1, keepdim=True)
        return image_features


    def global_local_features(self, x):
        pixel_values = self.normalizer(x)
        outputs = self.model(pixel_values=pixel_values)
        features = outputs.last_hidden_state

        global_feature = features[:, 0, :]
        global_feature = global_feature / global_feature.norm(dim=1, keepdim=True)
        local_feature = features[:, 1:, :]
        local_feature = local_feature / local_feature.norm(dim=1, keepdim=True)
        # print(features.size())
        # print(global_feature.size())
        # print(local_feature.size())
        # sys.exit()
        return global_feature, local_feature

'''.replace('\n', '\r\n')
    
    UPSTREAM['kmeans'] = {'repo': 'subhadarship/kmeans_pytorch', 'commit': 'f7f36bd1cb4e3a761d73d584866d0a9c6b4d2805'}
    
    UPSTREAM['kmeans']["sources"] = {}
    
    UPSTREAM['kmeans']["sources"]['kmeans_pytorch/__init__.py'] = r'''from functools import partial

import numpy as np
import torch
from tqdm import tqdm

from .soft_dtw_cuda import SoftDTW


def initialize(X, num_clusters, seed):
    """
    initialize cluster centers
    :param X: (torch.tensor) matrix
    :param num_clusters: (int) number of clusters
    :param seed: (int) seed for kmeans
    :return: (np.array) initial state
    """
    num_samples = len(X)
    if seed == None:
        indices = np.random.choice(num_samples, num_clusters, replace=False)
    else:
        np.random.seed(seed) ; indices = np.random.choice(num_samples, num_clusters, replace=False)
    initial_state = X[indices]
    return initial_state


def kmeans(
        X,
        num_clusters,
        distance='euclidean',
        cluster_centers=[],
        tol=1e-4,
        tqdm_flag=True,
        iter_limit=0,
        device=torch.device('cpu'),
        gamma_for_soft_dtw=0.001,
        seed=None,
):
    """
    perform kmeans
    :param X: (torch.tensor) matrix
    :param num_clusters: (int) number of clusters
    :param distance: (str) distance [options: 'euclidean', 'cosine'] [default: 'euclidean']
    :param seed: (int) seed for kmeans
    :param tol: (float) threshold [default: 0.0001]
    :param device: (torch.device) device [default: cpu]
    :param tqdm_flag: Allows to turn logs on and off
    :param iter_limit: hard limit for max number of iterations
    :param gamma_for_soft_dtw: approaches to (hard) DTW as gamma -> 0
    :return: (torch.tensor, torch.tensor) cluster ids, cluster centers
    """
    if tqdm_flag:
        print(f'running k-means on {device}..')

    if distance == 'euclidean':
        pairwise_distance_function = partial(pairwise_distance, device=device, tqdm_flag=tqdm_flag)
    elif distance == 'cosine':
        pairwise_distance_function = partial(pairwise_cosine, device=device)
    elif distance == 'soft_dtw':
        sdtw = SoftDTW(use_cuda=device.type == 'cuda', gamma=gamma_for_soft_dtw)
        pairwise_distance_function = partial(pairwise_soft_dtw, sdtw=sdtw, device=device)
    else:
        raise NotImplementedError

    # convert to float
    X = X.float()

    # transfer to device
    X = X.to(device)

    # initialize
    if type(cluster_centers) == list:  # ToDo: make this less annoyingly weird
        initial_state = initialize(X, num_clusters, seed=seed)
    else:
        if tqdm_flag:
            print('resuming')
        # find data point closest to the initial cluster center
        initial_state = cluster_centers
        dis = pairwise_distance_function(X, initial_state)
        choice_points = torch.argmin(dis, dim=0)
        initial_state = X[choice_points]
        initial_state = initial_state.to(device)

    iteration = 0
    if tqdm_flag:
        tqdm_meter = tqdm(desc='[running kmeans]')
    while True:

        dis = pairwise_distance_function(X, initial_state)

        choice_cluster = torch.argmin(dis, dim=1)

        initial_state_pre = initial_state.clone()

        for index in range(num_clusters):
            selected = torch.nonzero(choice_cluster == index).squeeze().to(device)

            selected = torch.index_select(X, 0, selected)

            # https://github.com/subhadarship/kmeans_pytorch/issues/16
            if selected.shape[0] == 0:
                selected = X[torch.randint(len(X), (1,))]

            initial_state[index] = selected.mean(dim=0)

        center_shift = torch.sum(
            torch.sqrt(
                torch.sum((initial_state - initial_state_pre) ** 2, dim=1)
            ))

        # increment iteration
        iteration = iteration + 1

        # update tqdm meter
        if tqdm_flag:
            tqdm_meter.set_postfix(
                iteration=f'{iteration}',
                center_shift=f'{center_shift ** 2:0.6f}',
                tol=f'{tol:0.6f}'
            )
            tqdm_meter.update()
        if center_shift ** 2 < tol:
            break
        if iter_limit != 0 and iteration >= iter_limit:
            break

    return choice_cluster.cpu(), initial_state.cpu()


def kmeans_predict(
        X,
        cluster_centers,
        distance='euclidean',
        device=torch.device('cpu'),
        gamma_for_soft_dtw=0.001,
        tqdm_flag=True
):
    """
    predict using cluster centers
    :param X: (torch.tensor) matrix
    :param cluster_centers: (torch.tensor) cluster centers
    :param distance: (str) distance [options: 'euclidean', 'cosine'] [default: 'euclidean']
    :param device: (torch.device) device [default: 'cpu']
    :param gamma_for_soft_dtw: approaches to (hard) DTW as gamma -> 0
    :return: (torch.tensor) cluster ids
    """
    if tqdm_flag:
        print(f'predicting on {device}..')

    if distance == 'euclidean':
        pairwise_distance_function = partial(pairwise_distance, device=device, tqdm_flag=tqdm_flag)
    elif distance == 'cosine':
        pairwise_distance_function = partial(pairwise_cosine, device=device)
    elif distance == 'soft_dtw':
        sdtw = SoftDTW(use_cuda=device.type == 'cuda', gamma=gamma_for_soft_dtw)
        pairwise_distance_function = partial(pairwise_soft_dtw, sdtw=sdtw, device=device)
    else:
        raise NotImplementedError

    # convert to float
    X = X.float()

    # transfer to device
    X = X.to(device)

    dis = pairwise_distance_function(X, cluster_centers)
    choice_cluster = torch.argmin(dis, dim=1)

    return choice_cluster.cpu()


def pairwise_distance(data1, data2, device=torch.device('cpu'), tqdm_flag=True):
    if tqdm_flag:
        print(f'device is :{device}')
    
    # transfer to device
    data1, data2 = data1.to(device), data2.to(device)

    # N*1*M
    A = data1.unsqueeze(dim=1)

    # 1*N*M
    B = data2.unsqueeze(dim=0)

    dis = (A - B) ** 2.0
    # return N*N matrix for pairwise distance
    dis = dis.sum(dim=-1).squeeze()
    return dis


def pairwise_cosine(data1, data2, device=torch.device('cpu')):
    # transfer to device
    data1, data2 = data1.to(device), data2.to(device)

    # N*1*M
    A = data1.unsqueeze(dim=1)

    # 1*N*M
    B = data2.unsqueeze(dim=0)

    # normalize the points  | [0.3, 0.4] -> [0.3/sqrt(0.09 + 0.16), 0.4/sqrt(0.09 + 0.16)] = [0.3/0.5, 0.4/0.5]
    A_normalized = A / A.norm(dim=-1, keepdim=True)
    B_normalized = B / B.norm(dim=-1, keepdim=True)

    cosine = A_normalized * B_normalized

    # return N*N matrix for pairwise distance
    cosine_dis = 1 - cosine.sum(dim=-1).squeeze()
    return cosine_dis


def pairwise_soft_dtw(data1, data2, sdtw=None, device=torch.device('cpu')):
    if sdtw is None:
        raise ValueError('sdtw is None - initialize it with SoftDTW')

    # transfer to device
    data1, data2 = data1.to(device), data2.to(device)

    # (batch_size, seq_len, feature_dim=1)
    A = data1.unsqueeze(dim=2)

    # (cluster_size, seq_len, feature_dim=1)
    B = data2.unsqueeze(dim=2)

    distances = []
    for b in B:
        # (1, seq_len, 1)
        b = b.unsqueeze(dim=0)
        A, b = torch.broadcast_tensors(A, b)
        # (batch_size, 1)
        sdtw_distance = sdtw(b, A).view(-1, 1)
        distances.append(sdtw_distance)

    # (batch_size, cluster_size)
    dis = torch.cat(distances, dim=1)
    return dis
'''
    
    UPSTREAM['kmeans']["sources"]['LICENSE'] = r'''MIT License

Copyright (c) 2020 subhadarshi

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.
'''
    PROVENANCE = {'foa': {'repo': 'jiaxiaojunQAQ/FOA-Attack', 'commit': '6aa9ad463a1e812b7f77ff2e5c4b1712e7c78c4b', 'sha256': {'generate_adversarial_samples_foa_attack.py': '80ffe84b27454922e65c9cff243cc16c703f17fe4f958b2cc2a97ada8b4313fb', 'surrogates/FeatureExtractors/Base.py': 'c5984a4a9622799c1e57580f51c2f59bea860ff04bc42b74bca80a3c8b787f84', 'surrogates/FeatureExtractors/ClipB16.py': '3a0f05ea4f3c90b7a1d32a9abdbd05fb3ca04a757f4196bd0e927d1527242f3d', 'surrogates/FeatureExtractors/ClipB32.py': 'a262ddaaa7483c16ad589599d8da68edd5a16a54a0662060d08994d3e732c6bd', 'surrogates/FeatureExtractors/ClipLaion.py': 'eee6aeaf7a9adcbfc51e17c3619a6b4872116b3ffd95e883247f7a13a71b4314', 'config/ensemble_3models.yaml': '7287a73da7a2fea4eac39b97ecd9fce2ef367d8d3aa0f0cae9da160db4e4f8bb'}}, 'mpc': {'repo': 'LiYuanBoJNU/MPCAttack', 'commit': '7b195a5bcbaeaf1e8534d101199810a2782caad3', 'license': 'MIT License\n\nCopyright (c) 2026 yb_li\n\nPermission is hereby granted, free of charge, to any person obtaining a copy\nof this software and associated documentation files (the "Software"), to deal\nin the Software without restriction, including without limitation the rights\nto use, copy, modify, merge, publish, distribute, sublicense, and/or sell\ncopies of the Software, and to permit persons to whom the Software is\nfurnished to do so, subject to the following conditions:\n\nThe above copyright notice and this permission notice shall be included in all\ncopies or substantial portions of the Software.\n\nTHE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR\nIMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,\nFITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE\nAUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER\nLIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,\nOUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE\nSOFTWARE.\n', 'sha256': {'generate_adversarial_examples_MPCAttack.py': '93eca3842823d695ee6c305acc230366799766f584a849d427aa52f6deedf5fb', 'utils.py': '04af89073f23dbfbb8182069632c308f1d37531685c09bd5f766055485472681', 'surrogates/FeatureExtractors/Base.py': '044599954723aa6db3b3e9d24430d6241fb151c8b44a0f4711f6d88836e4c661', 'surrogates/FeatureExtractors/ClipB16.py': 'fff34f53b6d574baf08c85a16ed8a4332e2e0ebbd4ba7b03ceb18dbe553e71e3', 'surrogates/FeatureExtractors/ClipB32.py': '6d27a385383ae8ab1b792b5d41216e914bd3286c465c2f0f6616483061216132', 'surrogates/FeatureExtractors/ClipLaion.py': 'c1189888e08edb24ed8ed5c4e740f12018b1c2b969dc8757eca3f97dd3778ea5', 'surrogates/FeatureExtractors/InternVL3_1B.py': '1e71c0c4ff0b96206db5b7316cf57d5fc053d80ba6a527f2bc6505842626b684', 'surrogates/FeatureExtractors/DINOv2_Base.py': 'bc2d442c1b81001bd2658622ba5f2e3af04f656dafe98c02eecb88a1aa0e15e5'}}, 'kmeans': {'repo': 'subhadarship/kmeans_pytorch', 'commit': 'f7f36bd1cb4e3a761d73d584866d0a9c6b4d2805', 'sha256': {'kmeans_pytorch/__init__.py': '5d9cf15c5a9c82a4c3ecfcdccca3a82d4f1c8eaa21d31f483d4272128d6cbb19', 'LICENSE': '63961947e658a5d6f891fef3b15afccb86671d76dfd7d16bb1909b95204caabc'}}}

In [ ]:
# Cell 3 — Compile pinned author definitions into isolated namespaces, without imports/entrypoints.
if globals().get('_NOTEBOOK_STAGE') == 'attack':
    def source_definitions(source, names):
        tree = ast.parse(source)
        nodes = [node for node in tree.body if getattr(node, 'name', None) in names]
        if {n.name for n in nodes} != set(names):
            raise RuntimeError(f'Missing author definitions: {names}')
        return ast.fix_missing_locations(ast.Module(body=nodes, type_ignores=[]))
    
    def compile_author(repo, path, names, namespace=None):
        namespace = dict(globals()) if namespace is None else namespace
        exec(compile(source_definitions(UPSTREAM[repo]['sources'][path], names), f'<{repo}:{path}>', 'exec'), namespace)
        return namespace
    
    # Euclidean FOA calls execute unmodified initialize/kmeans/pairwise_distance functions.
    # SoftDTW is not part of FOA and is deliberately not imported.
    _km = compile_author('kmeans', 'kmeans_pytorch/__init__.py', ['initialize', 'kmeans', 'pairwise_distance', 'pairwise_cosine'])
    _author_kmeans = _km['kmeans']
    
    def kmeans(*args, **kwargs):
        # The upstream flag only controls prints/tqdm; clustering and RNG are unchanged.
        kwargs['tqdm_flag'] = False
        return _author_kmeans(*args, **kwargs)
    
    class LockedFactory:
        def __init__(self, cls, share=False):
            self.cls, self.share = cls, share
    
        def from_pretrained(self, model_id, **kwargs):
            if not RUN_ATTACK:
                raise RuntimeError('No model downloads during preflight.')
            revision = MODEL_REVISIONS[model_id]
            kwargs['revision'] = revision
            key = (model_id, str(kwargs.get('torch_dtype', 'default')))
            if self.share and key in MODEL_STORE:
                return MODEL_STORE[key]
            model = self.cls.from_pretrained(model_id, **kwargs)
            if self.share:
                MODEL_STORE[key] = model
            info = {'model_id': model_id, 'revision': revision,
                'resolved_config_revision': getattr(getattr(model, 'config', None), '_commit_hash', None),
                'dtype': str(getattr(model, 'dtype', None)), 'class': type(model).__name__}
            MODEL_METADATA.setdefault(model_id, {'model_id': model_id, 'revision': revision, 'artifacts': {}})['artifacts'][self.cls.__name__] = info
            return model
    
    CLIPModel = LockedFactory(HFCLIPModel, share=True)
    
    def resolve_revisions(model_ids, existing=None):
        if existing is not None:
            if set(model_ids) != set(existing) or any(not re.fullmatch(r'[0-9a-f]{40}', x) for x in existing.values()):
                raise ValueError('Resume model revisions differ from this configuration.')
            return dict(existing)
        api = HfApi()
        return {name: api.model_info(name, revision=ATTENTION_REVISION if name == ATTENTION_MODEL_ID else MODEL_REVISION).sha for name in model_ids}
    
    def install_model_factories(namespace):
        namespace.update(CLIPModel=CLIPModel,
            CLIPProcessor=LockedFactory(CLIPProcessor), AutoTokenizer=LockedFactory(AutoTokenizer),
            AutoProcessor=LockedFactory(AutoProcessor), AutoModel=LockedFactory(AutoModel),
            AutoModelForImageTextToText=LockedFactory(AutoModelForImageTextToText))
        return namespace
    
    def upstream_fgsm(namespace=None):
        ns = dict(globals()) if namespace is None else namespace
        ns.update(MainConfig=object)
        return compile_author('foa', 'generate_adversarial_samples_foa_attack.py', ['fgsm_attack'], ns)['fgsm_attack']
    
    def upstream_mpc_score():
        return compile_author('mpc', 'utils.py', ['info_nce_loss'])['info_nce_loss']
    
    AUTHOR_MPC_SCORE = upstream_mpc_score()
    
    def verify_embedded_sources():
        for repo, meta in PROVENANCE.items():
            for path, expected in meta['sha256'].items():
                actual = hashlib.sha256(UPSTREAM[repo]['sources'][path].encode()).hexdigest()
                if actual != expected:
                    raise RuntimeError(f'Embedded reference modified: {repo}/{path}')
    
    verify_embedded_sources()

In [ ]:
# FOA: exact author definitions. Edit experimental behavior in the hybrid cells.
if globals().get('_NOTEBOOK_STAGE') == 'attack':
    @contextmanager
    def suppress_output():
        with open(os.devnull, 'w') as fnull:
            old_stdout = sys.stdout
            old_stderr = sys.stderr
            sys.stdout = fnull
            sys.stderr = fnull
            try:
                yield
            finally:
                sys.stdout = old_stdout
                sys.stderr = old_stderr
    
    class BaseFeatureExtractor(nn.Module):
        def __init__(self):
            super(BaseFeatureExtractor, self).__init__()
            pass
    
        @abstractmethod
        def forward(self, x: Tensor) -> Tensor:
            pass
    
    class EnsembleFeatureExtractor_ot(BaseFeatureExtractor):
        def __init__(self, extractors: List[BaseFeatureExtractor],cluster_number=5):
            super(EnsembleFeatureExtractor_ot, self).__init__()
            self.extractors = nn.ModuleList(extractors)
            self.cluster_number = cluster_number
    
        def forward(self, x: Tensor) -> Tensor:
            # features = []
            # for model in self.extractors:
            #     features.append(model(x).squeeze())
            # features = torch.cat(features, dim=0)
            features = {}  # 不拼接，改为字典存储
            features_local = {}
            for i, model in enumerate(self.extractors):
                # features[i] = model(x).squeeze()
                x_tensor, x_embedding = model.global_local_features(x.to(x.device))
                features[i] = x_tensor.squeeze()
                cluster_center = self.get_cluster_center(x_embedding[0],x.device).unsqueeze(0)
                features_local[i]=cluster_center
    
            return features,features_local
    
        def get_cluster_center(self, embedding_img,device):
            # self.setup_seed(20)
            # for i in range(10):
            # np.random.seed(20)
            with suppress_output():
                cluster_ids_x, cluster_center = kmeans(
                    X=embedding_img,
                    num_clusters=self.cluster_number,
                    distance='euclidean',
                    device=device
                )
                cluster_center = cluster_center.to(device)
            # print(cluster_ids_x)
            return cluster_center
    
    class EnsembleFeatureLoss_OT_foa_attack(nn.Module):
        def __init__(self, extractors: List[BaseFeatureExtractor],cluster_number=5):
            super(EnsembleFeatureLoss_OT_foa_attack, self).__init__()
            self.extractors = nn.ModuleList(extractors)
            self.ground_truth = []
            self.ground_truth_local = []
            self.previous_loss_list=[]
            self.previous_loss_local_list = []
            self.cluster_number = cluster_number
    
    
        @torch.no_grad()
        def set_ground_truth(self, x: Tensor):
            self.ground_truth.clear()
            self.ground_truth_local.clear()
            for model in self.extractors:
                x_tensor, x_embedding = model.global_local_features(x.to(x.device))
                x_embedding = x_embedding.squeeze(0)
                cluster_center = self.get_cluster_center(x_embedding,x.device).unsqueeze(0)
                self.ground_truth.append(x_tensor)
                self.ground_truth_local.append(cluster_center)
    
        def __call__(self, feature_dict: Dict[int, Tensor],feature_local_dict: Dict[int, Tensor], y: Any = None) -> Tensor:
            loss_list = []
            loss_local_list = []
            for index, model in enumerate(self.extractors):
                gt_local = self.ground_truth_local[index].squeeze(0)
                gt = self.ground_truth[index]
                feature = feature_dict[index].unsqueeze(0)
                feature_local = feature_local_dict[index].squeeze(0)
                # print("gt_local",gt_local.shape)
                # print("feature_local", feature_local.shape)
                local_loss = self.OT(gt_local, feature_local)
                
                # feat_loss = torch.mean(torch.sum(feature * gt, dim=1))
                feat_loss = self.OT(gt,feature)
    
                loss_list.append(feat_loss)
                loss_local_list.append(local_loss)
    
            total_losses = [
                loss_list[i] + 0.2 * loss_local_list[i]
                for i in range(len(self.extractors))
            ]
            # 初始化 previous_loss_list（首次）
            if len(self.previous_loss_list) == 0:
                self.previous_loss_list = [l.detach() for l in total_losses]
    
            weights = []
            for i in range(len(self.extractors)):
                ratio = total_losses[i].item() / (self.previous_loss_list[i].item() + 1e-8)
                weights.append(ratio)
                # 归一化 softmax 计算动态权重
            
            T = 1.0
            K = len(weights)
            weights_np = np.array(weights)
            weights_softmax = np.exp(weights_np / T)
            weights_softmax /= np.sum(weights_softmax)
            weights_softmax *= K  # 可选：缩放为 K
    
    
            # 初始化 previous_loss_list（首次）
            for i in range(len(self.extractors)):
                self.previous_loss_list[i] = total_losses[i].detach()
    
            # 加权总损失
            total_loss = sum(
                weights_softmax[i] * total_losses[i]
                for i in range(len(self.extractors))
            )
            return total_loss
    
        def get_cluster_center(self, embedding_img,device):
            # self.setup_seed(20)
            # for i in range(10):
            # np.random.seed(20)
            with suppress_output():
                cluster_ids_x, cluster_center = kmeans(
                    X=embedding_img,
                    num_clusters=self.cluster_number, # TODO
                    distance='euclidean',
                    device=device,
                    iter_limit=100,  # 最大迭代次数，默认是 10
                )
                cluster_center = cluster_center.to(device)
            # print(cluster_ids_x)
            return cluster_center
    
        def OT(self, src_dis, tgt_dis):
            # print(src_dis.shape, tgt_dis.shape)
            src_dis_norm = F.normalize(src_dis, dim=1)
            tgt_dis_norm = F.normalize(tgt_dis, dim=1)
            sim = torch.einsum('md,nd->mn', src_dis_norm, tgt_dis_norm).contiguous()
            wdist = 1 - sim
            xx = torch.full((src_dis.shape[0],), 1.0 / src_dis.shape[0], dtype=sim.dtype, device=sim.device)
            yy = torch.full((tgt_dis.shape[0],), 1.0 / tgt_dis.shape[0], dtype=sim.dtype, device=sim.device)
            with torch.no_grad():
                KK = torch.exp(-wdist / 0.1)
                T = self.Sinkhorn(KK, xx, yy)
            if torch.isnan(T).any():
                return None
            sim_op = torch.sum(T * sim, dim=(0, 1))
            loss = torch.sum(sim_op)
            return loss
    
        def Sinkhorn(self, K, u, v):
            r = torch.ones_like(u)
            c = torch.ones_like(v)
            thresh = 1e-2
            for i in range(100):
                r0 = r
                r = u / (K @ c.unsqueeze(-1)).squeeze(-1)
                c = v / (K.t() @ r.unsqueeze(-1)).squeeze(-1)
                err = (r - r0).abs().mean()
                if err.item() < thresh:
                    break
            T = torch.outer(r, c) * K
            return T

In [ ]:
# Cell 5 — Author model adapters share CLIP weights, but keep each method's own preprocessing.
if globals().get('_NOTEBOOK_STAGE') == 'attack':
    CLIP_IDS = {
        'B16': 'openai/clip-vit-base-patch16',
        'B32': 'openai/clip-vit-base-patch32',
        'Laion': 'laion/CLIP-ViT-G-14-laion2B-s12B-b42K',
    }
    
    def clip_namespace(repo):
        ns = install_model_factories(dict(globals()))
        for name in BACKBONES:
            compile_author(repo, f'surrogates/FeatureExtractors/Clip{name}.py', [f'Clip{name}FeatureExtractor'], ns)
        return ns
    
    FOA_MODELS_NS = clip_namespace('foa')
    MPC_MODELS_NS = clip_namespace('mpc')
    MPC_EXTRAS_NS = install_model_factories(dict(globals()))
    compile_author('mpc', 'surrogates/FeatureExtractors/InternVL3_1B.py',
                   ['dynamic_preprocess_tensor', 'InternVL3_1B_FeatureExtractor'], MPC_EXTRAS_NS)
    compile_author('mpc', 'surrogates/FeatureExtractors/DINOv2_Base.py', ['DINOv2FeatureExtractor'], MPC_EXTRAS_NS)
    
    class TracedFOALoss(EnsembleFeatureLoss_OT_foa_attack):
        """Only observe detached scores; execute the inherited author loss unchanged."""
        def __call__(self, *args, **kwargs):
            previous = [float(v.detach()) for v in self.previous_loss_list]
            result = super().__call__(*args, **kwargs)
            scores = [float(v) for v in self.previous_loss_list]
            if not previous:
                previous = scores
            ratios = np.asarray([v / (p + 1e-8) for v, p in zip(scores, previous)])
            weights = np.exp(ratios); weights = weights / weights.sum() * len(weights)
            if not torch.isfinite(result) or not np.isfinite(weights).all():
                raise FloatingPointError('FOA returned non-finite score/weights; no numerical fallback applied.')
            self.last_record = {'model_scores': scores, 'weights': weights.tolist(), 'score': float(result.detach())}
            return result
    
    class AttackModels:
        def __init__(self, full_mpc=False):
            self.device = torch.device(ATTACK_DEVICE)
            self.foa_models, self.mpc_models = [], []
            for name in BACKBONES:
                self.foa_models.append(FOA_MODELS_NS[f'Clip{name}FeatureExtractor']().eval().requires_grad_(False).to(self.device))
                # LockedFactory returns the exact same CLIP weights, with MPC's own normalizer/tokenizer.
                self.mpc_models.append(MPC_MODELS_NS[f'Clip{name}FeatureExtractor']().eval().requires_grad_(False).to(self.device))
            self.foa_extractor = EnsembleFeatureExtractor_ot(self.foa_models, cluster_number=CLUSTER_NUMBER)
            self.internvl = self.dino = None
            if full_mpc:
                self.internvl = MPC_EXTRAS_NS['InternVL3_1B_FeatureExtractor']().eval().requires_grad_(False).to(self.device)
                self.dino = MPC_EXTRAS_NS['DINOv2FeatureExtractor']().eval().requires_grad_(False).to(self.device)
    
        def new_foa_loss(self):
            return TracedFOALoss(self.foa_models, cluster_number=CLUSTER_NUMBER)
    
        def to(self, device):
            for model in self.foa_models:  # mpc_models share these model weights.
                model.to(device)
            for model in (self.internvl, self.dino):
                if model is not None:
                    model.to(device)
            return self
    
        @contextlib.contextmanager
        def qwen_phase(self):
            # Never offload while a differentiable attack graph is live.
            if OFFLOAD_CLIP_DURING_QWEN:
                self.to('cpu'); cuda_cleanup()
            try:
                with preserve_rng():
                    yield
            finally:
                if OFFLOAD_CLIP_DURING_QWEN:
                    self.to(self.device)
    
        def clip_images(self, image, amp=False):
            with amp_context(amp):
                return [model(image).float() for model in self.mpc_models]
    
        @torch.no_grad()
        def clip_texts(self, texts):
            return [model.text_features_get(texts).float() for model in self.mpc_models]
    
        def foa_features(self, image, amp=False):
            # K-means/OT run FP32 even in the explicitly labeled hybrid AMP mode.
            if not amp:
                return self.foa_extractor(image)
            global_features, local_features = {}, {}
            for i, model in enumerate(self.foa_models):
                with amp_context(True):
                    global_f, local_f = model.global_local_features(image)
                global_features[i] = global_f.float().squeeze()
                local_features[i] = self.foa_extractor.get_cluster_center(local_f.float()[0], image.device).unsqueeze(0)
            return global_features, local_features
    
        def mpc_features(self, image, amp=False):
            clip = self.clip_images(image, amp=amp)
            blocks = [torch.cat(clip, dim=1)]
            if self.internvl is not None:
                blocks.extend([self.internvl(image).mean(dim=1).float(), self.dino(image)])
            return torch.cat(blocks, dim=1), clip
    
        @torch.no_grad()
        def anchors(self, source, target, text_record):
            img_src = self.clip_images(source)
            img_tgt = self.clip_images(target)
            text_src = self.clip_texts(text_record['source'])
            text_tgt = self.clip_texts(text_record['target'])
            src, tgt = mpc_clip_anchors(img_src, img_tgt, text_src, text_tgt, FUSION_LAMBDA)
            if self.internvl is not None:
                # Exact author order: all CLIP blocks, InternVL mean tokens, DINO CLS.
                src_intern = self.internvl(source).mean(dim=1).float()
                tgt_intern = self.internvl(target).mean(dim=1).float()
                src_dino, tgt_dino = self.dino(source), self.dino(target)
                src, tgt = torch.cat([src, src_intern, src_dino], dim=1), torch.cat([tgt, tgt_intern, tgt_dino], dim=1)
            negatives = text_record.get('negatives', [])
            neg_text = self.clip_texts([row['caption'] for row in negatives]) if negatives else []
            return {'source': src, 'target': tgt, 'target_text': text_tgt, 'negative_text': neg_text}
    
    def required_model_ids():
        ids = set(CLIP_IDS.values())
        if any(VARIANTS[v]['mpc'] for v in RUN_VARIANTS):
            ids.add(CAPTION_MODEL_ID)
        if any(VARIANTS[v]['full_mpc'] for v in RUN_VARIANTS):
            ids.add('facebook/dinov2-base')
        if any(VARIANTS[v]['negatives'] != 'none' or VARIANTS[v]['region'] != 'none' for v in RUN_VARIANTS):
            ids.add(ATTENTION_MODEL_ID)
        return sorted(ids)

In [ ]:
# Cell 6 — Pair once, export evaluation-compatible PNGs and JSON checkpoints.
if globals().get('_NOTEBOOK_STAGE') == 'attack':
    EXTENSIONS = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}
    
    def discover_dir(name, override):
        if override is not None:
            root = Path(override)
            if not root.is_dir():
                raise FileNotFoundError(root)
            return root
        if not DATASET_ROOT.is_dir():
            raise FileNotFoundError(f'DATASET_ROOT does not exist: {DATASET_ROOT}; edit Cell 1.')
        matches = ([DATASET_ROOT] if DATASET_ROOT.name == name else []) + [p for p in DATASET_ROOT.rglob(name) if p.is_dir()]
        if len(matches) != 1:
            raise ValueError(f'Expected one {name} directory, found {matches}; set explicit directory.')
        return matches[0]
    
    def to_pixels(pic):
        arr = np.array(pic, np.uint8, copy=True)
        return torch.from_numpy(arr).view(pic.size[1], pic.size[0], len(pic.getbands())).permute(2, 0, 1).contiguous().float()
    
    IMAGE_TRANSFORM = transforms.Compose([
        transforms.Resize(INPUT_RES, interpolation=transforms.InterpolationMode.BICUBIC),
        transforms.CenterCrop(INPUT_RES), transforms.Lambda(lambda im: im.convert('RGB')), transforms.Lambda(to_pixels),
    ])
    
    def load_pixels(path):
        with Image.open(path) as image:
            # ImageFolder's default PIL loader converts to RGB before applying transforms.
            return IMAGE_TRANSFORM(image.convert('RGB')).unsqueeze(0)
    
    def prepare_pairs():
        if PAIR_MANIFEST is not None:
            manifest = Path(PAIR_MANIFEST)
            rows = [json.loads(line) for line in manifest.read_text(encoding='utf-8').splitlines() if line.strip()]
            pairs = []
            for row in rows:
                roles = row.get('images', row)
                index = row.get('index')
                if type(index) is not int or index < 0:
                    raise ValueError('PAIR_MANIFEST requires a nonnegative integer index.')
                pair = {'index': index, 'sample_id': f'{index:05d}'}
                for role in ('source', 'target'):
                    p = Path(roles[role]); p = p if p.is_absolute() else manifest.parent / p
                    pair[role] = str(p.resolve())
                pairs.append(pair)
        else:
            source = datasets.ImageFolder(discover_dir('bigscale', SOURCE_DIR))
            target = datasets.ImageFolder(discover_dir('target_images', TARGET_DIR))
            if len(source) != len(target):
                raise ValueError(f'Source/target lengths differ ({len(source)}/{len(target)}); provide PAIR_MANIFEST instead of silent truncation.')
            pairs = [{'index': i, 'sample_id': f'{i:05d}', 'source': s[0], 'target': t[0]}
                     for i, (s, t) in enumerate(zip(source.samples, target.samples))]
        pairs = pairs[SAMPLE_START:] if NUM_SAMPLES is None else pairs[SAMPLE_START:SAMPLE_START + NUM_SAMPLES]
        if not pairs or (NUM_SAMPLES is not None and len(pairs) < NUM_SAMPLES):
            raise ValueError('Not enough selected image pairs.')
        if len({p['index'] for p in pairs}) != len(pairs):
            raise ValueError('Duplicate manifest indices.')
        for pair in pairs:
            pair['input_sha256'] = {role: sha256_file(pair[role]) for role in ('source', 'target')}
        return pairs
    
    class PairRoleDataset(torch.utils.data.Dataset):
        def __init__(self, pairs, role):
            self.pairs, self.role = pairs, role
        def __len__(self):
            return len(self.pairs)
        def __getitem__(self, i):
            return load_pixels(self.pairs[i][self.role]).squeeze(0), i
    
    def pair_batches(pairs):
        # Two independent DataLoader iterators preserve author base-seed consumption/order.
        src = torch.utils.data.DataLoader(PairRoleDataset(pairs, 'source'), batch_size=1, shuffle=False, num_workers=0)
        tgt = torch.utils.data.DataLoader(PairRoleDataset(pairs, 'target'), batch_size=1, shuffle=False, num_workers=0)
        for (source, i), (target, j) in zip(src, tgt):
            if int(i.item()) != int(j.item()):
                raise AssertionError('Pair index mismatch.')
            yield pairs[int(i.item())], source, target
    
    def pixels_to_pil(pixels):
        arr = pixels.detach().cpu().squeeze(0).clamp(0, 255).round().byte().permute(1, 2, 0).numpy()
        return Image.fromarray(arr)
    
    def tensor_pixel_hash(pixels):
        return hashlib.sha256(pixels.detach().cpu().contiguous().float().numpy().tobytes()).hexdigest()
    
    def save_result(directory, pair, source, target, adversarial):
        directory = Path(directory)
        directory.mkdir(parents=True, exist_ok=True)
        if not torch.isfinite(adversarial).all():
            raise FloatingPointError('Invalid adversarial pixels.')
        linf = float((adversarial.detach().cpu() * 255 - source.cpu()).abs().max())
        if linf > EPSILON_255 + 1e-3:
            raise AssertionError(f'Tensor budget violated: {linf} > {EPSILON_255}')
        images = {}
        for role, tensor in [('source', source / 255), ('target', target / 255), ('adversarial', adversarial)]:
            relative = f'samples/{pair["sample_id"]}/{role}.png'
            path = directory / relative; path.parent.mkdir(parents=True, exist_ok=True)
            torchvision.utils.save_image(tensor.detach().cpu(), path)
            images[role] = relative
        a = np.asarray(Image.open(directory / images['adversarial'])).astype(np.int16)
        s = np.asarray(Image.open(directory / images['source'])).astype(np.int16)
        saved_linf = int(np.abs(a - s).max())
        if saved_linf > EPSILON_255 + 1e-6:
            raise AssertionError(f'PNG budget violated: {saved_linf} > {EPSILON_255}')
        return {**pair, **images, 'original_paths': {r: pair[r] for r in ('source', 'target')},
                'image_sha256': {r: sha256_file(directory / p) for r, p in images.items()},
                'linf_255': linf, 'saved_linf_255': saved_linf, 'epsilon_255': EPSILON_255}
    
    def complete_record_ok(directory, row):
        for role in ('source', 'target', 'adversarial'):
            p = Path(directory) / row[role]
            if not p.is_file() or sha256_file(p) != row['image_sha256'][role]:
                return False
        return True
    
    def state_checkpoint(foa_loss):
        return {'rng': capture_rng(), 'foa_previous': [v.detach().cpu().tolist() for v in foa_loss.previous_loss_list] if foa_loss is not None else []}
    
    def restore_checkpoint(state, foa_loss):
        restore_rng(state['rng'])
        if foa_loss is not None:
            foa_loss.previous_loss_list = [torch.tensor(v, device=ATTACK_DEVICE) for v in state['foa_previous']]

## Negative concepts và loss

Qwen nhận **target image + target caption** và sinh counterfactual thay đúng một subject/object phrase.
Bank cố định theo mẫu; hard top-k lấy mean cosine với ảnh adversarial qua ba CLIP mỗi 10 bước.
Đây là kiểm tra cấu trúc, không chứng nhận mọi counterfactual sai về mặt hình ảnh; xem `text_records.json`.
Không có fallback negative theo template.

Với $s^+=\cos(z_a,t_t)$, $s_i^-=\cos(z_a,n_i)$, mỗi loss tính trong không gian riêng từng CLIP rồi trung bình:

| `NEGATIVE_LOSS` | Minimize | Ý nghĩa |
|---|---|---|
| `infonce` | $\log(1+\sum_i e^{(s_i^- -s^+)/\tau_n})$ | Target discrimination, mặc định |
| `margin` | $\frac1k\sum_i[m+s_i^- -s^+]_+$ | Target phải hơn từng negative một margin |
| `hard_margin` | $[m+\max_i s_i^- -s^+]_+$ | Tập trung negative khó nhất |
| `repulsion` | $\frac1k\sum_i s_i^-$ | Chỉ đẩy xa negatives, cần alignment đi kèm |

Hybrid **maximize** $J=w_F S_{FOA}+w_M S_{MPC}-w_N L_{neg}$; mặc định $(1,1,0.5)$.
Trọng số FOA chỉ dựa trên FOA scores, không dựa trên tổng hybrid loss.
MPC-CLIP concatenate embeddings trước tính CE; không lấy trung bình CE từng backbone.

In [ ]:
# Cell 7 — Editable objectives. Scores are maximized; negative losses are minimized.
if globals().get('_NOTEBOOK_STAGE') == 'attack':
    def mpc_clip_anchors(source_images, target_images, source_texts, target_texts, lam=0.6):
        source = torch.cat([lam * im + (1 - lam) * text for im, text in zip(source_images, source_texts)], dim=1)
        target = torch.cat([lam * im + (1 - lam) * text for im, text in zip(target_images, target_texts)], dim=1)
        return source, target  # Do not independently renormalize fused anchors.
    
    def mpc_score(adv, target, source, temperature=TEMPERATURE, omega=POSITIVE_OMEGA):
        adv, target, source = (F.normalize(x, dim=-1) for x in (adv, target, source))
        pos = (adv * target).sum(-1) * omega / temperature
        neg = (adv * source).sum(-1) / temperature
        return -F.cross_entropy(torch.stack([pos, neg], dim=1), torch.zeros(adv.shape[0], device=adv.device, dtype=torch.long))
    
    def negative_loss_from_scores(positive, negatives, kind=NEGATIVE_LOSS, tau=NEGATIVE_TEMPERATURE, margin=NEGATIVE_MARGIN):
        if negatives.ndim != 2 or positive.shape != negatives.shape[:1] or negatives.shape[1] == 0:
            raise ValueError('Expected positive [B], negatives [B,K], K>0.')
        if kind == 'infonce':
            gaps = (negatives - positive[:, None]) / tau
            return torch.logsumexp(torch.cat([torch.zeros_like(positive[:, None]), gaps], dim=1), dim=1).mean()
        if kind == 'margin':
            return F.relu(margin + negatives - positive[:, None]).mean()
        if kind == 'hard_margin':
            return F.relu(margin + negatives.max(dim=1).values - positive).mean()
        if kind == 'repulsion':
            return negatives.mean()
        raise ValueError(kind)
    
    def concept_negative_loss(image_features, anchors, negative_ids):
        ids = torch.as_tensor(negative_ids, device=image_features[0].device, dtype=torch.long)
        losses = []
        for image, positive, bank in zip(image_features, anchors['target_text'], anchors['negative_text']):
            z = F.normalize(image.float(), dim=-1)
            positive = F.normalize(positive.float(), dim=-1)
            negative = F.normalize(bank.index_select(0, ids).float(), dim=-1)
            losses.append(negative_loss_from_scores((z * positive).sum(-1), z @ negative.T))
        if len(losses) != len(image_features):
            raise ValueError('Negative bank missing a CLIP backbone.')
        return torch.stack(losses).mean()
    
    @torch.no_grad()
    def negative_scores(image_features, negative_text_features):
        if len(image_features) != len(negative_text_features):
            raise ValueError('Image/text backbone mismatch.')
        return torch.stack([(F.normalize(z.float(), dim=-1) @ F.normalize(bank.float(), dim=-1).T).mean(0)
                            for z, bank in zip(image_features, negative_text_features)]).mean(0)
    
    def select_negatives(scores, k, mode, rng):
        scores = scores.detach().cpu()
        if not torch.isfinite(scores).all() or k > scores.numel():
            raise ValueError('Not enough finite negative scores.')
        if mode == 'random':
            return rng.sample(range(len(scores)), k)
        # Stable tie-breaking by candidate index.
        return sorted(range(len(scores)), key=lambda i: (-float(scores[i]), i))[:k]
    
    def initialize_delta(source_cpu, mpc_only):
        if mpc_only:
            # Author initializes on CPU with a 16/255 Gaussian, although the attack image uses [0,255].
            # Keep this unusual scale verbatim rather than changing it to 16 pixel units.
            return (16 / 255 * torch.randn_like(source_cpu).to(ATTACK_DEVICE)).requires_grad_(True)
        return torch.zeros_like(source_cpu, device=ATTACK_DEVICE, requires_grad=True)
    
    def projected_ascent(delta, gradient, momentum, decay, alpha=ALPHA_255, epsilon=EPSILON_255):
        momentum = momentum * decay + gradient
        updated = torch.clamp(delta.detach() + alpha * torch.sign(momentum), min=-epsilon, max=epsilon)
        return updated.requires_grad_(True), momentum.detach()

In [ ]:
# Cell 8 — Cache InternVL captions and image-conditioned Qwen counterfactuals.
if globals().get('_NOTEBOOK_STAGE') == 'attack':
    CAPTION_PROMPT = 'Describe this image in one concise sentence, no longer than 20 words.'
    
    def clean_text(text):
        if not isinstance(text, str):
            raise ValueError('Text must be a string.')
        return ' '.join(text.split())
    
    """CF-VLM counterfactual text edit policy, adapted for attack negative captions.

Sources: NeurIPS 2025 CF-VLM, Section 2.2 and Appendix A (page 17),
https://github.com/HCP-AI-Research-Lab/CF-VLM/blob/main/process.py.
This is the text edit policy, not a reproduction of CF-VLM image generation or
fine-tuning. JSON edit traces are our addition for auditing/cache validation.
Structural validation cannot certify semantic minimality or visual falsity.
"""
    import hashlib
    import json
    
    NEGATIVE_POLICY_VERSION = 'cfvlm_text_edits_v2_audited'
    CF_ATTRIBUTE_TYPES = (
        'color', 'material', 'shape', 'size', 'position_orientation', 'state_pose',
        'object_category', 'parts_structure', 'movement_status', 'quantity',
        'expression_emotion',
    )
    CF_PROMPT_POLICY = """Generate counterfactual rewrites of the factual target caption.
Each candidate must make exactly ONE semantic intervention:
(A) change ONE attribute of ONE object: color, material, shape, size,
position/orientation, state/pose, object category, parts/structure, movement
status, quantity, or facial expression/emotion; OR
(B) change ONE cause-effect link already described by the caption. Change the
cause and, when needed, its dependent outcome together so the alternate event
is coherent. These two linked text edits count as one causal intervention.
Do not invent a causal chain absent from the caption. If it has no genuine
cause-effect link, use attribute edits only; do not label a category swap causal.
Keep every unrelated object, attribute, action, relation and background intact.
Never replace a whole scene or change weather and location together. For
example, snowy forest -> sunny meadow is NOT a color edit and is invalid.
Position/orientation changes the spatial placement of the SAME object.
State/pose changes one physical state, not the time/history of the whole scene.
Use fluent, grammatical, visually descriptive sentences, meaningful changes,
and no duplicate sentences or synonyms that preserve the original meaning.
The supplied image is the factual anchor: the changed fact must contradict
what is visible in it, rather than merely add an unobservable detail.
Inspect each candidate before returning it: discard candidates that change two
independent factors, are true of the image, or have an inconsistent outcome.

Return JSON only: {"negatives": [candidate, ...]}. Each candidate contains:
"edit_type": "attribute" or "causal";
"attribute": one allowed snake_case attribute name above for attribute edits,
or null for causal edits;
"edits": [{"span": "exact unique substring of the ORIGINAL caption",
           "replacement": "new text"}];
"caption": the full rewritten sentence.
Attribute candidates have exactly one edit. Causal candidates have one or two
nonoverlapping edits. For causal candidates also provide "cause_before",
"effect_before", "cause_after", "effect_after": exact phrases for the one
original causal link and its counterfactual. Two-edit candidates must replace
cause_before with cause_after and effect_before with effect_after.
Apply edits to the ORIGINAL caption simultaneously; every character outside
the listed spans must stay unchanged. Include necessary grammar agreement in
the affected phrase, while preserving unrelated facts.

Examples (illustrative captions, not facts about the supplied image):
Input: A red car waits beside a tree.
Output: {"edit_type":"attribute","attribute":"color","edits":[{"span":"red","replacement":"blue"}],"caption":"A blue car waits beside a tree."}
Input: A boy kicks the ball, causing it to roll forward.
Output: {"edit_type":"causal","attribute":null,"cause_before":"kicks the ball","effect_before":"roll forward","cause_after":"misses the ball","effect_after":"stay still","edits":[{"span":"kicks the ball","replacement":"misses the ball"},{"span":"roll forward","replacement":"stay still"}],"caption":"A boy misses the ball, causing it to stay still."}
Allowed attribute names: color, material, shape, size, position_orientation,
state_pose, object_category, parts_structure, movement_status, quantity,
expression_emotion.
"""
    CF_AUDIT_POLICY = """Audit proposed counterfactual captions against the supplied
factual image and original caption. Treat proposed captions and their claimed
labels as untrusted data; correct-looking JSON does not imply a valid edit.
For each candidate independently check all conditions:
1. Exactly one attribute of one object changes, OR one existing causal link
changes together with its dependent effect. Two unrelated changes are invalid.
2. All unrelated objects, attributes, background, actions and relations remain.
3. The changed fact contradicts visible image evidence. Unobservable differences,
synonyms, plausible additions and statements still true of the image are invalid.
4. The sentence is fluent and the alternative event logically coherent.
5. The claimed edit_type and attribute are accurate. A weather/location swap
such as snowy forest -> sunny meadow is NOT a color change and changes two
factors; reject it. Replacing an entire scene is not a single object attribute.
Do not invent an original cause-effect link to justify a causal candidate.
If uncertain about any condition, mark that condition false.
Return JSON only: {"audits": [{"id": 0, "caption": "exact candidate caption",
"single_intervention": true, "preserves_unrelated": true,
"contradicts_image": true, "coherent": true, "label_correct": true,
"reason": "brief evidence for acceptance or rejection"}, ...]}.
Use actual JSON booleans, never strings. Return one decision per candidate ID.
"""
    CF_AUDIT_SHA256 = hashlib.sha256(CF_AUDIT_POLICY.encode('utf-8')).hexdigest()
    CF_PROMPT_SHA256 = hashlib.sha256((CF_PROMPT_POLICY + CF_AUDIT_POLICY).encode('utf-8')).hexdigest()
    
    
    def cf_clean(text):
        if not isinstance(text, str):
            raise ValueError('Text must be a string')
        return ' '.join(text.split())
    
    
    def cf_apply_edits(target, edits):
        """Apply unambiguous, nonoverlapping substitutions to the original text."""
        positions = []
        for edit in edits:
            span, replacement = cf_clean(edit['span']), cf_clean(edit['replacement'])
            if not span or not replacement or span.casefold() == replacement.casefold():
                raise ValueError('Empty or identity edit')
            if target.count(span) != 1:
                raise ValueError('Edit span must occur exactly once')
            start = target.index(span)
            positions.append((start, start + len(span), replacement))
        positions.sort()
        if any(a[1] > b[0] for a, b in zip(positions, positions[1:])):
            raise ValueError('Overlapping edits')
        result = target
        for start, end, replacement in reversed(positions):
            result = result[:start] + replacement + result[end:]
        return result
    
    
    def cf_validate_candidates(payload, target_caption, existing=(), max_candidates=12):
        if isinstance(payload, dict):
            payload = payload.get('negatives', [])
        if not isinstance(payload, list):
            raise ValueError('Response must contain a negatives list')
        target, valid, seen = cf_clean(target_caption), [], set()
        # Revalidate existing entries too. Legacy noun-only banks have no edit_type
        # and are deliberately ineligible for this policy.
        for row in [*existing, *payload]:
            if not isinstance(row, dict):
                continue
            try:
                kind, attribute = row['edit_type'], row.get('attribute')
                edits = row['edits']
                if not isinstance(edits, list) or not edits:
                    continue
                edits = [{'span': cf_clean(e['span']), 'replacement': cf_clean(e['replacement'])} for e in edits]
                caption = cf_clean(row['caption'])
                if kind == 'attribute':
                    if attribute not in CF_ATTRIBUTE_TYPES or len(edits) != 1:
                        continue
                    trace = {}
                elif kind == 'causal':
                    if attribute is not None or len(edits) not in (1, 2):
                        continue
                    trace = {key: cf_clean(row[key]) for key in
                             ('cause_before', 'effect_before', 'cause_after', 'effect_after')}
                    if any(not v for v in trace.values()):
                        continue
                    if any(trace[k] not in target for k in ('cause_before', 'effect_before')):
                        continue
                    if any(trace[k] not in caption for k in ('cause_after', 'effect_after')):
                        continue
                    if trace['cause_before'].casefold() == trace['cause_after'].casefold():
                        continue
                    if trace['effect_before'].casefold() == trace['effect_after'].casefold():
                        continue
                    if len(edits) == 2 and {(e['span'], e['replacement']) for e in edits} != {
                        (trace['cause_before'], trace['cause_after']),
                        (trace['effect_before'], trace['effect_after']),
                    }:
                        continue
                else:
                    continue
                if cf_apply_edits(target, edits) != caption:
                    continue
                if caption.casefold() == target.casefold() or caption.casefold() in seen:
                    continue
                if any(marker in caption for marker in ('<|', '|>')):
                    continue
                candidate = {'caption': caption, 'edit_type': kind, 'attribute': attribute,
                             'edits': edits, **trace, 'negative_policy': NEGATIVE_POLICY_VERSION}
                if cf_audit_valid(row.get('semantic_audit'), caption):
                    candidate['semantic_audit'] = dict(row['semantic_audit'])
                valid.append(candidate)
                seen.add(caption.casefold())
                if len(valid) >= max_candidates:
                    break
            except (ValueError, KeyError, TypeError, AttributeError):
                continue
        return valid
    
    
    def cf_negative_prompt(target_caption, attempt=0, accepted=(), count=4, feedback=()):
        return (CF_PROMPT_POLICY + '\nFactual target caption: ' + json.dumps(cf_clean(target_caption))
                + f'\nGenerate {count} new candidates. Attempt {attempt + 1}.'
                + '\nAvoid these already accepted captions: '
                + json.dumps([r['caption'] for r in accepted], ensure_ascii=False)
                + '\nPreviously rejected candidates and reasons; avoid repeating them: '
                + json.dumps(list(feedback)[-8:], ensure_ascii=False))
    
    
    def cf_negative_audit_prompt(target_caption, candidates):
        data = {'original_caption': cf_clean(target_caption),
                'candidates': [{'id': i, **{k: row[k] for k in ('caption', 'edit_type', 'attribute', 'edits')}}
                               for i, row in enumerate(candidates)]}
        return CF_AUDIT_POLICY + '\nInput data:\n' + json.dumps(data, ensure_ascii=False)
    
    
    def cf_audit_valid(audit, caption):
        return (isinstance(audit, dict) and audit.get('caption') == caption
                and audit.get('policy') == NEGATIVE_POLICY_VERSION
                and audit.get('audit_prompt_sha256') == CF_AUDIT_SHA256
                and all(audit.get(k) is True for k in ('single_intervention', 'preserves_unrelated',
                                                       'contradicts_image', 'coherent', 'label_correct')))
    
    
    def cf_validate_audits(payload, candidates):
        """Fail closed on absent/duplicate IDs, false/uncertain flags or wrong text."""
        if not isinstance(payload, dict) or not isinstance(payload.get('audits'), list):
            raise ValueError('Audit response must contain an audits list')
        by_id, duplicate = {}, set()
        for audit in payload['audits']:
            if not isinstance(audit, dict):
                continue
            index = audit.get('id')
            if type(index) is not int or not 0 <= index < len(candidates):
                continue
            if index in by_id:
                duplicate.add(index)
            by_id[index] = audit
        approved = []
        for index, candidate in enumerate(candidates):
            if index in duplicate or index not in by_id:
                continue
            audit = {**by_id[index], 'policy': NEGATIVE_POLICY_VERSION, 'audit_prompt_sha256': CF_AUDIT_SHA256}
            if cf_audit_valid(audit, candidate['caption']):
                approved.append({**candidate, 'semantic_audit': audit})
        return approved
    
    
    def cf_only_audited(candidates):
        return [row for row in candidates if cf_audit_valid(row.get('semantic_audit'), row.get('caption'))]
    
    def validate_negative_candidates(payload, target_caption, existing=()):
        return cf_validate_candidates(payload, target_caption, existing, CF_NEGATIVE_CANDIDATES)
    
    def parse_json_response(text):
        decoder = json.JSONDecoder()
        # Accept a fenced JSON object/array, but never eval model output.
        for match in re.finditer(r'[\[{]', text):
            try:
                value, _ = decoder.raw_decode(text[match.start():])
                if isinstance(value, (dict, list)):
                    return value
            except json.JSONDecodeError:
                pass
        raise ValueError('No JSON object/array in Qwen response.')
    
    @torch.inference_mode()
    def qwen_generate_negatives_batch(backend, requests, audit=False):
        images, texts = [], []
        for request in requests:
            pixels, caption, attempt, accepted = request[:4]
            feedback = request[4] if len(request) > 4 else ()
            images.append(pixels_to_pil(pixels).resize((ATTENTION_INPUT_RES, ATTENTION_INPUT_RES), Image.Resampling.BICUBIC))
            prompt = (cf_negative_audit_prompt(caption, accepted) if audit else
                      cf_negative_prompt(caption, attempt, accepted, CF_NEGATIVE_CANDIDATES - len(accepted), feedback))
            messages = [{'role': 'user', 'content': [{'type': 'image'}, {'type': 'text', 'text': prompt}]}]
            texts.append(backend.processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))
        tokenizer = backend.processor.tokenizer
        previous_padding = tokenizer.padding_side
        tokenizer.padding_side = 'left'
        inputs = None
        backend.model.model.rope_deltas = None
        try:
            inputs = backend.processor(text=texts, images=images, do_resize=False, padding=True, return_tensors='pt').to(backend.device)
            ids = backend.model.generate(**inputs, do_sample=False, max_new_tokens=CONCEPT_MAX_NEW_TOKENS)
            eos = backend.model.generation_config.eos_token_id
            eos = [eos] if isinstance(eos, int) else list(eos or [])
            result = []
            for output in ids:
                suffix = output[inputs['input_ids'].shape[1]:]
                raw = tokenizer.decode(suffix, skip_special_tokens=True)
                # A finished row may be padded while another row reaches the cap.
                truncated = len(suffix) >= CONCEPT_MAX_NEW_TOKENS and not any(int(t) in eos for t in suffix)
                result.append((raw, truncated))
            return result
        except torch.cuda.OutOfMemoryError:
            if len(requests) == 1:
                raise
            inputs = None
            cuda_cleanup()
            middle = len(requests) // 2
            event('text', 'negative_batch_oom_split', requested=len(requests))
            return (qwen_generate_negatives_batch(backend, requests[:middle], audit=audit)
                    + qwen_generate_negatives_batch(backend, requests[middle:], audit=audit))
        finally:
            tokenizer.padding_side = previous_padding
            backend.model.model.rope_deltas = None
    
    def qwen_generate_negatives(backend, target_pixels, target_caption, attempt, accepted):
        return qwen_generate_negatives_batch(backend, [(target_pixels, target_caption, attempt, accepted)])[0]
    
    def prepare_text_records(pairs, include_negatives=True):
        global ATTENTION_BACKEND
        cache_path = RUN_DIR / 'text_records.json'
        records = json.loads(cache_path.read_text(encoding='utf-8')) if cache_path.is_file() else {}
        need_captions = any(VARIANTS[v]['mpc'] for v in RUN_VARIANTS)
        need_negatives = include_negatives and any(VARIANTS[v]['negatives'] != 'none' for v in RUN_VARIANTS)
        captioner = None
        try:
            for pair in pairs:
                sid = pair['sample_id']; previous = records.get(sid, {})
                if previous and previous.get('input_sha256') != pair['input_sha256']:
                    raise ValueError(f'Text cache input mismatch: {sid}')
                if previous.get('source') and previous.get('target'):
                    continue
                record = {'input_sha256': pair['input_sha256'], 'source': None, 'target': None,
                          'negatives': [], 'caption_model': CAPTION_MODEL_ID, 'caption_prompt': CAPTION_PROMPT}
                override = MANUAL_TEXT_OVERRIDES.get(sid, {})
                try:
                    for role in ('source', 'target'):
                        if override.get(role):
                            caption = clean_text(override[role])
                            record[f'{role}_provenance'] = 'manual_override'
                        elif need_captions:
                            if captioner is None:
                                with preserve_rng():
                                    captioner = MPC_EXTRAS_NS['InternVL3_1B_FeatureExtractor']().eval().requires_grad_(False).to(CAPTION_DEVICE)
                            # Original MPC caption generator, including its original 50-token limit.
                            with preserve_rng():
                                caption = clean_text(captioner.generate_description(load_pixels(pair[role])))
                            record[f'{role}_provenance'] = 'mpc_author_captioner'
                        else:
                            caption = ''
                        if need_captions and not caption:
                            raise ValueError(f'Empty {role} caption.')
                        record[role] = caption
                    records[sid] = record
                    event('text', 'captions_cached', sample_id=sid)
                except Exception as exc:
                    record['caption_error'] = f'{type(exc).__name__}: {exc}'
                    records[sid] = record
                    event('text', 'caption_error', sample_id=sid, error=record['caption_error'])
                if len(records) % 32 == 0:
                    write_json(cache_path, records)
            write_json(cache_path, records)
        finally:
            del captioner
            cuda_cleanup()
    
        if need_negatives:
            policy = {'version': NEGATIVE_POLICY_VERSION, 'prompt_sha256': CF_PROMPT_SHA256,
                      'candidates': CF_NEGATIVE_CANDIDATES, 'batch_size': NEGATIVE_GENERATION_BATCH_SIZE}
            for pair in pairs:
                record = records[pair['sample_id']]
                previous = record.get('negative_policy')
                if previous != policy:
                    if record.get('negatives'):
                        event('text', 'stale_negative_bank_rejected', sample_id=pair['sample_id'])
                    record.update(negatives=[], negative_attempts=[], negative_rejections=[], negative_policy=policy)
                    record.pop('negative_error', None)
                if not record.get('caption_error'):
                    record['negatives'] = cf_only_audited(validate_negative_candidates(record.get('negatives', []), record['target']))
            write_json(cache_path, records)
            for attempt in range(TEXT_RETRIES):
                pending = [p for p in pairs if not records[p['sample_id']].get('caption_error')
                           and len(records[p['sample_id']]['negatives']) < CF_NEGATIVE_CANDIDATES]
                if not pending:
                    break
                for offset in range(0, len(pending), NEGATIVE_GENERATION_BATCH_SIZE):
                    batch = pending[offset:offset + NEGATIVE_GENERATION_BATCH_SIZE]
                    requests = []
                    generated_pairs = []
                    for pair in batch:
                        sid = pair['sample_id']; record = records[sid]
                        override = MANUAL_TEXT_OVERRIDES.get(sid, {})
                        if 'negatives' in override:
                            if override.get('negative_policy') != policy:
                                raise ValueError('Manual negative bank must declare the exact current policy')
                            record['negatives'] = cf_only_audited(validate_negative_candidates(override['negatives'], record['target']))
                            record['negative_attempts'].append({'provenance': 'manual_override'})
                        else:
                            requests.append((load_pixels(pair['target']), record['target'], attempt, record['negatives'], record.get('negative_rejections', [])))
                            generated_pairs.append(pair)
                    if requests:
                        backend = ensure_attention_backend(MODEL_REVISIONS[ATTENTION_MODEL_ID])
                        with preserve_rng():
                            outputs = qwen_generate_negatives_batch(backend, requests)
                        audit_requests, audit_pairs, proposed = [], [], []
                        for pair, (raw, truncated) in zip(generated_pairs, outputs):
                            record = records[pair['sample_id']]
                            try:
                                if truncated:
                                    raise ValueError('Qwen output reached token cap without EOS')
                                candidates = cf_validate_candidates(parse_json_response(raw), record['target'],
                                                                    max_candidates=CF_NEGATIVE_CANDIDATES-len(record['negatives']))
                                record['negative_attempts'].append({'attempt': attempt + 1, 'raw': raw, 'proposed': len(candidates)})
                                if candidates:
                                    audit_requests.append((load_pixels(pair['target']), record['target'], attempt, candidates))
                                    audit_pairs.append(pair)
                                    proposed.append(candidates)
                            except Exception as exc:
                                record['negative_attempts'].append({'attempt': attempt + 1, 'raw': raw, 'error': f'{type(exc).__name__}: {exc}'})
                        if audit_requests:
                            with preserve_rng():
                                audits = qwen_generate_negatives_batch(backend, audit_requests, audit=True)
                            for pair, candidates, (raw_audit, truncated) in zip(audit_pairs, proposed, audits):
                                record = records[pair['sample_id']]
                                info = record['negative_attempts'][-1]
                                info['audit_raw'] = raw_audit
                                try:
                                    if truncated:
                                        raise ValueError('Qwen audit reached token cap without EOS')
                                    audit_payload = parse_json_response(raw_audit)
                                    approved = cf_validate_audits(audit_payload, candidates)
                                    approved_captions = {r['caption'] for r in approved}
                                    reasons = {r.get('id'): r.get('reason', 'Rejected by semantic audit')
                                               for r in audit_payload.get('audits', []) if isinstance(r, dict)}
                                    record.setdefault('negative_rejections', []).extend(
                                        {'caption': r['caption'], 'reason': reasons.get(i, 'Missing/invalid audit')}
                                        for i, r in enumerate(candidates) if r['caption'] not in approved_captions)
                                    record['negatives'] = cf_only_audited(validate_negative_candidates(approved, record['target'], record['negatives']))
                                    info.update(audit_approved=len(approved), accepted=len(record['negatives']))
                                except Exception as exc:
                                    info['audit_error'] = f'{type(exc).__name__}: {exc}'
                    # Checkpoint every GPU batch, including incomplete banks and raw
                    # responses. No fabricated fallback or silent cohort filtering.
                    for pair in batch:
                        record = records[pair['sample_id']]
                        if len(record['negatives']) < CF_NEGATIVE_CANDIDATES:
                            record['negative_error'] = f"Need {CF_NEGATIVE_CANDIDATES} CF-VLM candidates; got {len(record['negatives'])}"
                        else:
                            record.pop('negative_error', None)
                        event('text', 'negative_bank_cached', sample_id=pair['sample_id'],
                              candidates=len(record['negatives']), error=record.get('negative_error'))
                    write_json(cache_path, records)
        return records


## Qwen causal self-attention và chuỗi vùng

Teacher-force **source caption cố định** vào assistant suffix; lấy query rows của content tokens
tới các image-token keys tại layer 29. Không dùng CLIP gradient-weighted rollout.
Softmax trên **toàn bộ causal keys** trước khi chọn image keys; head mean và token normalization
giữ protocol v3.2. “Causal” là decoder causal self-attention, không phải bằng chứng attribution nhân quả.

Hotspots có 10 mức diện tích từ 0.1 tới 1.0, tối đa ba box/mức, IoU cap 0.3 với fallback như v3.2.
Dynamic refresh mỗi 30 bước, giữ index theo **bước tuyệt đối**, không bắt đầu lại stage đầu tiên.
Crop tự giới hạn gradient vào vùng lấy mẫu; không thêm pixel mask hoặc temporal-credit/EMA.
Random-region control dùng cùng phase/area/shape và crop primitive nhưng randomize vị trí box.

In [ ]:
"""Fixed-area hotspot ranking with the original IoU fallback."""
if globals().get('_NOTEBOOK_STAGE') == 'attack':
    
    
    
    def compute_box_iou(box1, box2):
        # box: (x_min, x_max, y_min, y_max) inclusive
        x1_min, x1_max, y1_min, y1_max = box1
        x2_min, x2_max, y2_min, y2_max = box2
    
        inter_x_min = max(x1_min, x2_min)
        inter_x_max = min(x1_max, x2_max)
        inter_y_min = max(y1_min, y2_min)
        inter_y_max = min(y1_max, y2_max)
    
        inter_w = max(0, inter_x_max - inter_x_min + 1)
        inter_h = max(0, inter_y_max - inter_y_min + 1)
        inter_area = inter_w * inter_h
    
        area1 = (x1_max - x1_min + 1) * (y1_max - y1_min + 1)
        area2 = (x2_max - x2_min + 1) * (y2_max - y2_min + 1)
    
        union_area = area1 + area2 - inter_area
        return inter_area / union_area if union_area > 0 else 0.0
    
    
    def get_top_k_boxes_fixed_area(attention_map, target_ratio, k=1, iou_threshold=0.3, area_tolerance=0.05):
        """
    Find top-k boxes with maximum attention score that have area approximately equal to target_ratio * total_area.
    """
        H, W = attention_map.shape
        total_cells = H * W
        target_cells = int(target_ratio * total_cells)
        if target_cells < 1:
            target_cells = 1
    
        # When target ratio is effectively the full map, force search to stay near full area
        near_full = target_ratio >= 0.999
        area_lower = total_cells if near_full else max(1, int(target_cells * (1 - area_tolerance)))
        area_upper = total_cells if near_full else min(total_cells, int(target_cells * (1 + area_tolerance)))
    
        integral = np.cumsum(np.cumsum(attention_map, axis=0), axis=1)
    
        def get_sum(r1, c1, r2, c2):
            res = integral[r2, c2]
            if r1 > 0:
                res -= integral[r1 - 1, c2]
            if c1 > 0:
                res -= integral[r2, c1 - 1]
            if r1 > 0 and c1 > 0:
                res += integral[r1 - 1, c1 - 1]
            return res
    
        candidates = []
    
        for h in range(1, H + 1):
            w = int(round(target_cells / h))
            if w < 1:
                w = 1
            if w > W:
                w = W
    
            # Aspect ratio constraint to prevent elongated boxes
            if h / w > 3.0 or w / h > 3.0:
                continue
    
            area = h * w
            # Enforce area closeness to target to avoid undersized boxes dominating
            if area < area_lower or area > area_upper:
                continue
    
            for r in range(H - h + 1):
                for c in range(W - w + 1):
                    current_sum = get_sum(r, c, r + h - 1, c + w - 1)
                    # current_mean = current_sum / area
                    candidates.append((current_sum, (c, c + w - 1, r, r + h - 1)))
    
        # Sort by score descending
        candidates.sort(key=lambda x: x[0], reverse=True)
    
        # Select top-k with IoU filtering
        selected_boxes = []
        for _, box in candidates:
            if len(selected_boxes) >= k:
                break
    
            # Check IoU with already selected boxes
            is_valid = True
            for selected_box in selected_boxes:
                if compute_box_iou(box, selected_box) > iou_threshold:
                    is_valid = False
                    break
    
            if is_valid:
                selected_boxes.append(box)
    
        # Fallback: if we could not gather k boxes under the IoU cap,
        # fill the rest with the next best candidates even if they overlap.
        if len(selected_boxes) < k and candidates:
            for _, box in candidates:
                if len(selected_boxes) >= k:
                    break
                # Avoid exact duplicates
                if any(box == chosen for chosen in selected_boxes):
                    continue
                selected_boxes.append(box)
    
        return selected_boxes
    
    
    def build_schedule(attention_map, num_phases=10, top_k=3, iou_threshold=0.3):
        attention_map = np.asarray(attention_map)
        if attention_map.ndim != 2 or not attention_map.size or not np.isfinite(attention_map).all():
            raise ValueError("Attention must be a nonempty finite 2D map")
        if num_phases < 1 or top_k < 1 or not 0 <= iou_threshold <= 1:
            raise ValueError("Invalid hotspot schedule parameters")
        grid_h, grid_w = attention_map.shape
        schedule = []
        for i in range(num_phases):
            ratio = (i + 1) / num_phases
            boxes = get_top_k_boxes_fixed_area(attention_map, ratio, k=top_k, iou_threshold=iou_threshold)
            for j, (xmin, xmax, ymin, ymax) in enumerate(boxes):
                top = np.clip(ymin / grid_h, 0.0, 1.0)
                left = np.clip(xmin / grid_w, 0.0, 1.0)
                height = np.clip((ymax - ymin + 1) / grid_h, 0.0, 1.0 - top)
                width = np.clip((xmax - xmin + 1) / grid_w, 0.0, 1.0 - left)
                schedule.append(
                    dict(
                        threshold=ratio,
                        box=tuple(map(float, (top, left, height, width))),
                        phase_group=i + 1,
                        sub_phase=j + 1,
                    )
                )
        if not schedule:
            raise ValueError("No hotspot candidates found")
        return schedule
    
    class RegionBiasedRandomResizedCrop(nn.Module):
        """
    RandomResizedCrop with biased sampling of hotspot region.
    """
    
        def __init__(
            self,
            size,
            scale=(0.8, 1.0),
            hotspot_rect: tuple[int, int, int, int] = None,
            hotspot_prob: float = 0.8,
            antialias=True,
        ):
            super().__init__()
            # Use standard RandomResizedCrop internally
            self.standard_cropper = transforms.RandomResizedCrop(size=size, scale=scale, antialias=antialias)
    
            if hotspot_rect is None:
                # If hotspot is not provided, always use standard crop
                self.hotspot_prob = 0.0
            else:
                self.hotspot_prob = hotspot_prob
                self.top, self.left, self.height, self.width = hotspot_rect
    
        def forward(self, img: torch.Tensor) -> torch.Tensor:
            # With p_hot probability, crop from the hotspot region
            if random.random() < self.hotspot_prob:
                # 1. Crop the hotspot region from the original image
                img_hotspot = TF.crop(img, self.top, self.left, self.height, self.width)
    
                # 2. Apply standard RandomResizedCrop to the cropped hotspot region
                #    (If the hotspot is too small to crop, an error may occur, so exception handling)
                try:
                    return self.standard_cropper(img_hotspot), "hotspot"
                except ValueError:
                    # If the hotspot is too small to crop, just resize
                    return TF.resize(img_hotspot, self.standard_cropper.size, antialias=True), "hotspot"
            else:
                # With the remaining probability, use standard RandomResizedCrop on the entire image
                return self.standard_cropper(img), "global"
    
    
    def box_to_pixels(box, height, width):
        top, left, h, w = map(float, box)
        if not all(math.isfinite(x) for x in (top, left, h, w)):
            raise ValueError('Non-finite hotspot box.')
        t, l = max(0, min(height - 1, int(top * height))), max(0, min(width - 1, int(left * width)))
        ph, pw = max(1, int(h * height)), max(1, int(w * width))
        return t, l, min(ph, height - t), min(pw, width - l)
    
    def schedule_entry(schedule, step, steps):
        if not schedule or steps < len(schedule) or not 0 <= step < steps:
            raise ValueError('Need enough steps for every actual schedule entry.')
        block = steps // len(schedule)
        index = min(step // block, len(schedule) - 1)
        return index, schedule[index]
    
    class TracedSourceCrop(nn.Module):
        """SAGA hotspot crop or FOA full-image crop, with exact random crop rectangle recorded."""
        def __init__(self, size, scale, hotspot_rect=None):
            super().__init__()
            self.hotspot_rect = hotspot_rect
            self.saga_cropper = RegionBiasedRandomResizedCrop(size=size, scale=scale,
                                                            hotspot_rect=hotspot_rect, hotspot_prob=1.0)
            self.last_rectangle = None
    
        def forward(self, image):
            # Preserve SAGA's Python-RNG draw for its hotspot branch; baseline does not draw it.
            if self.hotspot_rect is None:
                top, left, height, width = 0, 0, image.shape[-2], image.shape[-1]
            else:
                random.random()
                top, left, height, width = self.hotspot_rect
            region = TF.crop(image, top, left, height, width)
            standard = self.saga_cropper.standard_cropper
            t, l, h, w = transforms.RandomResizedCrop.get_params(region, standard.scale, standard.ratio)
            self.last_rectangle = (top + t, left + l, h, w)
            return TF.resized_crop(region, t, l, h, w, standard.size,
                                   interpolation=standard.interpolation, antialias=standard.antialias)
    
    def checkpoint_steps(steps):
        interval = max(1, steps // 10)
        return sorted({0, steps, *range(interval, steps + 1, interval)})

In [ ]:
DEBUG = os.environ.get("DEBUG", "0") == "1"
if globals().get('_NOTEBOOK_STAGE') == 'attack':
    
    STOP_WORDS = {
        "i",
        "me",
        "my",
        "myself",
        "we",
        "our",
        "ours",
        "ourselves",
        "you",
        "you're",
        "you've",
        "you'll",
        "you'd",
        "your",
        "yours",
        "yourself",
        "yourselves",
        "he",
        "him",
        "his",
        "himself",
        "she",
        "she's",
        "her",
        "hers",
        "herself",
        "it",
        "it's",
        "its",
        "itself",
        "they",
        "them",
        "their",
        "theirs",
        "themselves",
        "what",
        "which",
        "who",
        "whom",
        "this",
        "that",
        "that'll",
        "these",
        "those",
        "am",
        "is",
        "are",
        "was",
        "were",
        "be",
        "been",
        "being",
        "have",
        "has",
        "had",
        "having",
        "do",
        "does",
        "did",
        "doing",
        "a",
        "an",
        "the",
        "and",
        "but",
        "if",
        "or",
        "because",
        "as",
        "until",
        "while",
        "of",
        "at",
        "by",
        "for",
        "with",
        "about",
        "against",
        "between",
        "into",
        "through",
        "during",
        "before",
        "after",
        "above",
        "below",
        "to",
        "from",
        "up",
        "down",
        "in",
        "out",
        "on",
        "off",
        "over",
        "under",
        "again",
        "further",
        "then",
        "once",
        "here",
        "there",
        "when",
        "where",
        "why",
        "how",
        "all",
        "any",
        "both",
        "each",
        "few",
        "more",
        "most",
        "other",
        "some",
        "such",
        "no",
        "nor",
        "not",
        "only",
        "own",
        "same",
        "so",
        "than",
        "too",
        "very",
        "s",
        "t",
        "can",
        "will",
        "just",
        "don",
        "don't",
        "should",
        "should've",
        "now",
        "d",
        "ll",
        "m",
        "o",
        "re",
        "ve",
        "y",
        "ain",
        "aren",
        "aren't",
        "couldn",
        "couldn't",
        "didn",
        "didn't",
        "doesn",
        "doesn't",
        "hadn",
        "hadn't",
        "hasn",
        "hasn't",
        "haven",
        "haven't",
        "isn",
        "isn't",
        "ma",
        "mightn",
        "mightn't",
        "mustn",
        "mustn't",
        "needn",
        "needn't",
        "shan",
        "shan't",
        "shouldn",
        "shouldn't",
        "wasn",
        "wasn't",
        "weren",
        "weren't",
        "won",
        "won't",
        "wouldn",
        "wouldn't",
    }
    
    
    def clean_token(token: str) -> str:
        """Lowercase, strip, and drop punctuation to normalize tokens."""
        return token.strip().lower().translate(str.maketrans("", "", string.punctuation))
    
    
    def is_content_word(token: str) -> bool:
        """Return True if token is non-empty after cleaning and not a stop word."""
        cleaned = clean_token(token)
        return bool(cleaned) and cleaned not in STOP_WORDS
    
    
    def filter_generated_tokens(tokenizer, token_ids: Sequence[int], use_content_words: bool) -> list[int]:
        """
    Return indices of tokens to use. If use_content_words is True, keep only content
    words; otherwise return all indices. Falls back to all tokens if filtering empties.
    """
        if not use_content_words:
            return list(range(len(token_ids)))
    
        logger.debug(f"[Use content words: {use_content_words}] Only keeping content words")
        valid_indices = []
        for idx, token_id in enumerate(token_ids):
            token_str = tokenizer.decode([token_id], skip_special_tokens=False)
            if is_content_word(token_str):
                valid_indices.append(idx)
            elif DEBUG:
                logger.debug(f"Filtering out token '{token_str}' at index {idx}")
    
        if not valid_indices:
            return list(range(len(token_ids)))
    
        return valid_indices
    
    
    def attention_layout(input_ids, concept_start, concept_length, image_token_id, vision_start_id, vision_end_id, grid_shape):
        ids = input_ids.reshape(-1)
        image_positions = torch.nonzero(ids == image_token_id, as_tuple=False).flatten()
        starts = torch.nonzero(ids == vision_start_id, as_tuple=False).flatten()
        ends = torch.nonzero(ids == vision_end_id, as_tuple=False).flatten()
        if len(starts) != 1 or len(ends) != 1 or len(image_positions) != grid_shape[0] * grid_shape[1]:
            raise ValueError('Expected one image and matching image-token grid.')
        expected = torch.arange(int(starts[0]) + 1, int(ends[0]), device=ids.device)
        if not torch.equal(image_positions, expected):
            raise ValueError('Vision tokens must be contiguous and exclude marker tokens.')
        if concept_length < 1 or concept_start <= int(ends[0]) or concept_start + concept_length != len(ids):
            raise ValueError('Concept token interval must be the appended assistant text.')
        return image_positions
    
    def normalize_concept_attention(token_to_vision):
        values = token_to_vision.detach().float()
        if values.ndim != 2 or not values.numel() or not torch.isfinite(values).all() or (values < 0).any():
            raise ValueError('Invalid concept-to-vision attention weights.')
        masses = values.sum(-1)
        if (masses <= 1e-12).any():
            raise ValueError('Concept has zero image-attention mass; not substituting a map.')
        probability = (values / masses[:, None]).mean(0)
        probability = probability / probability.sum()
        minimum, maximum = probability.min(), probability.max()
        minmax = (probability - minimum) / (maximum - minimum + 1e-8)
        return probability, minmax, masses
    
    
    from transformers.models.qwen3_vl.modeling_qwen3_vl import apply_rotary_pos_emb, repeat_kv
    
    class _AttentionReady(Exception):
        """Private control flow: no later layers are needed for a salient-region map."""
    
    def selected_causal_vision_rows(module, hidden_states, position_embeddings, query_rows, vision_positions):
        """Exact eager equations for selected rows, including all-key head softmax.

    This function receives a single unpadded, uncached image/text sequence.
    Do not softmax only image keys: that would change relative head weighting.
    """
        if hidden_states.shape[0] != 1:
            raise ValueError('Attention capture requires one unpadded sequence.')
        shape = (*hidden_states.shape[:-1], -1, module.head_dim)
        q = module.q_norm(module.q_proj(hidden_states).view(shape)).transpose(1, 2)
        k = module.k_norm(module.k_proj(hidden_states).view(shape)).transpose(1, 2)
        q, k = apply_rotary_pos_emb(q, k, *position_embeddings)
        q = q.index_select(2, query_rows)
        k = repeat_kv(k, module.num_key_value_groups)
        scores = torch.matmul(q, k.transpose(2, 3)) * module.scaling
        future = torch.arange(k.shape[2], device=k.device)[None, :] > query_rows[:, None]
        scores = scores.masked_fill(future[None, None], torch.finfo(scores.dtype).min)
        weights = F.softmax(scores, dim=-1, dtype=torch.float32).to(q.dtype)
        return weights[0].index_select(2, vision_positions).float().mean(0)
    
    class ConceptAttentionExtractor:
        """SDPA prefix, selected Q/K rows at layer 29, early exit, per-image vision cache."""
        def __init__(self, resolved_revision=None):
            self.device = torch.device(ATTENTION_DEVICE)
            self.signature = self.current_signature()
            requested_revision = resolved_revision or ATTENTION_REVISION
            resolved_config = AutoConfig.from_pretrained(ATTENTION_MODEL_ID, revision=requested_revision)
            load_revision = getattr(resolved_config, '_commit_hash', None) or requested_revision
            self.processor = AutoProcessor.from_pretrained(ATTENTION_MODEL_ID, revision=load_revision)
            quantization = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16,
                llm_int8_skip_modules=['visual'])
            self.model = AutoModelForImageTextToText.from_pretrained(
                ATTENTION_MODEL_ID, revision=load_revision, torch_dtype=torch.float16,
                quantization_config=quantization, device_map={'': self.device.index},
                attn_implementation='sdpa', low_cpu_mem_usage=True, trust_remote_code=False).eval()
            self.model.requires_grad_(False)
            layers = self.model.model.language_model.layers
            if not 0 <= ATTENTION_LAYER < len(layers):
                raise ValueError('Attention layer index is unavailable.')
            self.attention_module = layers[ATTENTION_LAYER].self_attn
            self._lock = threading.RLock()
            self._prepared = None
            self.metadata = {'model_id': ATTENTION_MODEL_ID,
                'resolved_revision': getattr(self.model.config, '_commit_hash', None),
                'layer': ATTENTION_LAYER, 'dtype': 'float16', 'quantization': 'nf4_double',
                'device': str(self.device), 'input_res': ATTENTION_INPUT_RES,
                'mode': 'teacher_forced_concept_input_rows', 'attn_implementation': 'sdpa_prefix_selected_eager_rows',
                'early_exit': 'before selected self-attention value/output projection and MLP',
                'vision_cache_scope': 'same checkpoint image only', 'use_cache': False}
    
        @staticmethod
        def current_signature():
            return (ATTENTION_MODEL_ID, ATTENTION_REVISION, ATTENTION_DEVICE, ATTENTION_LAYER,
                    ATTENTION_INPUT_RES, ATTENTION_PROMPT, MAX_CONCEPT_TOKENS, CONTENT_WORDS_ONLY)
    
        @contextlib.contextmanager
        def image_context(self, image):
            # One thread owns model hooks, the processor and its mutable RoPE state.
            device_context = torch.cuda.device(self.device) if self.device.type == 'cuda' else contextlib.nullcontext()
            with self._lock, torch.inference_mode(), device_context:
                if self.current_signature() != self.signature:
                    raise RuntimeError('Attention settings changed during a run; concept protocol must stay fixed.')
                if self.model.training:
                    raise RuntimeError('Attention must run in eval mode without dropout/RNG draws.')
                if self._prepared is not None:
                    raise RuntimeError('Nested image contexts are not supported.')
                image = image.convert('RGB').resize((ATTENTION_INPUT_RES, ATTENTION_INPUT_RES), Image.Resampling.BICUBIC)
                messages = [{'role': 'user', 'content': [{'type': 'image'}, {'type': 'text', 'text': ATTENTION_PROMPT}]}]
                prompt = self.processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
                base = self.processor(text=[prompt], images=[image], do_resize=False, return_tensors='pt')
                self._prepared = {key: value.to(self.device) for key, value in base.items()}
                visual = self.model.model.visual
                original_forward = visual.forward
                had_override = 'forward' in visual.__dict__
                cache = {}
                def cached_visual(*args, **kwargs):
                    if 'features' not in cache:
                        cache['features'] = original_forward(*args, **kwargs)
                    return cache['features']
                visual.forward = cached_visual
                try:
                    yield self
                finally:
                    if had_override:
                        visual.forward = original_forward
                    else:
                        del visual.forward
                    self._prepared = None
                    cache.clear()
                    # An early exit can leave RoPE deltas set; every concept computes fresh positions.
                    self.model.model.rope_deltas = None
    
        @torch.inference_mode()
        def extract_prepared(self, concept):
            if self.current_signature() != self.signature:
                raise RuntimeError('Attention settings changed during a run; restart with a fixed configuration.')
            if self._prepared is None:
                raise RuntimeError('Use image_context before extract_prepared.')
            if not isinstance(concept, str) or not concept.strip():
                raise ValueError('Concept must be nonempty text.')
            inputs = dict(self._prepared)
            tokenizer = self.processor.tokenizer
            concept_ids = tokenizer.encode(concept, add_special_tokens=False)
            if not concept_ids or len(concept_ids) > MAX_CONCEPT_TOKENS:
                raise ValueError(f'Concept needs 1..{MAX_CONCEPT_TOKENS} tokens; no silent truncation.')
            if any(token in tokenizer.all_special_ids for token in concept_ids):
                raise ValueError('Concept must not contain special control tokens.')
            start = inputs['input_ids'].shape[1]
            suffix = torch.tensor([concept_ids], device=self.device, dtype=inputs['input_ids'].dtype)
            inputs['input_ids'] = torch.cat((inputs['input_ids'], suffix), dim=1)
            inputs['attention_mask'] = torch.ones_like(inputs['input_ids'])
            inputs.pop('position_ids', None); inputs.pop('token_type_ids', None)
            grid = inputs['image_grid_thw'][0].cpu()
            merge = self.processor.image_processor.merge_size
            if int(grid[0]) != 1 or int(grid[1]) % merge or int(grid[2]) % merge:
                raise ValueError('Expected one still-image divisible merged grid.')
            shape = (int(grid[1]) // merge, int(grid[2]) // merge)
            config = self.model.config
            vision = attention_layout(inputs['input_ids'][0], start, len(concept_ids),
                config.image_token_id, config.vision_start_token_id, config.vision_end_token_id, shape)
            offsets = filter_generated_tokens(tokenizer, concept_ids, CONTENT_WORDS_ONLY)
            queries = torch.tensor([start + offset for offset in offsets], device=self.device)
            captured = {}
            def capture(module, args, kwargs):
                if kwargs.get('past_key_values') is not None:
                    raise RuntimeError('KV cache is forbidden during attention capture.')
                hidden = kwargs.get('hidden_states', args[0] if args else None)
                position_embeddings = kwargs.get('position_embeddings')
                if hidden is None or position_embeddings is None:
                    raise RuntimeError('Unexpected Qwen self-attention signature; install pinned Transformers.')
                captured['weights'] = selected_causal_vision_rows(module, hidden, position_embeddings, queries, vision).cpu()
                raise _AttentionReady()
            handle = self.attention_module.register_forward_pre_hook(capture, with_kwargs=True)
            self.model.model.rope_deltas = None
            try:
                try:
                    self.model.model(**inputs, use_cache=False, output_attentions=False,
                                     output_hidden_states=False, return_dict=True)
                except _AttentionReady:
                    pass
            finally:
                handle.remove()
            if 'weights' not in captured:
                raise RuntimeError('Selected attention layer was not reached.')
            probability, minmax, masses = normalize_concept_attention(captured['weights'])
            return {'probability': probability.reshape(shape).numpy(), 'minmax': minmax.reshape(shape).numpy(),
                'metadata': {'concept': concept, 'token_ids': concept_ids, 'selected_offsets': offsets,
                    'selected_token_texts': [tokenizer.decode([concept_ids[i]]) for i in offsets],
                    'concept_query_rows': queries.cpu().tolist(), 'vision_positions': vision.cpu().tolist(),
                    'grid_shape': list(shape), 'mean_raw_vision_mass': float(masses.mean()),
                    'method': 'teacher_forced_concept_input_rows', 'layer': ATTENTION_LAYER,
                    'backend': 'sdpa_prefix_selected_eager_rows', 'early_exit': True}}
    
        def extract(self, image, concept):
            with self.image_context(image):
                return self.extract_prepared(concept)
    
    def ensure_attention_backend(resolved_revision=None):
        global ATTENTION_BACKEND
        if ATTENTION_BACKEND is not None and ConceptAttentionExtractor.current_signature() != ATTENTION_BACKEND.signature:
            raise RuntimeError('Qwen configuration changed; restart session before reloading.')
        if (ATTENTION_BACKEND is not None and resolved_revision is not None
                and ATTENTION_BACKEND.metadata.get('resolved_revision') != resolved_revision):
            raise RuntimeError('Cached Qwen revision differs from the resumed run; restart the session.')
        if ATTENTION_BACKEND is None:
            with preserve_rng():
                ATTENTION_BACKEND = ConceptAttentionExtractor(resolved_revision=resolved_revision)
        return ATTENTION_BACKEND

In [ ]:
# Cell 11 — Reference-preserving update order and editable hybrid orchestration.
if globals().get('_NOTEBOOK_STAGE') == 'attack':
    class RecordedRandomCrop(transforms.RandomResizedCrop):
        def forward(self, image):
            t, l, h, w = self.get_params(image, self.scale, self.ratio)
            self.last_rectangle = (t, l, h, w)
            return TF.resized_crop(image, t, l, h, w, self.size, self.interpolation, antialias=self.antialias)
    
    class ResearchSourceCrop(TracedSourceCrop):
        def forward(self, image):
            if self.hotspot_rect is not None and HOTSPOT_PROB < 1.0:
                # Explicit optional probability ablation. Default is v3.2's always-hotspot primitive.
                if random.random() >= HOTSPOT_PROB:
                    plain = RecordedRandomCrop(PAPER_INPUT_RES, scale=PAPER_CROP_SCALE)
                    result = plain(image); self.last_rectangle = plain.last_rectangle
                    return result
            return super().forward(image)
    
    def sample_rng(sample_id, purpose):
        value = hashlib.sha256(f'{SEED}:{sample_id}:{purpose}'.encode()).digest()
        return random.Random(int.from_bytes(value[:8], 'big'))
    
    def randomize_schedule(schedule, rng):
        result = []
        for entry in schedule:
            _, _, h, w = entry['box']
            result.append({**entry, 'box': (rng.uniform(0, max(0, 1 - h)), rng.uniform(0, max(0, 1 - w)), h, w)})
        return result
    
    def record_attention(models, source, current, guide, directory, step, region_mode, region_rng, backend=None):
        with models.qwen_phase():
            backend = backend or ensure_attention_backend(MODEL_REVISIONS[ATTENTION_MODEL_ID])
            results = backend.extract(pixels_to_pil(current), guide)
            if hasattr(backend, 'metadata'):
                MODEL_METADATA[ATTENTION_MODEL_ID] = dict(backend.metadata)
                if RUN_DIR is not None:
                    write_json(RUN_DIR / 'model_metadata.json', MODEL_METADATA)
        maps_dir = directory / 'maps'; maps_dir.mkdir(parents=True, exist_ok=True)
        checkpoint = directory / 'checkpoints' / f'step_{step:05d}.png'
        checkpoint.parent.mkdir(parents=True, exist_ok=True)
        pixels_to_pil(current).save(checkpoint)
        np.savez_compressed(maps_dir / f'step_{step:05d}.npz', probability=results['probability'], minmax=results['minmax'])
        meta = {**results['metadata'], 'computed_at_step': step, 'guide': guide,
                'attention_input_png_sha256': sha256_file(checkpoint), 'attack_float_pixel_sha256': tensor_pixel_hash(current),
                'source_pixel_sha256': tensor_pixel_hash(source)}
        write_json(maps_dir / f'step_{step:05d}.json', meta)
        schedule = build_schedule(results['minmax'], NUM_PHASES, HOTSPOT_TOP_K, HOTSPOT_IOU)
        if region_mode == 'random':
            schedule = randomize_schedule(schedule, region_rng)
        write_json(directory / 'schedules' / f'step_{step:05d}.json',
                   {'computed_at_step': step, 'applies_from_update': step + 1, 'schedule': schedule,
                    'absolute_step_indexing': True, 'region_mode': region_mode, 'metadata': meta})
        return schedule
    
    def attack_objective(models, foa_loss, anchors, source, target, delta, source_crop, target_crop, spec, negative_ids, amp):
        row = {}
        adv = source + delta
        zero = adv.new_zeros(())
        foa_score = mpc_value = neg_loss = zero
        if spec['foa']:
            with torch.no_grad():
                # Original FOA refreshes the target crop every step; never replaces it with fixed anchors.
                foa_loss.set_ground_truth(target_crop(target))
            if isinstance(source_crop, nn.Identity):
                features = models.foa_features(adv, amp=amp)
                foa_score = foa_loss(*features)
                row['foa_full'] = dict(foa_loss.last_record)
                cropped = adv
            else:
                # Full score still advances model-history weights before crop score, exactly as upstream.
                # Its graph does not contribute to the crop gradient and can be omitted safely.
                with torch.no_grad():
                    full = models.foa_features(adv, amp=amp)
                    foa_loss(*full)
                    row['foa_full'] = dict(foa_loss.last_record)
                del full
                cropped = source_crop(adv)
                features = models.foa_features(cropped, amp=amp)
                foa_score = foa_loss(*features)
                row['foa_crop'] = dict(foa_loss.last_record)
        else:
            cropped = source_crop(adv)
        if spec['mpc'] or negative_ids:
            z_adv, clip_features = models.mpc_features(cropped, amp=amp)
            if spec['mpc']:
                mpc_value = mpc_score(z_adv, anchors['target'], anchors['source'])
                row['mpc_similarity_source'] = float(F.cosine_similarity(z_adv, anchors['source'], dim=-1).mean().detach())
                row['mpc_similarity_target'] = float(F.cosine_similarity(z_adv, anchors['target'], dim=-1).mean().detach())
            if negative_ids:
                neg_loss = concept_negative_loss(clip_features, anchors, negative_ids)
        if spec['foa'] and not spec['mpc']:
            objective = foa_score
        elif not spec['foa']:
            objective = mpc_value
        else:
            objective = LOSS_WEIGHTS['foa'] * foa_score + LOSS_WEIGHTS['mpc'] * mpc_value - LOSS_WEIGHTS['negative'] * neg_loss
        row.update(foa_score=float(foa_score.detach()), mpc_score=float(mpc_value.detach()),
                   negative_loss=float(neg_loss.detach()), objective=float(objective.detach()),
                   crop=getattr(source_crop, 'last_rectangle', None), target_crop=getattr(target_crop, 'last_rectangle', None))
        return objective, row
    
    # v4.3.1 — Mixed-precision recovery for hybrid ONLY. Original reference baselines are unchanged.
    # Retrying must undo the source/target crop RNG draws AND stateful FOA loss history.
    def snapshot_foa_state(foa_loss):
        if foa_loss is None:
            return None
        keys = ('previous_loss_list', 'previous_loss_local_list', 'ground_truth', 'ground_truth_local')
        return {key: [x.detach().clone() for x in getattr(foa_loss, key, [])] for key in keys}
    
    def restore_foa_state(foa_loss, snapshot):
        if snapshot is not None:
            for key, saved in snapshot.items():
                setattr(foa_loss, key, [v.detach().clone() for v in saved])
    
    def hybrid_image_gradient(objective, delta, scale):
        # Full backward avoids the intermittent partial-backward GraphTask failure.
        # Models are frozen; delta is the only trainable leaf in this graph.
        if not delta.is_leaf or not delta.requires_grad:
            raise ValueError('Hybrid delta must be a trainable leaf')
        previous = delta.grad
        delta.grad = None
        try:
            with torch.autograd.set_multithreading_enabled(False):
                (objective * scale).backward()
            if delta.grad is None:
                raise RuntimeError('Objective did not produce an image gradient')
            return delta.grad.detach() / scale
        finally:
            delta.grad = previous
    
    def finite_gradient_or_error(models, foa_loss, anchors, source, target, delta, source_crop,
                                 target_crop, spec, negative_ids, requested_amp, variant, sample_id, step):
        # Mixed FP16 can overflow gradients even when scalar objective is finite.
        # No silent NaN-to-zero, no implicit gradient clipping and no skipped update.
        # Ordered attempts: requested precision/scale -> AMP without scaling -> FP32.
        attempts = [(bool(requested_amp), float(GRADIENT_SCALE) if requested_amp else 1.0)]
        if requested_amp and GRADIENT_SCALE != 1.0:
            attempts.append((True, 1.0))
        if requested_amp and HYBRID_AMP_FP32_RETRY:
            attempts.append((False, 1.0))
        saved_rng = capture_rng() if len(attempts) > 1 else None
        saved_foa = snapshot_foa_state(foa_loss) if len(attempts) > 1 else None
        errors = []
        for i, (amp_now, scale) in enumerate(attempts):
            if i:
                restore_rng(saved_rng)
                restore_foa_state(foa_loss, saved_foa)
            objective = gradient = None
            row = {}
            try:
                with saved_tensor_context():
                    objective, row = attack_objective(models, foa_loss, anchors, source, target, delta,
                                                      source_crop, target_crop, spec, negative_ids, amp_now)
                    if not bool(torch.isfinite(objective).all()):
                        raise FloatingPointError('Objective non-finite')
                    gradient = hybrid_image_gradient(objective, delta, scale)
                    if not bool(torch.isfinite(gradient).all()):
                        bad = int((~torch.isfinite(gradient)).sum().item())
                        raise FloatingPointError(f'Gradient non-finite ({bad}/{gradient.numel()} elements)')
                row.update(precision_mode='amp_fp16' if amp_now else 'fp32', gradient_scale=scale,
                           precision_retries=i, gradient_abs_max=float(gradient.detach().abs().max()))
                if errors:
                    row['precision_retry_errors'] = errors
                    event('precision', 'recovered', sample_id=sample_id, variant=variant,
                          step=step + 1, mode=row['precision_mode'], attempts=i + 1)
                return gradient, row
            except FloatingPointError as exc:
                errors.append({'precision': 'amp_fp16' if amp_now else 'fp32', 'gradient_scale': scale,
                               'exception': f'{type(exc).__name__}: {exc}',
                               'components': {key: row.get(key) for key in ('foa_score', 'mpc_score', 'negative_loss', 'objective')}})
            finally:
                del objective
                if gradient is not None and not bool(torch.isfinite(gradient).all()):
                    del gradient
        # Return to the pre-attempt state on failure (important for the following sample).
        if saved_rng is not None:
            restore_rng(saved_rng)
            restore_foa_state(foa_loss, saved_foa)
        raise FloatingPointError('Non-finite hybrid gradient after precision attempts; '
                                 f'variant={variant}, sample={sample_id}, step={step + 1}, diagnostics={errors}')
    
    def run_one(models, source_cpu, target_cpu, text_record, variant, directory, sample_id, foa_loss=None, backend=None, steps=None, progress_bar=None, sample_position=None, sample_total=None):
        spec = VARIANTS[variant]
        steps = STEPS if steps is None else steps
        directory = Path(directory); directory.mkdir(parents=True, exist_ok=True)
        # Author MPC initializes on the original CPU image before moving source/target to CUDA.
        mpc_only = spec['mpc'] and not spec['foa']
        delta = initialize_delta(source_cpu, mpc_only)
        source, target = source_cpu.to(ATTACK_DEVICE), target_cpu.to(ATTACK_DEVICE)
        anchors = None
        if spec['mpc'] or spec['negatives'] != 'none':
            with preserve_rng():
                anchor_record = text_record if spec['negatives'] != 'none' else {**text_record, 'negatives': []}
                anchors = models.anchors(source, target, anchor_record)
        if spec['foa'] and foa_loss is None:
            foa_loss = models.new_foa_loss()
        crop_enabled = USE_SOURCE_CROP if spec['foa'] else True
        source_crop = RecordedRandomCrop(INPUT_RES, scale=PAPER_CROP_SCALE) if crop_enabled else nn.Identity()
        target_crop = RecordedRandomCrop(INPUT_RES, scale=PAPER_CROP_SCALE) if USE_TARGET_CROP else nn.Identity()
        amp = HYBRID_AMP and variant.startswith('hybrid_')
        decay = MPC_MOMENTUM if mpc_only else HYBRID_MOMENTUM if variant.startswith('hybrid_') else 0.0
        momentum = torch.zeros_like(delta)
        negative_rng, region_rng = sample_rng(sample_id, 'negative'), sample_rng(sample_id, 'region')
        negative_ids, scores = [], []
        schedule, map_step = None, None
        trace = directory / 'steps.jsonl'
        trace.write_text('', encoding='utf-8')
        if str(ATTACK_DEVICE).startswith('cuda'):
            torch.cuda.reset_peak_memory_stats(ATTACK_DEVICE)
        if torch.cuda.is_available() and torch.device(ATTENTION_DEVICE).index is not None and torch.device(ATTENTION_DEVICE).index < torch.cuda.device_count():
            torch.cuda.reset_peak_memory_stats(ATTENTION_DEVICE)
        started = time.perf_counter()
        if progress_bar is not None:
            progress_bar.set_postfix_str(
                f'sample={sample_position}/{sample_total} id={sample_id} step=0/{steps} phase=attack', refresh=True)
        for step in range(steps):
            step_started = time.perf_counter()
            if spec['region'] != 'none' and (step == 0 or (spec['region'] in {'dynamic', 'random'} and step % REFRESH_EVERY == 0)):
                if progress_bar is not None:
                    progress_bar.set_postfix_str(
                        f'sample={sample_position}/{sample_total} id={sample_id} step={step}/{steps} phase=attention',
                        refresh=True)
                schedule = record_attention(models, source, source + delta.detach(), text_record['source'], directory, step, spec['region'], region_rng, backend)
                map_step = step
            hotspot, schedule_index, entry = None, None, {}
            if schedule is not None:
                schedule_index, entry = schedule_entry(schedule, step, steps)
                hotspot = box_to_pixels(entry['box'], *source.shape[-2:])
                source_crop = ResearchSourceCrop(INPUT_RES, PAPER_CROP_SCALE, hotspot)
            if spec['negatives'] != 'none' and step % MINE_EVERY == 0:
                with preserve_rng(), torch.no_grad():
                    candidate_scores = negative_scores(models.clip_images(source + delta.detach()), anchors['negative_text'])
                negative_ids = select_negatives(candidate_scores, K_NEG, spec['negatives'], negative_rng)
                scores = candidate_scores.cpu().tolist()
            gradient, row = finite_gradient_or_error(models, foa_loss, anchors, source, target, delta,
                source_crop, target_crop, spec, negative_ids, amp, variant, sample_id, step)
            delta, momentum = projected_ascent(delta, gradient, momentum, decay)
            del gradient
            row.update(step=step + 1, linf_internal_255=float(delta.detach().abs().max()),
                       negative_ids=negative_ids, negative_scores=scores, attention_step=map_step,
                       hotspot=hotspot, schedule_index=schedule_index, phase_group=entry.get('phase_group'),
                       sub_phase=entry.get('sub_phase'), step_seconds=time.perf_counter() - step_started,
                       elapsed_seconds=time.perf_counter() - started)
            append_jsonl(trace, row)
            if (step + 1) % SAVE_EVERY == 0 or step + 1 == steps:
                checkpoint = directory / 'checkpoints' / f'step_{step + 1:05d}.png'
                checkpoint.parent.mkdir(parents=True, exist_ok=True)
                pixels_to_pil(source + delta.detach()).save(checkpoint)
            if progress_bar is not None:
                progress_bar.set_postfix_str(
                    f'sample={sample_position}/{sample_total} id={sample_id} step={step + 1}/{steps} phase=attack',
                    refresh=False)
                progress_bar.update(1)
            if (step + 1) % LOG_EVERY == 0 or step + 1 == steps:
                event('attack', 'progress', sample_id=sample_id, variant=variant, step=step + 1,
                      objective=round(row['objective'], 5), negatives=negative_ids,
                      eta_seconds=round(row['elapsed_seconds'] / (step + 1) * (steps - step - 1), 1))
        if spec['region'] != 'none':
            if progress_bar is not None:
                progress_bar.set_postfix_str(
                    f'sample={sample_position}/{sample_total} id={sample_id} step={steps}/{steps} phase=final_attention',
                    refresh=True)
            # Final diagnostic map is saved but never fed back into the completed optimization.
            record_attention(models, source, source + delta.detach(), text_record['source'], directory, steps, spec['region'], region_rng, backend)
        result = ((source + delta.detach()) / 255.0).clamp(0, 1).cpu()
        stats = {'elapsed_seconds': time.perf_counter() - started, 'steps': steps, 'amp': amp,
                 'hybrid_amp_fp32_retry': HYBRID_AMP_FP32_RETRY, 'hybrid_configured_gradient_scale': GRADIENT_SCALE,
                 'momentum': decay, 'max_internal_delta_255': float(delta.detach().abs().max()),
                 'peak_cuda_bytes': torch.cuda.max_memory_allocated(ATTACK_DEVICE) if str(ATTACK_DEVICE).startswith('cuda') else 0,
                 'peak_cuda_bytes_by_device': {f'cuda:{i}': torch.cuda.max_memory_allocated(i) for i in range(torch.cuda.device_count())}}
        write_json(directory / 'attack_stats.json', stats)
        return result, stats

In [ ]:
# Cell 12 — Meaningful checks, no pretrained weights or remote model calls.
if globals().get('_NOTEBOOK_STAGE') == 'attack':
    @contextlib.contextmanager
    def check_scratch_directory(prefix):
        # Path.mkdir's normal inherited ACL works in managed Windows workspaces too.
        import uuid
        parent = Path.cwd().resolve()
        folder = parent / (prefix + uuid.uuid4().hex)
        folder.mkdir()
        try:
            yield folder
        finally:
            if folder.resolve().parent != parent:
                raise RuntimeError('Refusing cleanup outside the generated check directory.')
            shutil.rmtree(folder)
    
    @contextlib.contextmanager
    def override_globals(**values):
        old = {key: globals()[key] for key in values}
        globals().update(values)
        try:
            yield
        finally:
            globals().update(old)
    
    class TinyFeatureExtractor(nn.Module):
        def __init__(self, gain=1.0):
            super().__init__(); self.gain = gain
        def global_local_features(self, x):
            patches = F.adaptive_avg_pool2d(x / 255, (4, 4)).flatten(2).transpose(1, 2)
            patches = torch.cat([patches, patches.square() * self.gain], dim=-1)
            return F.normalize(patches.mean(1), dim=1), F.normalize(patches, dim=1)
        def forward(self, x):
            return F.normalize(self.global_local_features(x)[0], dim=-1)
    
    class TinyModels:
        def __init__(self, device='cpu'):
            self.foa_models = [TinyFeatureExtractor(0.8).to(device), TinyFeatureExtractor(1.4).to(device)]
            self.foa_extractor = EnsembleFeatureExtractor_ot(self.foa_models, cluster_number=3)
            self.device = torch.device(device)
        def new_foa_loss(self):
            return TracedFOALoss(self.foa_models, cluster_number=3)
        def foa_features(self, x, amp=False):
            return self.foa_extractor(x)
        def clip_images(self, x, amp=False):
            return [model(x) for model in self.foa_models]
        def mpc_features(self, x, amp=False):
            features = self.clip_images(x)
            return torch.cat(features, 1), features
        @torch.no_grad()
        def anchors(self, source, target, record):
            s, t = self.clip_images(source), self.clip_images(target)
            texts = [F.normalize(torch.arange(1, 7, device=source.device).float()[None], dim=-1) for _ in s]
            src, tgt = mpc_clip_anchors(s, t, texts, texts)
            bank = [torch.cat([text.flip(-1), -text], dim=0) for text in texts]
            return {'source': src, 'target': tgt, 'target_text': texts, 'negative_text': bank}
        @contextlib.contextmanager
        def qwen_phase(self):
            with preserve_rng():
                yield
    
    class TinyAttentionBackend:
        def extract(self, image, concept):
            pixels = torch.from_numpy(np.asarray(image).copy()).float().permute(2, 0, 1)[None]
            values = F.adaptive_avg_pool2d(pixels.mean(1, keepdim=True), (4, 4))[0, 0].numpy() + 1
            return {'probability': values / values.sum(), 'minmax': (values - values.min()) / (values.max() - values.min() + 1e-8),
                    'metadata': {'synthetic': True, 'concept': concept, 'grid_shape': [4, 4]}}
    
    def check_foa_parity():
        original_ns = compile_author('foa', 'surrogates/FeatureExtractors/Base.py',
                                     ['EnsembleFeatureExtractor_ot', 'EnsembleFeatureLoss_OT_foa_attack'])
        original_loss = original_ns['EnsembleFeatureLoss_OT_foa_attack']
        models = TinyModels()
        target = torch.rand(1, 3, 16, 16) * 180 + 20
        actual_loss = models.new_foa_loss()
        expected_loss = original_loss(models.foa_models, cluster_number=3)
        for iteration in range(3):
            image = (torch.rand_like(target) * 160 + 30).requires_grad_(True)
            seed_all(50 + iteration)
            actual_loss.set_ground_truth(target)
            actual = actual_loss(*models.foa_features(image))
            grad_actual = torch.autograd.grad(actual, image)[0]
            assert float(grad_actual.abs().sum()) > 0
            seed_all(50 + iteration)
            expected_loss.set_ground_truth(target)
            expected = expected_loss(*models.foa_features(image))
            grad_expected = torch.autograd.grad(expected, image)[0]
            torch.testing.assert_close(actual, expected, rtol=0, atol=0)
            torch.testing.assert_close(grad_actual, grad_expected, rtol=0, atol=0)
            assert all(torch.equal(a, b) for a, b in zip(actual_loss.previous_loss_list, expected_loss.previous_loss_list))
        original = upstream_fgsm({**globals(), 'log_metrics': lambda *a, **kw: None, 'tqdm': lambda seq, **kw: seq})
        with check_scratch_directory('v43_foa_') as folder, override_globals(
            ATTACK_DEVICE='cpu', INPUT_RES=16, PAPER_INPUT_RES=16, HYBRID_AMP=False,
            SAVED_TENSORS_ON_CPU=False, LOG_EVERY=999, SAVE_EVERY=999,
        ):
            for cropped in (False, True):
                actual_loss, expected_loss = models.new_foa_loss(), models.new_foa_loss()
                cfg = SimpleNamespace(optim=SimpleNamespace(steps=4, alpha=ALPHA_255, epsilon=EPSILON_255),
                                      model=SimpleNamespace(use_source_crop=cropped))
                # Keep the same loss objects for two samples to test cross-sample history.
                source = torch.linspace(20, 210, 3 * 16 * 16).reshape(1, 3, 16, 16)
                seed_all(123)
                expected_images, expected_histories = [], []
                for sample in range(2):
                    crop = transforms.RandomResizedCrop(16, scale=PAPER_CROP_SCALE) if cropped else nn.Identity()
                    expected_images.append(original(cfg, models.foa_extractor, expected_loss, crop,
                        transforms.RandomResizedCrop(16, scale=PAPER_CROP_SCALE), sample, source + sample, target))
                    expected_histories.append([v.clone() for v in expected_loss.previous_loss_list])
                seed_all(123)
                with override_globals(USE_SOURCE_CROP=cropped, USE_TARGET_CROP=True):
                    for sample in range(2):
                        actual, _ = run_one(models, source + sample, target, {}, 'foa_reference',
                            Path(folder) / f'{cropped}_{sample}', str(sample), actual_loss, steps=4)
                        torch.testing.assert_close(actual, expected_images[sample], rtol=0, atol=0)
                        assert all(torch.equal(a, b) for a, b in zip(actual_loss.previous_loss_list, expected_histories[sample]))
                # A JSON roundtrip must preserve history/RNG for the next sample.
                checkpoint = json.loads(json.dumps(state_checkpoint(actual_loss)))
                before = json_hash(capture_rng()); actual_loss.previous_loss_list.clear()
                random.random(); torch.rand(3); np.random.rand()
                restore_checkpoint(checkpoint, actual_loss)
                assert json_hash(capture_rng()) == before and len(actual_loss.previous_loss_list) == 2
    
    def check_mpc_parity():
        ns = dict(globals())
        ns.update(args=SimpleNamespace(lam=0.6, device='cpu'), device='cpu')
        source = torch.randn(1, 3, 4, 4); target = -source
        s_images = [F.normalize(torch.randn(1, d), dim=-1) for d in (5, 7, 9)]
        t_images = [F.normalize(torch.randn_like(x), dim=-1) for x in s_images]
        s_texts = [F.normalize(torch.randn_like(x), dim=-1) for x in s_images]
        t_texts = [F.normalize(torch.randn_like(x), dim=-1) for x in s_images]
        class MockClips:
            def __call__(self, image):
                return dict(enumerate(s_images if image is source else t_images))
            def get_text_features(self, text):
                return dict(enumerate(s_texts if text == 'source' else t_texts))
        intern_source = F.normalize(torch.randn(1, 4, 6), dim=1)
        intern_target = F.normalize(torch.randn_like(intern_source), dim=1)
        dino_source, dino_target = torch.randn(1, 8), torch.randn(1, 8)
        ns.update(img_src=source, img_tgt=target, src_text='source', tar_text='target', clip_models=MockClips(),
                  internvl_model=lambda x: intern_source if x is source else intern_target,
                  dino_model=lambda x: dino_source if x is source else dino_target)
        run = next(n for n in ast.parse(UPSTREAM['mpc']['sources']['generate_adversarial_examples_MPCAttack.py']).body if isinstance(n, ast.FunctionDef) and n.name == 'run_attack')
        loop = next(n for n in run.body if isinstance(n, ast.For))
        anchor_block = next(n for n in loop.body if isinstance(n, ast.With))
        exec(compile(ast.Module(body=[anchor_block], type_ignores=[]), '<mpc-author-anchors>', 'exec'), ns)
        ours_src, ours_tgt = mpc_clip_anchors(s_images, t_images, s_texts, t_texts)
        torch.testing.assert_close(ours_src, ns['clip_src'], rtol=0, atol=0)
        torch.testing.assert_close(ours_tgt, ns['clip_tgt'], rtol=0, atol=0)
        for ours, intern, dino, expected in [(ours_src, intern_source, dino_source, ns['z_src']), (ours_tgt, intern_target, dino_target, ns['z_tgt'])]:
            torch.testing.assert_close(torch.cat([ours, intern.mean(1).float(), dino], 1), expected, rtol=0, atol=0)
        for omega in (1.0, 2.0):
            adv = torch.randn_like(ns['z_tgt']).requires_grad_(True)
            ours = mpc_score(adv, ns['z_tgt'], ns['z_src'], 0.2, omega)
            ref = AUTHOR_MPC_SCORE(adv, ns['z_tgt'], ns['z_src'], temperature=0.2, omega=omega)
            torch.testing.assert_close(ours, ref, rtol=0, atol=0)
            torch.testing.assert_close(torch.autograd.grad(ours, adv)[0], torch.autograd.grad(ref, adv)[0], rtol=0, atol=0)
        # Original delta initialization and original loop update statements.
        init = [n for n in loop.body if isinstance(n, (ast.Assign, ast.Expr)) and
                ((isinstance(n, ast.Assign) and any(isinstance(t, ast.Name) and t.id == 'delta' for t in n.targets)) or
                 (isinstance(n, ast.Expr) and isinstance(n.value, ast.Call) and isinstance(n.value.func, ast.Attribute) and n.value.func.attr == 'requires_grad_'))]
        ns.update(image_org=source)
        with override_globals(ATTACK_DEVICE='cpu'):
            seed_all(101); ours = initialize_delta(source, True)
            seed_all(101); exec(compile(ast.Module(body=init, type_ignores=[]), '<mpc-author-init>', 'exec'), ns)
            torch.testing.assert_close(ours, ns['delta'], rtol=0, atol=0)
        attack_loop = next(n for n in loop.body if isinstance(n, ast.For))
        updates = [n for n in attack_loop.body if isinstance(n, ast.Assign) and
                   any((isinstance(t, ast.Name) and t.id == 'momentum') or
                       (isinstance(t, ast.Attribute) and isinstance(t.value, ast.Name) and t.value.id == 'delta') for t in n.targets)]
        ns.update(momentum=torch.zeros_like(ours), args=SimpleNamespace(alpha=ALPHA_255, epsilon=EPSILON_255))
        ours_momentum = torch.zeros_like(ours)
        for _ in range(5):
            gradient = torch.randn_like(ours); ns['grad'] = gradient
            exec(compile(ast.Module(body=updates, type_ignores=[]), '<mpc-author-update>', 'exec'), ns)
            ours, ours_momentum = projected_ascent(ours, gradient, ours_momentum, 0.9)
            torch.testing.assert_close(ours, ns['delta'], rtol=0, atol=0)
            torch.testing.assert_close(ours_momentum, ns['momentum'], rtol=0, atol=0)
    
    def check_negatives_and_regions():
        p = torch.tensor([0.1], requires_grad=True)
        n = torch.tensor([[0.3, 0.2]], requires_grad=True)
        for kind in ('infonce', 'margin', 'hard_margin', 'repulsion'):
            loss = negative_loss_from_scores(p, n, kind)
            gp, gn = torch.autograd.grad(loss, (p, n), allow_unused=True)
            assert gp is None or bool((gp <= 0).all())
            assert bool((gn >= 0).all()) and float(gn.sum()) > 0
        assert torch.isfinite(negative_loss_from_scores(torch.tensor([-1000.]), torch.tensor([[1000.]]), 'infonce', tau=0.01))
        assert select_negatives(torch.tensor([0.2, 0.7, 0.7]), 2, 'hard', random.Random(1)) == [1, 2]
        bank = [
            {'edit_type': 'attribute', 'attribute': 'object_category', 'edits': [{'span': 'cat', 'replacement': 'dog'}], 'caption': 'A dog sits.'},
            {'edit_type': 'attribute', 'attribute': 'object_category', 'edits': [{'span': 'cat', 'replacement': 'dog'}], 'caption': 'A dog sits.'},
            {'edit_type': 'attribute', 'attribute': 'object_category', 'edits': [{'span': 'cat', 'replacement': 'bird'}], 'caption': 'A bird flies.'},
        ]
        assert len(validate_negative_candidates(bank, 'A cat sits.')) == 1
        assert len(validate_negative_candidates(bank, 'A cat and cat sit.')) == 0
        assert parse_json_response('```json\n{"negatives": []}\n```') == {'negatives': []}
        values = np.arange(64, dtype=np.float32).reshape(8, 8) + 1
        schedule = build_schedule(values, 10, 3, 0.3)
        phases = [schedule_entry(schedule, step, 300)[1]['phase_group'] for step in range(300)]
        assert phases == sorted(phases) and phases[0] == 1 and phases[-1] == 10
        refreshed = build_schedule(values[::-1].copy(), 10, 3, 0.3)
        assert schedule_entry(refreshed, 150, 300)[0] > 0  # Refresh must not reset absolute progress.
        randomized = randomize_schedule(schedule, random.Random(1))
        assert all(a['box'][2:] == b['box'][2:] for a, b in zip(schedule, randomized))
        seed_all(13); before = json_hash(capture_rng())
        with preserve_rng():
            random.random(); np.random.rand(); torch.rand(3)
        assert before == json_hash(capture_rng())
        nr, rr = sample_rng('x', 'negative'), sample_rng('x', 'region')
        nr.sample(range(10), 2)
        assert rr.random() == sample_rng('x', 'region').random()
        # Strict visual layout: markers excluded, merged grid exactly matches image-token count.
        positions = attention_layout(torch.tensor([1, 8, 9, 9, 9, 9, 10, 2, 3]), 7, 2, 9, 8, 10, (2, 2))
        assert positions.tolist() == [2, 3, 4, 5]
        try:
            attention_layout(torch.tensor([1, 8, 9, 10, 2]), 4, 1, 9, 8, 10, (2, 2))
        except ValueError:
            pass
        else:
            raise AssertionError('Grid mismatch was accepted.')
    
    def check_qwen_attention_rows():
        from transformers.models.qwen3_vl.configuration_qwen3_vl import Qwen3VLTextConfig
        from transformers.models.qwen3_vl.modeling_qwen3_vl import Qwen3VLTextAttention, Qwen3VLTextRotaryEmbedding
        config = Qwen3VLTextConfig(hidden_size=32, intermediate_size=64, num_hidden_layers=1,
                                 num_attention_heads=4, num_key_value_heads=2, head_dim=8,
                                 rope_scaling={'rope_type': 'default', 'mrope_section': [1, 1, 2]})
        config._attn_implementation = 'eager'
        module = Qwen3VLTextAttention(config, layer_idx=0).eval()
        hidden = torch.randn(1, 9, 32)
        positions = torch.arange(9)[None, None].expand(3, 1, -1)
        rotary = Qwen3VLTextRotaryEmbedding(config=config)
        embeddings = rotary(hidden, positions)
        mask = torch.full((1, 1, 9, 9), torch.finfo(hidden.dtype).min).triu(1)
        with torch.no_grad():
            _, weights = module(hidden_states=hidden, position_embeddings=embeddings, attention_mask=mask)
            queries, vision = torch.tensor([3, 6, 8]), torch.tensor([1, 2, 4, 5])
            selected = selected_causal_vision_rows(module, hidden, embeddings, queries, vision)
            expected = weights[0].index_select(1, queries).index_select(2, vision).float().mean(0)
        torch.testing.assert_close(selected, expected, rtol=1e-5, atol=1e-7)
        assert float(selected[0, 2:].abs().sum()) == 0.0  # Vision keys later than query are masked.
        probability, _, masses = normalize_concept_attention(selected)
        assert abs(float(probability.sum()) - 1) < 1e-6 and bool((masses > 0).all())
    
    def check_synthetic_gpu():
        if not torch.cuda.is_available():
            return {'status': 'skipped', 'reason': 'CUDA unavailable'}
        device = 'cuda:0'
        with check_scratch_directory('v43_cuda_') as folder, override_globals(
            ATTACK_DEVICE=device, INPUT_RES=16, PAPER_INPUT_RES=16, HYBRID_AMP=False,
            NUM_PHASES=2, HOTSPOT_TOP_K=1, REFRESH_EVERY=2, MINE_EVERY=2,
            LOG_EVERY=999, SAVE_EVERY=999,
        ):
            models = TinyModels(device)
            source, target = (torch.rand(1, 3, 16, 16) * 180 + 20).round(), (torch.rand(1, 3, 16, 16) * 180 + 20).round()
            pair = {'index': 0, 'sample_id': '00000', 'source': 'synthetic', 'target': 'synthetic'}
            for variant in ('foa_reference', 'hybrid_dynamic'):
                output, _ = run_one(models, source, target, {'source': 'A source object.', 'target': 'A target object.'},
                    variant, Path(folder) / variant / 'samples/00000', '00000', backend=TinyAttentionBackend(), steps=4)
                row = save_result(Path(folder) / variant, pair, source.round(), target.round(), output)
                assert row['saved_linf_255'] <= EPSILON_255
            del models
            cuda_cleanup()
        return {'status': 'passed', 'device': torch.cuda.get_device_name(0), 'scope': 'synthetic encoders, no pretrained-model claim'}
    
    def check_hybrid_nonfinite_recovery():
        # Synthetic regression test for finite-gradient path, AMP overflow fallback and
        # exact replay of stochastic crop + stateful FOA history. Real T4 still required.
        class State:
            def __init__(self):
                self.previous_loss_list = [torch.tensor(0.25)]
                self.previous_loss_local_list = []
                self.ground_truth = []
                self.ground_truth_local = []
        original = globals()['attack_objective']
        state = State()
        tensor = torch.tensor([0.25], requires_grad=True)
        history = []
        class NaNBackward(torch.autograd.Function):
            @staticmethod
            def forward(ctx, value):
                return value.clone()
            @staticmethod
            def backward(ctx, gradient):
                return torch.full_like(gradient, float('nan'))
        def synthetic_objective(models, loss, anchors, source, target, delta, source_crop, target_crop, spec, negative_ids, amp):
            rng_draw = random.random()
            history.append((amp, rng_draw))
            loss.previous_loss_list = [loss.previous_loss_list[0] + 1]
            # simulates a finite objective whose FP16-style backward is nonfinite.
            value = NaNBackward.apply(delta.sum()) if amp else delta.sum()
            return value, {'foa_score': 1.0, 'mpc_score': 2.0, 'negative_loss': 0.1, 'objective': 1.0}
        globals()['attack_objective'] = synthetic_objective
        try:
            with override_globals(GRADIENT_SCALE=1.0, HYBRID_AMP_FP32_RETRY=True, SAVED_TENSORS_ON_CPU=False):
                seed_all(123); before = capture_rng()
                grad, row = finite_gradient_or_error(None, state, None, None, None, tensor, None, None,
                    {'foa': True}, [], True, 'hybrid_dynamic', 'test', 0)
                assert row['precision_mode'] == 'fp32' and row['precision_retries'] == 1
                torch.testing.assert_close(grad, torch.ones_like(tensor))
                assert len(history) == 2 and history[0][1] == history[1][1]
                assert float(state.previous_loss_list[0]) == 1.25  # one accepted step, not two
                restore_rng(before)
                history.clear()
                state.previous_loss_list = [torch.tensor(0.25)]
                grad, row = finite_gradient_or_error(None, state, None, None, None, tensor, None, None,
                    {'foa': True}, [], False, 'hybrid_dynamic', 'test', 0)
                assert row['precision_mode'] == 'fp32' and row['precision_retries'] == 0
        finally:
            globals()['attack_objective'] = original
    
    def check_full_backward_gradient():
        x = torch.tensor([0.2, -0.4, 0.7], requires_grad=True)
        def objective(z):
            return (z.sin() * z.square()).sum() + (z * 3).sum() + z.cos().sum()
        expected = torch.autograd.grad(objective(x), x)[0]
        previous = torch.full_like(x, 99)
        x.grad = previous
        actual = hybrid_image_gradient(objective(x), x, 1024.0)
        torch.testing.assert_close(actual, expected)
        assert x.grad is previous
    
    def run_self_checks():
        report = {'transformers': transformers.__version__, 'torch': torch.__version__, 'checks': {}}
        for name, fn in [('embedded_source_hashes', verify_embedded_sources), ('foa_loss_gradient_loop_history_resume', check_foa_parity),
                         ('mpc_anchors_score_gradient_init_momentum', check_mpc_parity), ('negative_losses_selection_regions_rng', check_negatives_and_regions),
                         ('qwen_selected_rows_eager_causality', check_qwen_attention_rows),
                         ('hybrid_nonfinite_precision_replay', check_hybrid_nonfinite_recovery),
                         ('full_backward_gradient_parity', check_full_backward_gradient)]:
            with preserve_rng():
                fn()
            report['checks'][name] = 'passed'
        if GPU_SMOKE:
            with preserve_rng():
                report['gpu_smoke'] = check_synthetic_gpu()
        print(json.dumps(report, indent=2))
        return report
    
    CHECK_REPORT = run_self_checks() if RUN_SELF_CHECKS else {'checks': {}, 'status': 'disabled'}

In [ ]:
# Cell 13 — Preflight, immutable run identity, per-variant manifests and boundary resume.
if globals().get('_NOTEBOOK_STAGE') == 'attack':
    CONFIG_KEYS = [
        'NUM_SAMPLES', 'SAMPLE_START', 'RUN_VARIANTS', 'SEED', 'BACKBONES', 'INPUT_RES', 'STEPS',
        'EPSILON_255', 'ALPHA_255', 'CLUSTER_NUMBER', 'PAPER_CROP_SCALE', 'USE_SOURCE_CROP', 'USE_TARGET_CROP',
        'FUSION_LAMBDA', 'TEMPERATURE', 'POSITIVE_OMEGA', 'MPC_MOMENTUM', 'HYBRID_MOMENTUM',
        'LOSS_WEIGHTS', 'NEGATIVE_LOSS', 'NEGATIVE_TEMPERATURE', 'NEGATIVE_MARGIN', 'K_NEG', 'MAX_NEGATIVES',
        'MINE_EVERY', 'REFRESH_EVERY', 'NUM_PHASES', 'HOTSPOT_TOP_K', 'HOTSPOT_IOU', 'HOTSPOT_PROB',
        'ATTENTION_MODEL_ID', 'ATTENTION_REVISION', 'ATTENTION_LAYER', 'ATTENTION_INPUT_RES',
        'ATTENTION_PROMPT', 'MAX_CONCEPT_TOKENS', 'CONTENT_WORDS_ONLY', 'CAPTION_MODEL_ID', 'MODEL_REVISION',
        'TEXT_RETRIES', 'CONCEPT_MAX_NEW_TOKENS', 'MANUAL_TEXT_OVERRIDES', 'HYBRID_AMP', 'GRADIENT_SCALE',
        'HYBRID_AMP_FP32_RETRY', 'GRADIENT_ENGINE',
        'SAVED_TENSORS_ON_CPU', 'OFFLOAD_CLIP_DURING_QWEN', 'ATTACK_DEVICE', 'ATTENTION_DEVICE', 'CAPTION_DEVICE',
        'LOG_EVERY', 'SAVE_EVERY', 'NEGATIVE_POLICY_VERSION', 'CF_PROMPT_SHA256',
        'CF_NEGATIVE_CANDIDATES', 'NEGATIVE_GENERATION_BATCH_SIZE',
    ]
    
    def run_identity(pairs):
        return {'configuration': {key: globals()[key] for key in CONFIG_KEYS}, 'variants': VARIANTS,
                'pairs': pairs, 'code_hashes': runtime_code_hashes(),
                'upstream': {k: {'repo': v['repo'], 'commit': v['commit'], 'sha256': v['sha256']} for k, v in PROVENANCE.items()},
                'packages': {name: package_metadata.version(name) for name in ('torch', 'torchvision', 'transformers', 'tokenizers', 'huggingface-hub', 'accelerate')}}
    
    def verify_devices_and_runtime():
        if transformers.__version__ != '4.57.1':
            raise RuntimeError('Attack runtime must use transformers==4.57.1. Enable installation in a fresh kernel.')
        if torch.get_default_dtype() != torch.float32:
            raise RuntimeError('References expect default torch.float32.')
        devices = [ATTACK_DEVICE]
        if any(VARIANTS[v]['mpc'] for v in RUN_VARIANTS):
            devices.append(CAPTION_DEVICE)
        if any(VARIANTS[v]['negatives'] != 'none' or VARIANTS[v]['region'] != 'none' for v in RUN_VARIANTS):
            devices.append(ATTENTION_DEVICE)
        for value in devices:
            d = torch.device(value)
            if d.type != 'cuda' or not torch.cuda.is_available() or (d.index or 0) >= torch.cuda.device_count():
                raise RuntimeError(f'Requested {value} is unavailable. Default requires Kaggle T4 x2; edit device placement explicitly.')
        if MODEL_STORE:
            raise RuntimeError('CLIP weights from a previous execution remain. Restart kernel for a clean seeded run.')
    
    def rebuild_manifest(directory, pairs, fingerprint):
        rows = []
        for pair in pairs:
            complete = directory / 'samples' / pair['sample_id'] / 'complete.json'
            if not complete.is_file():
                continue
            row = json.loads(complete.read_text(encoding='utf-8'))
            if row['run_fingerprint'] != fingerprint or not complete_record_ok(directory, row):
                raise RuntimeError(f'Completed sample changed: {complete}')
            rows.append(row)
        manifest = directory / 'manifest.jsonl'
        pending = manifest.with_suffix('.jsonl.tmp')
        pending.write_text(''.join(json.dumps(row, ensure_ascii=False, allow_nan=False) + '\n' for row in rows), encoding='utf-8')
        pending.replace(manifest)
        return rows
    
    def run_experiment():
        global RUN_DIR, MODEL_REVISIONS, ATTENTION_BACKEND
        validate_configuration()
        verify_devices_and_runtime()
        pairs = prepare_pairs()
        identity = run_identity(pairs)
        if RESUME_RUN_DIR is not None:
            RUN_DIR = Path(RESUME_RUN_DIR)
            old = json.loads((RUN_DIR / 'run_config.json').read_text(encoding='utf-8'))
            if old['identity_sha256'] != json_hash(identity):
                raise ValueError('Resume identity mismatch (config/code/inputs/packages); use a new output run.')
            MODEL_REVISIONS = resolve_revisions(required_model_ids(), old['model_revisions'])
            fingerprint = old['run_fingerprint']
        else:
            MODEL_REVISIONS = resolve_revisions(required_model_ids())
            fingerprint = json_hash({'identity': identity, 'model_revisions': MODEL_REVISIONS})
            stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
            RUN_DIR = OUTPUT_ROOT / f'v433_cfvlm_{stamp}_{fingerprint[:10]}'
            RUN_DIR.mkdir(parents=True, exist_ok=False)
            write_json(RUN_DIR / 'run_config.json', {'identity': identity, 'identity_sha256': json_hash(identity),
                'model_revisions': MODEL_REVISIONS, 'run_fingerprint': fingerprint,
                'short_run_diagnostic': STEPS != 300, 'reference_precision': 'upstream_fp32_clip_bf16_internvl',
                'hybrid_precision': 'fp32_default_amp_fp32_recoverable', 'version': '4.3.3-h1-cfvlm'})
            write_json(RUN_DIR / 'pairs.json', pairs)
        write_json(RUN_DIR / 'self_checks.json', CHECK_REPORT)
        event('run', 'start', pairs=len(pairs), variants=RUN_VARIANTS, directory=str(RUN_DIR))
        with preserve_rng():
            texts = prepare_text_records(pairs, include_negatives=False)
        if ATTENTION_BACKEND is not None:
            MODEL_METADATA[ATTENTION_MODEL_ID] = dict(ATTENTION_BACKEND.metadata)
        write_json(RUN_DIR / 'model_metadata.json', MODEL_METADATA)
        statuses = {}
        for variant in RUN_VARIANTS:
            spec = VARIANTS[variant]
            if spec['negatives'] != 'none':
                with preserve_rng():
                    texts = prepare_text_records(pairs, include_negatives=True)
            directory = RUN_DIR / variant; directory.mkdir(parents=True, exist_ok=True)
            # Prefix completion is required because FOA carries weights and all references carry RNG.
            seen_gap = False
            for pair in pairs:
                done = directory / 'samples' / pair['sample_id'] / 'complete.json'
                if not done.is_file():
                    seen_gap = True
                elif seen_gap:
                    raise RuntimeError(f'Non-prefix completion in {variant}; cannot restore reference history safely.')
            seed_all(SEED)
            models, foa_loss = None, None
            completed = 0
            sid = None
            progress_bar = None
            try:
                models = AttackModels(full_mpc=spec['full_mpc'])
                foa_loss = models.new_foa_loss() if spec['foa'] else None
                write_json(RUN_DIR / 'model_metadata.json', MODEL_METADATA)
                if SHOW_ATTACK_PROGRESS:
                    # One aggregate bar per variant, same processed-step view as v3.2.
                    already_done = sum((directory / 'samples' / p['sample_id'] / 'complete.json').is_file()
                                       for p in pairs)
                    progress_bar = tqdm(total=len(pairs) * STEPS, initial=already_done * STEPS,
                                        desc=f'Attack {variant}', unit='step', mininterval=1.5,
                                        dynamic_ncols=True, leave=True)
                for position, (pair, source, target) in enumerate(pair_batches(pairs), start=1):
                    sid = pair['sample_id']; sample_dir = directory / 'samples' / sid
                    complete = sample_dir / 'complete.json'
                    if complete.is_file():
                        row = json.loads(complete.read_text(encoding='utf-8'))
                        state_path = sample_dir / 'state.json'
                        if row['run_fingerprint'] != fingerprint or not complete_record_ok(directory, row) or sha256_file(state_path) != row['state_sha256']:
                            raise RuntimeError(f'Completed artifacts or state changed: {sid}')
                        restore_checkpoint(json.loads(state_path.read_text(encoding='utf-8')), foa_loss)
                        completed += 1
                        event('resume', 'skip_complete', sample_id=sid, variant=variant)
                        continue
                    text = texts.get(sid, {})
                    if spec['mpc'] and (text.get('caption_error') or not text.get('source') or not text.get('target')):
                        raise RuntimeError(f'{sid}: {text.get("caption_error", "Missing caption")}')
                    if spec['negatives'] != 'none' and len(text.get('negatives', [])) < CF_NEGATIVE_CANDIDATES:
                        raise RuntimeError(f'{sid}: {text.get("negative_error", "Missing negative bank")}')
                    sample_dir.mkdir(parents=True, exist_ok=True)
                    event('attack', 'sample_start', sample_id=sid, variant=variant,
                          position=position, total=len(pairs))
                    output, stats = run_one(models, source, target, text, variant, sample_dir, sid, foa_loss,
                                            progress_bar=progress_bar, sample_position=position, sample_total=len(pairs))
                    row = save_result(directory, pair, source, target, output)
                    write_json(sample_dir / 'state.json', state_checkpoint(foa_loss))
                    row.update(variant=variant, run_fingerprint=fingerprint, steps=STEPS,
                        short_run_diagnostic=STEPS != 300, stats=stats,
                        text_record_sha256=json_hash(text), state_sha256=sha256_file(sample_dir / 'state.json'),
                        negative_captions=[r['caption'] for r in text.get('negatives', [])])
                    write_json(complete, row)  # Commit only after images and state are durable.
                    completed += 1
                    rebuild_manifest(directory, pairs, fingerprint)
                    write_json(directory / 'status.json', {'status': 'running', 'completed': completed, 'total': len(pairs)})
                    del output
                statuses[variant] = {'status': 'complete', 'completed': completed, 'total': len(pairs)}
            except Exception as exc:
                error = {'variant': variant, 'sample_id': sid, 'error': f'{type(exc).__name__}: {exc}', 'traceback': traceback.format_exc()}
                append_jsonl(directory / 'errors.jsonl', error)
                statuses[variant] = {'status': 'failed', 'completed': completed, 'total': len(pairs), 'error': error['error']}
                event('attack', 'variant_failed', variant=variant, error=error['error'])
                if not CONTINUE_ON_SAMPLE_ERROR:
                    raise
            finally:
                if progress_bar is not None:
                    progress_bar.close()
                del foa_loss, models
                MODEL_STORE.clear()
                cuda_cleanup()
                rebuild_manifest(directory, pairs, fingerprint)
                write_json(directory / 'status.json', statuses.get(variant, {'status': 'interrupted', 'completed': completed}))
        write_json(RUN_DIR / 'summary.json', statuses)
        all_complete = all(row['status'] == 'complete' for row in statuses.values())
        write_json(RUN_DIR / 'status.json', {'status': 'complete' if all_complete else 'partial_failure', 'variants': statuses})
        if EXPORT_ZIP:
            path = shutil.make_archive(str(RUN_DIR), 'zip', root_dir=RUN_DIR)
            event('export', 'zip_saved', path=path)
        event('run', 'finished', all_complete=all_complete, directory=str(RUN_DIR))
        return RUN_DIR
    
    if RUN_ATTACK and not SETUP_ONLY:
        RUN_DIR = run_experiment()
    else:
        print('Preflight only: no pip installation, pretrained model download or attack run.')
        print('Variants:', RUN_VARIANTS, '| reference steps:', STEPS, '| k_neg:', K_NEG)
        try:
            PREFLIGHT_PAIRS = prepare_pairs()
            print('Validated image pairs:', len(PREFLIGHT_PAIRS), '| first:', PREFLIGHT_PAIRS[0])
        except (FileNotFoundError, ValueError) as exc:
            print('Dataset preflight:', exc)
        print('Set RUN_ATTACK=True after reviewing Cell 1. Full-model GPU smoke is still required on Kaggle.')

## v4.3.2 — Finite-gradient safety / evaluation

- Real Kaggle ZIP from v4.3: both reference variants finished, dynamic saved step 0 attention then failed at its first backward (nonfinite gradient). Causal layer attention extraction succeeded.
- Dynamic uses FP32 by default (`HYBRID_AMP=False`, `GRADIENT_SCALE=1.0`), preserving the FOA+MPC+negative objective and the attention refresh schedule.
- If AMP is enabled manually, the hybrid-only numerical guard retries the same crop/random draws and restores FOA loss history before an AMP unscaled and FP32 retry; it writes `precision_retries`, `precision_mode`, and recovery diagnostics to `steps.jsonl`. It never suppresses a nonfinite FP32 gradient.
- References (`foa_reference`/`mpc_clip`) keep their original precision and update semantics. Outputs are a **new experiment**, not comparable as bitwise identical to previous hybrid AMP output; do not resume v4.3 directory.
- Synthetic tests exercise recovery; **full GPU validation must be performed on Kaggle** before declaring model execution successful.

## Đánh giá và đọc trace

- `foa_reference`: implementation FOA K=10. `mpc_clip`: MPC thiếu nhánh ảnh InternVL/DINO.
  `mpc_full_reference`: đầy đủ encoder MPC; cần bộ nhớ phù hợp.
- `hybrid_plain` → `hybrid_negative`: đo negative loss; `hybrid_negative` → `hybrid_fixed`:
  đo attention sequencing; `hybrid_fixed` → `hybrid_dynamic`: đo fresh map.
- `hybrid_random_region` kiểm soát chọn vị trí vùng, `hybrid_random_neg` kiểm soát hard selection.
  Cùng bank/caption và cặp ảnh cho mọi variant. RNG mining/region độc lập với RNG crop/K-means.
- `steps.jsonl` ghi objective components, FOA weights của full và crop, negative IDs/scores,
  hotspot/crop, attention version, thời gian; maps và schedules có checkpoint pixel hash.
- `manifest.jsonl`: chỉ mẫu hoàn thành; `errors.jsonl`/`status.json` giữ lỗi. Resume ở ranh giới mẫu,
  khôi phục RNG và FOA history từ checkpoint JSON. Không resume giữa hai optimizer steps.
- Budget được kiểm tra trên cả tensor và PNG saved. Reference giữ final clamp như upstream;
  không clamp ảnh adversarial trước từng bước, ngoài clamp bên trong normalizer của tác giả.
- Dùng held-out victim và cùng evaluation protocol để đo ASR/AvgSim. Qwen đã làm surrogate
  attention không phải held-out victim độc lập; surrogate score tăng không chứng minh transfer tăng.

**Kiểm chứng cục bộ:** xem `CHECK_REPORT` (không tải model weights). `GPU_SMOKE=True` chạy
vòng FOA/hybrid với tiny frozen encoders trên CUDA để kiểm tra orchestration/budget;
đây không phải xác nhận chạy đủ encoder thật. Full-model smoke cần bật attack trên Kaggle.

In [ ]:
# Cell 14 — Artifact-only preview and partial export after interruption.
if globals().get('_NOTEBOOK_STAGE') == 'attack':
    PREVIEW_VARIANT = 'hybrid_dynamic'
    PREVIEW_INDEX = 0
    RUN_PREVIEW = False
    EXPORT_PARTIAL = False
    
    if RUN_PREVIEW:
        import matplotlib.pyplot as plt
        if RUN_DIR is None:
            raise ValueError('Set RUN_DIR to an existing run before preview.')
        folder = Path(RUN_DIR) / PREVIEW_VARIANT / 'samples' / f'{PREVIEW_INDEX:05d}'
        fig, axes = plt.subplots(1, 3, figsize=(12, 4))
        for ax, role in zip(axes, ('source', 'target', 'adversarial')):
            with Image.open(folder / f'{role}.png') as image:
                ax.imshow(image.copy())
            ax.set_title(role); ax.axis('off')
        plt.tight_layout(); plt.show()
        rows = [json.loads(line) for line in (folder / 'steps.jsonl').read_text(encoding='utf-8').splitlines()]
        fig, ax = plt.subplots(figsize=(9, 3))
        for name in ('foa_score', 'mpc_score', 'negative_loss'):
            ax.plot([r['step'] for r in rows], [r[name] for r in rows], label=name)
        ax.set_xlabel('Update'); ax.legend(); plt.tight_layout(); plt.show()
        maps = sorted((folder / 'maps').glob('*.npz'))
        if maps:
            selected = [maps[0], maps[len(maps) // 2], maps[-1]]
            fig, axes = plt.subplots(1, len(selected), figsize=(12, 4))
            for ax, path in zip(axes, selected):
                with np.load(path) as data:
                    ax.imshow(data['probability'], cmap='magma')
                ax.set_title(path.stem); ax.axis('off')
            plt.tight_layout(); plt.show()
    
    if EXPORT_PARTIAL:
        if RUN_DIR is None or not Path(RUN_DIR).is_dir():
            raise ValueError('Set RUN_DIR to an existing run before exporting.')
        print(shutil.make_archive(str(RUN_DIR) + '_partial', 'zip', root_dir=RUN_DIR))

In [ ]:
if _NOTEBOOK_STAGE == 'control':
    molab_launch()
